# DA3METRIC-LARGE — **Self-Supervised** Fine-Tuning Ablation Study on KITTI Eigen-Split  ·  **v6.2 (corrected, third review applied)**

> ## ⚠️ v6 / v6.1 / v6.2 supersede v5 — read this first
> The v5 Group 1 numbers **must not be used as an ablation result**. All eleven photometric / geometric "self-supervised" losses (and the five stationary-object losses) finished at an unscaled test RMSE of ≈ 2.938 m — identical to 4 significant figures across losses *and* across learning rates spanning 1e-4 … 1e-7, which is the signature of a network that was never updated — while the three `gt_*` losses and `lidar_anchor_only` reached 2.39 – 2.52 m. Post-mortem:
>
> | # | Finding (v5) | Severity | v6 fix |
> |---|---|---|---|
> | 1 | Every SSL loss contained `if pose is None … return |pred − gt|`, i.e. **silent supervised-depth fallback** | critical | Fallbacks deleted. Missing input ⇒ `SSLInputError`. Pure-SSL losses are called with `gt=None, mask=None`, so they *cannot* read LiDAR. |
> | 2 | `gt_*` losses were not in `NEED_POSE_LOSSES` ⇒ no pose was built ⇒ they fell back to depth L1 — **they were supervised controls, not oracle-pose runs**; Group 1 also used a dataset with no OXTS pose at all | critical | One `LOSS_SPECS` table (`needs_pred_pose / needs_gt_pose / needs_pred_next / needs_tracking / uses_gt_depth`) drives all routing and is asserted equal to the registry. Oracle runs are a **separate group (1b)** on the OXTS dataset with `assert has_gt_pose.all()`. |
> | 3 | PoseNet started at exactly R = I, t = 0, where the warp is independent of depth ⇒ **zero depth-gradient** (synthetic test: 2.7e-10 at identity vs 3.6e-5 with a 0.5 m motion) | major | No zero-init, output scale 0.1 (a ~1 m/frame translation needs a raw output ~10), **PoseNet warm-up with depth frozen** (a better *start*, not a guarantee of metric scale — see the scale caveat below), separate `POSE_LR`, per-group gradient clipping. |
> | 4 | Scheduler sized with `len(train_loader)` although it steps once per **optimizer update** (`ACCUM_STEPS=4`) ⇒ schedule 4× too long | major | `updates_per_epoch = ceil(len(loader)/ACCUM_STEPS)`; final partial accumulation window is flushed. |
> | 5 | *(new)* Warp geometry ran under **fp16 autocast**: `K·X` reaches ≈ 9.6e4 > 65 504 ⇒ `inf` ⇒ NaN gradients ⇒ `GradScaler` skipped the step. `combined_recommended` contains a 0.5×SILog LiDAR anchor yet ended at the *same* 2.938 as pure photometric losses — which strongly suggests that **no update was ever applied** | critical | Backprojection / projection / warping and all loss maths run in fp32 (autocast disabled). Non-finite updates are counted; a trial whose 1st epoch applies 0 updates aborts. |
> | 6 | *(new)* OXTS relative pose was in **IMU axes** (x fwd) not camera axes (z fwd): `T_ic` cancels in `inv(T_ic·W_j)·T_ic·W_i` | major (oracle only) | `T_cam = T_ic · inv(W_j) · W_i · inv(T_ic)` (conjugation). Verified: 1 m forward ⇒ `t = (0,0,−1)`. Missing calibration ⇒ sample dropped, no "approximate IMU frame". Check 0a validates it on LiDAR pixels. |
> | 7 | *(new)* Horizontal flip did not mirror `K` (cx) nor the GT pose; SAM3 masks were not flipped | moderate | Flip is drawn once per sample; `cx → W−1−cx`, `T → F·T·F`, SAM3 detections mirrored. |
> | 8 | *(new)* Tracking prefetch submitted its future **once** and re-read it every step ⇒ all steps used the first batch's masks | moderate | Proper one-step look-ahead iterator. |
> | 9 | *(new)* Group 5 recipes C = D and F = G were byte-identical configurations; groups 5/6 forced routing through `loss_name="combined_recommended"` | moderate | Distinct recipes + explicit `LossSpec` per recipe. |
> | 10 | Photometric L1 averaged over invalid pixels; `min_reproj` was single-source; "multi-scale" only blurred depth; "GeoNet" loss had no flow branch; temporal-warp mask was derived from the residual it minimised | moderate | Masked means; 3-frame `min` + auto-mask (Monodepth2); image-pyramid multi-resolution loss; loss renamed `rigid_flow_consistency`; visibility mask separated from residual (robust bounded L1). |
>
> **New workflow:** (0) *Pre-flight sanity suite* → (1) Group 1a SSL losses (+ controls) → (1b) oracle-pose diagnostic → (2–6). The groups refuse to start until the sanity suite shows the depth network is actually being updated.
> Results go to a **new output directory** (`…/ablation_study_v6_2`) and every checkpoint / cache / summary row carries an experiment hash, so results from other versions or configurations are never resumed.

> ### v6.1 — second-round review (all items applied)
>
> | # | Issue found in v6 | Fix in v6.1 |
> |---|---|---|
> | 1 | Gradient accumulation divided by `ACCUM_STEPS` even in a short final window | divide by the actual window size |
> | 2 | Groups 2–4 defaulted to the **hybrid** LiDAR loss | `BASE_LOSS_FOR_GROUPS = "combined_recommended_ssl"` (pure SSL); a warning is logged if you switch to a LiDAR-using loss |
> | 3 | "Every loss" smoke test skipped the 5 tracking-based losses | test 0b-2 runs them on real SAM3 tracks (scans a few batches), else on synthetic masks (reported in the table) |
> | 4 | "ByteTrack confirmed optimal" wording | removed; tracker is a configuration choice with no held-out tracking evaluation |
> | 5 | Oracle vs predicted-pose runs could use different sample sets / splits | one canonical sample list (samples with full OXTS pose+calibration) shared by all loaders + one shared index split; asserted equal |
> | 6 | `dyn_mask` supported by the temporal loss but never supplied | SAM3 class-prior dynamic mask (`get_batch_dynamic_masks`, current frame only since v6.2) wired into temporal / geometric / combined losses (`USE_DYNAMIC_MASK`, `LossSpec.needs_dyn_mask`) |
> | 7 | Pure-SSL training still read/decoded/transferred LiDAR | `load_gt=False` loaders for SSL training; LiDAR only for supervised/hybrid losses and for validation |
> | 8 | Part-9 stationary changes were monkey-patches | merged into the original classes; patch cell now only holds the optional geometric filter |
> | 9 | Stationary losses were mixed into the plain-SSL group | own category `stationary_ssl`, own **Group 1c** (`stat_` prefix) |
> | 10 | Pretrained baseline re-validated for every HPT trial | computed once per validation set, cached in memory + on disk |
> | 11 | Periodic checkpoints saved every epoch and never cleaned | `SAVE_EVERY_N_EPOCHS = 0`; cleanup also removes `temp_ep*` files |
> | 12 | Weak pose-warm-up cache key | key = code version + model id + hash of the exact samples/aug/root/offsets + all warm-up hyper-parameters |
> | 13 | Over-strong claim about metric pose scale | comments/docs corrected; sanity Exp C reports `pose_scale_ratio` = mean‖t_pred‖ / mean‖t_gt‖ |
> | 14 | Identity tie-break noise added after the min | added to each identity error before the min (Monodepth2) |
> | 15 | `FeatureMetricLoss` / multi-scale `K` rescaling used one ratio for x and y | separate width / height ratios |
> | 16 | `decoder_only` selected by name substrings, unchecked | `describe_trainable()` logs trainable/total params per top-level DA3 module and asserts the fraction lies in `EXPECTED_TRAINABLE_FRACTION[arch_mode]` |
> | 17 | Sanity criteria did not require a live PoseNet | `pose_grad_nonzero` required for every predicted-pose experiment |
>
> **Scale caveat for the thesis:** pure predicted-pose monocular self-supervision does not by itself guarantee metric scale. The photometric loss constrains depth and translation only jointly; the metric DA3 initialisation and the frozen-depth pose warm-up start the system in the right place, but scale can still drift. The oracle-pose group (known metric translation) is what removes this ambiguity.

> ### v6.2 — third-round review (all items applied)
>
> | # | Issue found in v6.1 | Fix in v6.2 |
> |---|---|---|
> | 1 | Output dir still `…/ablation_study_v6`; resume logic keyed only on label + LR, so old checkpoints / summary rows could be resumed | new dir `ablation_study_v6_2`; **experiment hash** of all training-relevant constants namespaces the checkpoint directory, the pose warm-up / baseline caches and every summary row; **per-trial hash** (loss definition + weights via `extra_repr`, arch, aug, optimizer, scheduler, routing spec, HPT config, samples) is part of each checkpoint name and is verified on resume; `get_completed_ablations()` only counts rows with the current hash and warns about foreign rows |
> | 2 | Dynamic mask united the t and t+1 SAM3 masks although the losses live in frame-t coordinates | **current-frame mask only** (warping the t+1 mask would need predicted geometry/pose) |
> | 3 | Accumulation divided by the nominal window size even when a non-finite batch contributed nothing | gradients are rescaled by `window_size / valid_in_window` before the step (verified: update = exact mean gradient of the finite batches) |
> | 4 | All five stationary losses triggered dynamic-mask generation (and duplicate SAM3 reads) | only `combined_stationary` needs it (`_TRD`); the other four use `_TR` |
> | 5 | "B vs C differ only in pose" overstated (independent random augmentation draws) | sanity experiments A–D and the smoke test run with `aug_mode="none"` (byte-identical inputs); documentation now says *same samples and split, identically distributed but independently drawn augmentation* for the ablation groups |
> | 6 | Sanity gate could pass while oracle infrastructure was missing / failed check 0a | with `RUN_ORACLE_POSE_ABLATION=True`, missing OXTS loaders, a failed 0a, a skipped B or an untestable oracle loss all **fail** the gate |
> | 7 | Warm-up cache key omitted several settings | key = hash of a canonical dict incl. weight decay, batch size, accumulation, clip norm, AMP mode / scaler, target resolution, offsets, samples, experiment hash |
> | 8 | Group 5 / 5b / 6 rows all reported `loss_fn = combined_recommended` | the recipe / weight-set id is the logical loss name (`recipe_A_photo_only`, `weights_photo_high`, …) |
> | 9 | Stale change-log row claimed Hungarian is "the theoretically correct choice" | rewritten: no optimality claim |
> | — | *Bug caught while testing v6.2* | the provenance hash block had been inserted in the middle of the config cell and would have missed every constant defined after it (tracking / stationary settings); it now sits at the very end and is tested |
>
> **Population note (thesis):** with `CANONICAL_SAMPLE_SET=True` **all** groups — including plain SSL Group 1a — use only samples with complete OXTS pose + calibration ("Design B": one common subset, so oracle-vs-predicted comparisons are exact). Group 1a therefore is *not* SSL over the full sequential KITTI training population; set `CANONICAL_SAMPLE_SET=False` for that design (then oracle-vs-predicted differ in sample set and a warning is logged).

---

**Companion to the fully-supervised ablation** (`DA3_supervised_FT_Ablation_HPT_v6.ipynb`). This notebook studies *self-supervised* and *hybrid* fine-tuning where the bulk of the learning signal comes from **geometric / photometric consistency between neighbouring frames** rather than from dense LiDAR labels.

---

## Why a separate self-supervised study?

For a metric depth foundation model already trained on huge amounts of data (DA3METRIC-LARGE), the supervised fine-tuning question is "which loss best uses my sparse LiDAR?". The self-supervised question is fundamentally different: **"which geometric prior best uses my video stream when labels are absent, noisy, or scarce?"**. The two regimes have different failure modes, different hyper-parameter sensitivities and different best-in-class losses — so we treat them as independent ablations and compare their winners at the end.

| Tier | Signal | Geometric correctness | Metric-scale capacity | Cost |
|------|--------|----------------------|----------------------|------|
| **1** | Pose + differentiable warping (photometric reprojection) | Excellent | Good (depth is already metric; pose learns metres) | Moderate |
| **2** | Rigid-flow (depth+pose induced) consistency | Moderate | Weak / medium | Higher |
| **3** | Temporal depth consistency via *warped* depth | Good (proper) | Medium | Low |
| **3a (heuristic)** | Naive "mean depth-delta" temporal regulariser | **Wrong (globally constant Δd assumption)** | None | Low |
| **4 (anchor)** | Sparse LiDAR SILog | Direct (supervised) | Excellent | Low |

## What this notebook does

| Step | Description |
|------|-------------|
| 0 | **Pre-flight sanity suite**: GT-pose convention check, gradient smoke test of every loss, and 4 short training experiments (A supervised, B oracle-pose photometric, C predicted-pose photometric, D temporal geometry) |
| 1 | **6-DoF PoseNet** (ResNet-18) with depth-frozen warm-up + fp32 differentiable inverse warping; **3-frame** (t−1, t, t+1) Monodepth2-style training |
| 2 | **Group 1a**: 13 self-supervised / regulariser losses plus 2 labelled controls (`lidar_anchor_only` supervised, `combined_recommended` hybrid); **Group 1c**: 5 stationary-object (SAM3-tracking) losses |
| 3 | **Group 1b**: oracle-pose diagnostic (GT OXTS pose + learned depth) — *not self-supervised* |
| 4 | Groups 2–4: architecture / augmentation / optimizer-scheduler (fixed **pure-SSL** base loss `BASE_LOSS_FOR_GROUPS = combined_recommended_ssl`) |
| 5 | Group 5: cumulative loss recipes A→G (all distinct); Group 6: loss-weight sweep |
| 6 | HPT over the **depth** learning rate (PoseNet has its own `POSE_LR`) + early stopping + strict TEST isolation; every trial also logs the **pretrained-baseline validation RMSE** (epoch 0) and gradient / skipped-update diagnostics |
| 7 | KITTI Eigen-split evaluation under 4 scale-alignment strategies; same CSV schema as the supervised study (+ `loss_category`, `baseline_val_rmse`) |

## Loss families (v6)

| Registry name | Category | Pose | Reference / note |
|---|---|---|---|
| `photo_l1` | SSL | PoseNet | Garg 2016, masked mean |
| `photo_ssim` | SSL | PoseNet | SSIM-only |
| `photo_ssim_l1` | SSL | PoseNet | Godard 2017 (mean over sources) |
| `min_reproj_photo` | SSL | PoseNet | **Monodepth2** (min over t±1 + auto-mask) |
| `multi_scale_photo` | SSL | PoseNet | image-pyramid variant of the above |
| `geometric_consistency` | SSL | PoseNet | SC-SfMLearner, Bian 2019 |
| `feature_metric` | SSL | PoseNet | *simplified* FeatDepth-style (frozen ResNet18 features) |
| `rigid_flow_consistency` | SSL | PoseNet | depth+pose induced flow — *not* GeoNet (no learned flow branch); v5 name `flow_depth_consistency` |
| `temporal_warp_consistency` | SSL | PoseNet | warped-depth consistency (+ SAM3 dynamic-class mask) + photometric anchor |
| `combined_monodepth2` | SSL | PoseNet | multi-resolution min-reproj + smoothness |
| `combined_recommended_ssl` | SSL | PoseNet | recommended recipe **without** the LiDAR anchor |
| `edge_aware_smooth` | regulariser | – | degenerate on its own (constant depth) |
| `naive_temporal_delta` | negative control | – | geometrically wrong heuristic |
| `stationary_*`, `full_3d_ego_motion`, `mask_chamfer`, `combined_stationary` | stationary-object SSL (Group 1c) | PoseNet | Part 5 (SAM3 tracks; erosion / area weighting / MAD built in) |
| `combined_recommended` | **hybrid** | PoseNet | uses 0.5×SILog on LiDAR on purpose |
| `lidar_anchor_only` | **supervised control** | – | upper-bound reference |
| `gt_photo_reproj`, `gt_rigid_flow_consistency`, `gt_temporal_warp_consistency` | **oracle pose** | OXTS | GT pose, learned depth, no GT depth in loss |

> Run order: ① set paths in the *Configuration* cell, ② run cells top-to-bottom. Cell 13b runs the sanity suite; the group cell refuses to start if it fails. The notebook is **resumable**: any ablation already present in `SUMMARY_CSV` is skipped.


## 0 · Imports


In [1]:
import os
# Set the visible device to index 1 (GPU 1)
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
from __future__ import annotations
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import copy
import csv
import glob
import json
import logging
import math
import shutil
import sys
import time
import warnings
from itertools import combinations
from collections import Counter
from dataclasses import dataclass, field
from scipy.optimize import linear_sum_assignment
from pathlib import Path
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.models as tvm
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, Subset
import torch.backends.cudnn as cudnn
from contextlib import nullcontext
from functools import lru_cache         # unsure about the mask size
from concurrent.futures import ThreadPoolExecutor
executor = ThreadPoolExecutor(max_workers=2)
cudnn.benchmark = True

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("ssl_ablation")
logging.getLogger("transformers").setLevel(logging.ERROR)

print("Imports OK  |  CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"  device: {torch.cuda.get_device_name(0)}")


Imports OK  |  CUDA: True
  device: Tesla V100-PCIE-32GB


## 1 · Configuration


In [3]:
# ─── PATHS ───────────────────────────────────────────────────────────────────
KITTI_ROOT      = "/home/arka/Downloads/MMDE/datasets/KITTI/tiniest"
TRAIN_ROOT      = os.path.join(KITTI_ROOT, "train")
TEST_ROOT       = os.path.join(KITTI_ROOT, "test")
CACHE_DIR       = "/home/arka/Downloads/MMDE/cache"

OUTPUT_BASE     = "/home/arka/Downloads/MMDE/outputs/self_supervised_FT_output/ablation_study_v6_2_1_stationary_fix"   # NEW dir: v5 / v6 / v6.1 / v6.2 outputs must never be resumed / overwritten (see Change Log)
CHECKPOINT_DIR  = os.path.join(OUTPUT_BASE, "checkpoints")
RESULTS_DIR     = os.path.join(OUTPUT_BASE, "results")
PLOT_DIR        = os.path.join(OUTPUT_BASE, "plots")
ANALYSIS_DIR    = os.path.join(OUTPUT_BASE, "analysis")
SUMMARY_CSV     = os.path.join(RESULTS_DIR, "ablation_summary.csv")
TEMP_CKPT_DIR   = os.path.join(OUTPUT_BASE, "temp_ckpts")
for d in [OUTPUT_BASE, CHECKPOINT_DIR, RESULTS_DIR, PLOT_DIR, ANALYSIS_DIR, TEMP_CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# ─── MODEL ────────────────────────────────────────────────────────────────────
MODEL_ID        = "depth-anything/DA3METRIC-LARGE"
ORTHO_NORM      = 300.0
FOCAL_LENGTH_X  = 721.5377         # KITTI default: avg(fx, fy) ≈ 721.5 px  (v6: forward_batch now uses avg(fx, fy) from K)
USE_MODEL_FOCAL = False            # DA3METRIC-LARGE doesn't have intrinsic prediction

# ─── TRAINING BUDGET ──────────────────────────────────────────────────────────
ABLATION_EPOCHS = 5                # epochs per HPT trial
BATCH_SIZE      = 4
ACCUM_STEPS     = 4                # effective batch = 16
LEARNING_RATE   = 1e-6              # default LR
HPT_LR_CANDIDATES  = [1e-5, 1e-6]
WEIGHT_DECAY    = 1e-4
WARMUP_EPOCHS   = 1
MAX_GRAD_NORM   = 1.0
MIXED_PREC      = True
NUM_WORKERS     = 8
MAX_TRAIN       = 0                 # 0 = all; set e.g. 500 for fast sweep
MAX_TEST        = 0                 # 0 = all;
gpu_duty_cycle = 1.0
"""
    gpu_duty_cycle: fraction of wall-time the GPU is actively computing.
                    0.5 means equal compute and sleep → ~50% utilisation.
                    Set to 0.2 for 20% utilisation.
                    Set to 0.8 for 80% utilisation.
                    Set to 1.0 to disable throttling entirely.
"""

# ─── DATASET ──────────────────────────────────────────────────────────────────
TARGET_H        = 378              # multiple of 14
TARGET_W        = 1246             # multiple of 14
MIN_DEPTH       = 1e-3
MAX_DEPTH       = 80.0
SSL_MIN_DEPTH   = 1e-3
SSL_MAX_DEPTH   = 80.0

# ─── ABLATION DIMENSIONS TO RUN ──────────────────────────────────────────────
# v6.2.1: this build is configured for a GROUP-1c-ONLY rerun of the corrected stationary-object
# losses. Every flag below is a plain, reversible on/off switch -- nothing about the ablation
# groups themselves was removed; flip these back to resume the full study.
RUN_LOSS_ABLATION         = False   # Group 1a -- unaffected by this patch; off so the rerun is Group-1c-only
RUN_ARCH_ABLATION         = False
RUN_AUG_ABLATION          = False
RUN_OPTIM_ABLATION        = False
RUN_COMBINATION_ABLATION  = False   # NEW Group 5 — cumulative recipes
RUN_WEIGHT_ABLATION       = False   # NEW Group 6 — weight sweep

# ─── SSL-SPECIFIC HYPERPARAMETERS ────────────────────────────────────────────
EPS                             = 1e-6
SSIM_ALPHA                      = 0.85
SMOOTHNESS_DECAY                = 1.0
MIN_REPROJ_AUTO_MASK_THRESHOLD  = 0.0       # 0 → strict auto-mask (Monodepth2 default)
TEMPORAL_WARP_THRESHOLD         = 0.05      # m — static-pixel threshold

# Default loss weights for combined_recommended
DEFAULT_W_PHOTO    = 1.0
DEFAULT_W_SMOOTH   = 1e-3
DEFAULT_W_TEMPORAL = 0.1
DEFAULT_W_ANCHOR   = 0.5
DEFAULT_W_FEAT     = 0.01
DEFAULT_W_GEOM     = 0.1

# ─── HPT & EARLY-STOPPING ────────────────────────────────────────────────────
EARLY_STOP_PATIENCE  = ABLATION_EPOCHS - 1
VAL_SPLIT_FRAC       = 0.10     # from 0.05
VAL_SPLIT_SEED       = 42
SAVE_EVERY_N_EPOCHS  = 0        # periodic checkpoints are redundant in HPT (latest + best suffice); 0 = disabled


LOSS_HPT_GRID  = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
ARCH_HPT_GRID  = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
AUG_HPT_GRID   = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
OPTIM_HPT_GRID = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
COMBO_HPT_GRID = [{"lr": lr} for lr in HPT_LR_CANDIDATES]
WEIGHT_HPT_GRID = [{"lr": lr} for lr in HPT_LR_CANDIDATES]


print("Configuration loaded  (v6: no-fallback losses, 3-frame training, pose warm-up, fp32 geometry).")
print(f"  patience={EARLY_STOP_PATIENCE}  val_frac={VAL_SPLIT_FRAC}  "
      f"lr_candidates={HPT_LR_CANDIDATES}")
print(f"  ssim_alpha={SSIM_ALPHA}  smooth_weight={DEFAULT_W_SMOOTH}  "
      f"temporal_weight={DEFAULT_W_TEMPORAL}  anchor_weight={DEFAULT_W_ANCHOR}")
# =============================================================================
# ─── v6 CORRECTIONS ──────────────────────────────────────────────────────────
# =============================================================================
# (1) 3-frame training (Monodepth2): sources are I_{t-1} and I_{t+1}.
#     Use (+1,) to fall back to the old 2-frame setting.  +1 MUST be present
#     because temporal / geometric / stationary losses use the t+1 frame.
SOURCE_OFFSETS = (-1, +1)

# (2) PoseNet bootstrapping.  v5 started PoseNet at EXACTLY identity, where the
#     warp is independent of depth  ->  zero depth-gradient.  v6:
#       * PoseNet last layer is no longer zero-initialised
#       * output scale 0.1 (a 1 m/frame translation needs a raw output ~10, not ~100)
#       * PoseNet is pre-trained for POSE_WARMUP_EPOCHS with the DA3 depth
#         FROZEN (photometric loss only, no GT depth) before joint training
#       * PoseNet gets its own learning rate (POSE_LR); the HPT grid only
#         sweeps the DEPTH learning rate.
#     CAVEAT (thesis): none of this removes the scale ambiguity of monocular
#     self-supervision.  The photometric loss constrains only depth*pose-translation
#     jointly, so a predicted-pose system can still drift in metric scale; the metric
#     DA3 initialisation and the frozen-depth warm-up merely start it in the right place.
#     The oracle-pose group (known metric translation) is what removes that ambiguity.
POSE_LR                 = 1e-4
POSE_OUT_SCALE          = 0.1
POSE_WARMUP_EPOCHS      = 1          # 0 disables pose pre-training
POSE_WARMUP_MAX_BATCHES = 0          # 0 = whole epoch
POSE_WARMUP_CODE_VERSION = "v6.2"    # bump when PoseNet / warm-up / warp code changes (part of the cache key)

# (3) Mixed precision.  Geometry (backprojection / projection / warping) and all
#     loss maths ALWAYS run in fp32 in v6 (fp16 overflowed: K·X reaches ~1e5 > 65504).
#     "fp16" (default, uses GradScaler) | "bf16" | "auto"
AMP_DTYPE_MODE = "fp16"
if AMP_DTYPE_MODE == "bf16" or (AMP_DTYPE_MODE == "auto" and torch.cuda.is_available()
                                and torch.cuda.is_bf16_supported()):
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16
USE_GRAD_SCALER = bool(MIXED_PREC and torch.cuda.is_available() and AMP_DTYPE == torch.float16)

# (4) Pre-flight sanity suite (Experiments 0, A–D).  Group runs are blocked
#     until the pipeline demonstrably updates the depth network.
RUN_SANITY_EXPERIMENTS   = True
REQUIRE_SANITY_PASS      = True
SANITY_TRAIN_BATCHES     = 120       # batches per sanity experiment (=> 120/ACCUM_STEPS optimizer updates)
SANITY_VAL_BATCHES       = 16
SANITY_MAX_SKIP_FRAC     = 0.25      # max tolerated fraction of skipped (non-finite) optimizer updates
SANITY_LR                = 1e-5      # depth LR used inside the sanity experiments

# (4b) Experimental-validity switches (v6.1)
CANONICAL_SAMPLE_SET     = True      # oracle-pose and predicted-pose runs use EXACTLY the same sample ids / split
USE_DYNAMIC_MASK         = True      # exclude potentially-moving-object pixels (SAM3 class prior) from the
                                     # temporal / geometric depth-consistency terms
DYN_MASK_DILATE_PX       = 3
RUN_STATIONARY_ABLATION  = True      # Group 1c: SAM3-tracking stationary-object losses (separate from Group 1a)
SANITY_STATIONARY_SCAN_BATCHES = 8   # v6.2.1: real-data coverage preflight -- batches to scan (cheap; not the full dataset)
SANITY_STATIONARY_MIN_PAIRS    = 1   # v6.2.1: minimum real stationary pairs required in that scan for the sanity gate to pass
STRICT_ARCH_CHECK        = True      # assert the trainable-parameter fraction of each arch_mode is in range
EXPECTED_TRAINABLE_FRACTION = {"decoder_only": (1e-3, 0.60),
                               "last2_blocks": (1e-4, 0.60),   # = last 30 % of head/decoder parameter tensors
                               "full":         (0.999, 1.0)}

# (5) Oracle-pose diagnostic group (GT OXTS pose + learned depth).  NOT self-supervised.
RUN_ORACLE_POSE_ABLATION = False   # v6.2.1: off for the Group-1c-only rerun (reversible; oracle-pose losses/definitions are untouched)

# ─── TRACKING & DYNAMIC-OBJECT FILTER (Part 3 / 4 / 7) ────────────────────────
ENABLE_OXTS            = True            # gate for GT-pose dataset/losses (Part 2b, already implemented)
ENABLE_TRACKING        = True            # gate for Part 3 mask tracking
ENABLE_DYNAMIC_FILTER  = True            # gate for Part 4 stationary-object filter
ENABLE_GT_POSE         = True            # stationary losses: prefer the pose passed as gt_pose (only set for oracle runs)
ENABLE_CHAMFER         = True            # Part 5 Loss 4 (wired in the next build stage)
ENABLE_UNIFORMITY      = True            # Part 5 Loss 2 (wired in the next build stage)

TRACKER_TYPE           = "hungarian"     # "iou" | "hungarian" | "bytetrack"  — a configuration choice, NOT a validated optimum (no held-out tracking evaluation exists)
SAM3_MASK_DIR          = "/home/arka/Downloads/MMDE/outputs/inst_seg_output" #os.path.join(KITTI_ROOT, "sam3_masks")
DYNAMIC_FILTER_MODE    = "both"          # "class" | "flow" | "both"
FLOW_RESIDUAL_PX       = 2.0             # px, residual object flow after ego-flow subtraction
TRACK_IOU_MATCH_THRESH = 0.15            # Tracker 1 minimum combined score to accept a match
TRACK_MAX_AGE          = 5               # frames a track survives with no matched detection
TRACK_MIN_HITS         = 1               # matched frames required before a track is "confirmed"
BYTETRACK_HIGH_THRESH  = 0.5             # SAM3 score split point: high vs low confidence detections
BYTETRACK_LOW_THRESH   = 0.1             # detections below this score are discarded entirely

# ─── STATIONARY-OBJECT LOSS WEIGHTS (Part 5 / Part 7) ─────────────────────────
DEFAULT_W_STATIONARY   = 0.5    # Loss 1 — StationaryObjectDepthMotionLoss
DEFAULT_W_UNIFORMITY   = 0.2    # Loss 2 — StationaryUniformMotionLoss
DEFAULT_W_3D_MOTION    = 0.5    # Loss 3 — Full3DEgoMotionLoss
DEFAULT_W_CHAMFER      = 0.1    # Loss 4 — MaskPointCloudChamferLoss
CHAMFER_MAX_POINTS     = 200    # sampled points per object per frame (Loss 4)

# ─── PART 9 CRITICAL-EVALUATION IMPROVEMENTS ──────────────────────────────────
MASK_ERODE_PX                   = 2       # px, morphological erosion before mask depth stats (Losses 1-4)
UNIFORMITY_METRIC               = "mad"   # "variance" | "mad" — Loss 2 dispersion metric (see Part 9 markdown)
ENABLE_GEOMETRIC_FILTER         = False   # extra outlier-rejection stage beyond Part 4 (default off — chicken-and-egg caveat, see Part 9 markdown)
GEOMETRIC_FILTER_WARMUP_EPOCHS  = 2       # epochs to wait before trusting the model's own depth as a filtering signal
GEOMETRIC_FILTER_MAD_THRESHOLD  = 3.0     # objects beyond this many MADs from their sample's median residual are dropped
GEOMETRIC_FILTER_MIN_OBJECTS    = 3       # skip filtering for samples with fewer objects than this (MAD undefined/unstable)

print(f"  output -> {OUTPUT_BASE}")

# =============================================================================
# EXPERIMENT PROVENANCE (v6.2)
# =============================================================================
# Every temp checkpoint, warm-up state and summary row is namespaced by a hash of ALL training-relevant
# constants above (paths, RUN_* switches, HPT grids, worker counts etc. are excluded), so a checkpoint
# produced under a different configuration / code version can never be resumed by accident.
# On top of this, each trial adds its own hash (loss definition + weights, arch, aug, optimizer, scheduler,
# routing spec, HPT config) — see `_trial_hash` in the runner.
import hashlib as _hashlib, json as _json
EXPERIMENT_CODE_VERSION = "v6.2.1-stationary-fix"       # bumped: track_and_filter_pair coordinate-system fix + class-name normalization (see Change Log) invalidate any Group-1c checkpoint computed under "v6.2" -- this alone changes EXPERIMENT_HASH via the existing mechanism below, so old Group 1c (and every other) checkpoint is never resumed.
_HASH_EXCLUDE = ("DIR", "ROOT", "CSV", "PATH", "OUTPUT", "CACHE", "RUN_", "NUM_WORKERS", "SANITY",
                 "REQUIRE_", "LOG", "EXPERIMENT", "CUDA", "GPU", "_GRID", "MAX_TEST", "STRICT_")
EXPERIMENT_CONFIG = {
    k: (list(v) if isinstance(v, tuple) else v)
    for k, v in sorted(globals().items())
    if k.isupper() and not k.startswith("_") and isinstance(v, (int, float, str, bool, tuple, list, dict))
    and not any(x in k for x in _HASH_EXCLUDE)
}
EXPERIMENT_CONFIG["TRAIN_ROOT"] = str(TRAIN_ROOT)          # data source + SAM3 masks change the experiment
EXPERIMENT_CONFIG["SAM3_MASK_DIR"] = str(SAM3_MASK_DIR)
EXPERIMENT_CONFIG["AMP_DTYPE"] = str(AMP_DTYPE)
EXPERIMENT_CONFIG["USE_GRAD_SCALER"] = bool(USE_GRAD_SCALER)
EXPERIMENT_CONFIG["CODE_VERSION"] = EXPERIMENT_CODE_VERSION
EXPERIMENT_HASH = _hashlib.md5(_json.dumps(EXPERIMENT_CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:10]
TEMP_CKPT_DIR = os.path.join(TEMP_CKPT_DIR, EXPERIMENT_HASH)          # hash-namespaced checkpoint directory
os.makedirs(TEMP_CKPT_DIR, exist_ok=True)
with open(os.path.join(RESULTS_DIR, f"experiment_config_{EXPERIMENT_HASH}.json"), "w") as _f:
    _json.dump(EXPERIMENT_CONFIG, _f, indent=1, sort_keys=True, default=str)
print(f"  experiment hash {EXPERIMENT_HASH}  (checkpoints: {TEMP_CKPT_DIR})")


Configuration loaded  (v6: no-fallback losses, 3-frame training, pose warm-up, fp32 geometry).
  patience=4  val_frac=0.1  lr_candidates=[1e-05, 1e-06]
  ssim_alpha=0.85  smooth_weight=0.001  temporal_weight=0.1  anchor_weight=0.5
  output -> /home/arka/Downloads/MMDE/outputs/self_supervised_FT_output/ablation_study_v6_2_1_stationary_fix
  experiment hash 093fdaab49  (checkpoints: /home/arka/Downloads/MMDE/outputs/self_supervised_FT_output/ablation_study_v6_2_1_stationary_fix/temp_ckpts/093fdaab49)


## 2 · Dataset (multi-frame sequences: t−1, t, t+1)


In [4]:
_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]
_train_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])

_DEFAULT_K = np.array([
    [721.5377,   0.0,   609.5593],
    [  0.0,   721.5377, 172.8540],
    [  0.0,     0.0,     1.0  ],
], dtype=np.float32)

_OFFSET_PREFIX = {+1: "next", -1: "prev"}          # batch-key prefix per source offset


def _read_kitti_depth(path: str) -> np.ndarray:
    raw = np.array(Image.open(path), dtype=np.uint16)
    return raw.astype(np.float32) / 256.0


def _parse_kitti_calib(date_dir: Path, drive_dir: Path) -> Optional[np.ndarray]:
    calib: Dict[str, np.ndarray] = {}
    for fname in ["calib_cam_to_cam.txt"]:
        for search in [date_dir, drive_dir]:
            p = search / fname
            if p.exists():
                for line in p.read_text().splitlines():
                    if ":" in line:
                        k, v = line.split(":", 1)
                        try:
                            calib[k.strip()] = np.array(v.split(), dtype=np.float64)
                        except Exception:
                            pass
    K = None
    try:
        if "P_rect_02" in calib:
            P = calib["P_rect_02"].reshape(3, 4); K = P[:3, :3].copy()
        elif "P2" in calib:
            P = calib["P2"].reshape(3, 4); K = P[:3, :3].copy()
    except Exception:
        pass
    return K


# ─── Augmentation (v6) ───────────────────────────────────────────────────────
# v5 flipped the images but left K's principal point (and, for OXTS runs, the
# GT pose) un-flipped -> wrong geometry on 50 % of the samples.  v6 draws the
# flip / colour-jitter parameters ONCE per sample, applies them to every frame
# of the sample, and updates K (and gt_pose, see the OXTS dataset) accordingly.
# Random crops stay excluded (they move the principal point).
#                   (flip, colour-jitter)
_AUG_SPECS = {"none": (False, False), "flip": (True, False),
              "base": (True, True),   "full": (True, True)}
AUG_FNS = _AUG_SPECS      # kept for backward-compat with prints in later cells


def _draw_aug_params(aug_mode: str, rng: np.random.RandomState) -> Dict:
    do_flip, do_jit = _AUG_SPECS.get(aug_mode, (False, False))
    flip = bool(do_flip and rng.rand() < 0.5)
    br = rng.uniform(0.8, 1.2) if do_jit else 1.0
    co = rng.uniform(0.8, 1.2) if do_jit else 1.0
    return dict(flip=flip, brightness=br, contrast=co, jitter=do_jit)


def _apply_color_jitter(rgb: np.ndarray, br: float, co: float) -> np.ndarray:
    f = rgb.astype(np.float32) / 255.0
    f = np.clip((f - f.mean()) * co + f.mean() * br, 0.0, 1.0)
    return (f * 255.0).astype(np.uint8)


# ─── Sequential multi-frame dataset (required for ALL SSL losses) ────────────
class KITTISequentialDepthDataset(Dataset):
    """
    Emits (I_t and its source frames I_{t+off}) from one KITTI drive, for every
    off in `source_offsets` (default (-1, +1) = Monodepth2 3-frame training).

    Batch keys
        current frame : rgb, rgb_raw, depth_gt, valid_mask, K, has_K, rgb_path, depth_path
        next   (+1)   : next_rgb, next_rgb_raw, next_depth_gt, next_valid_mask, next_K, ...
        prev   (-1)   : prev_rgb, prev_rgb_raw, prev_K, ...            (no GT depth is loaded)
        flipped       : bool  — whether the sample was horizontally flipped

    GT depth of the CURRENT frame (and +1 frame) is loaded only when load_gt=True
    (validation metrics / supervised controls).  Pure-SSL TRAINING uses load_gt=False,
    so no LiDAR is read, decoded or transferred; the trainer additionally passes
    gt=None to every loss that does not declare uses_gt_depth.
    """

    def __init__(self, root, split="train",
                 target_hw=(TARGET_H, TARGET_W),
                 min_depth=MIN_DEPTH, max_depth=MAX_DEPTH,
                 aug_mode="base", max_samples=0, seed=42,
                 source_offsets=SOURCE_OFFSETS,
                 load_gt: bool = True,
                 allowed_paths: Optional[set] = None):
        self.root      = Path(root)
        self.split     = split
        self.target_hw = target_hw
        self.min_depth = min_depth
        self.max_depth = max_depth
        self.aug_mode  = aug_mode
        self.augment   = (split == "train" and aug_mode != "none")
        self.rng       = np.random.RandomState(seed)
        self.offsets   = tuple(source_offsets)
        self.load_gt   = load_gt          # False for pure-SSL training: no LiDAR I/O / decode / transfer
        assert +1 in self.offsets, "source_offsets must contain +1"
        assert all(o in _OFFSET_PREFIX for o in self.offsets), \
            f"only offsets {list(_OFFSET_PREFIX)} are supported, got {self.offsets}"
        self._samples: List[Dict[int, Dict]] = []       # {0: rec, +1: rec, -1: rec}
        self._build_samples()
        if allowed_paths is not None:     # canonical sample set (identical ids for SSL and oracle-pose runs)
            self._samples = [s for s in self._samples if s[0]["rgb_path"] in allowed_paths]
        if max_samples > 0 and len(self._samples) > max_samples:
            idx = self.rng.choice(len(self._samples), max_samples, replace=False)
            self._samples = [self._samples[i] for i in sorted(idx)]
        log.info("[KITTI-Seq-%s] %d samples (offsets=%s, aug=%s)",
                 split, len(self._samples), self.offsets, aug_mode)

    # -- index ---------------------------------------------------------------
    def _build_drive_index(self) -> Dict:
        depth_glob = str(self.root / "*" / "*_sync" /
                         "proj_depth" / "groundtruth" / "image_02" / "*.png")
        depth_paths = sorted(glob.glob(depth_glob))
        if not depth_paths:
            depth_glob = str(self.root / "*" / "*" /
                             "proj_depth" / "groundtruth" / "image_02" / "*.png")
            depth_paths = sorted(glob.glob(depth_glob))

        by_drive: Dict = {}
        for dp in depth_paths:
            parts = Path(dp).parts
            try:
                gi        = list(parts).index("proj_depth")
                drive_dir = Path(*parts[:gi])
                date_dir  = drive_dir.parent
                fname     = parts[-1]
            except (ValueError, IndexError):
                continue
            rgb_path = drive_dir / "image_02" / "data" / fname
            if not rgb_path.exists():
                continue
            K = _parse_kitti_calib(date_dir, drive_dir)
            key = (date_dir.name, drive_dir.name)
            rec = dict(rgb_path=str(rgb_path), depth_path=dp, K=K,
                       frame_id=int(Path(fname).stem))
            by_drive.setdefault(key, []).append(rec)

        for key in by_drive:
            by_drive[key].sort(key=lambda r: r["frame_id"])
        return by_drive

    def _build_samples(self):
        # A sample exists only if EVERY required source frame is present and
        # frame ids are consecutive (offset == frame-id difference).
        for frames in self._build_drive_index().values():
            by_id = {r["frame_id"]: r for r in frames}
            for r in frames:
                fid = r["frame_id"]
                if all((fid + off) in by_id for off in self.offsets):
                    s = {0: r}
                    for off in self.offsets:
                        s[off] = by_id[fid + off]
                    self._samples.append(s)

    # -- loading -------------------------------------------------------------
    def _load_frame(self, rec: Dict, load_depth: bool = True) -> Dict:
        H, W    = self.target_hw
        rgb     = cv2.cvtColor(cv2.imread(rec["rgb_path"]), cv2.COLOR_BGR2RGB)
        orig_hw = rgb.shape[:2]
        rgb     = cv2.resize(rgb, (W, H), interpolation=cv2.INTER_LINEAR)

        depth = valid = None
        if load_depth:
            depth   = _read_kitti_depth(rec["depth_path"])
            valid_o = (depth > self.min_depth) & (depth < self.max_depth)
            depth = cv2.resize(depth, (W, H), interpolation=cv2.INTER_NEAREST)
            valid = cv2.resize(valid_o.astype(np.uint8), (W, H),
                               interpolation=cv2.INTER_NEAREST).astype(bool)
            depth = np.clip(depth, 0.0, self.max_depth)
            eigen_crop = np.zeros((H, W), dtype=bool)          # Eigen crop rows 44:396
            eigen_crop[44:min(44 + 352, H), :] = True
            valid = valid & eigen_crop

        K = rec["K"]
        if K is not None:
            K = K.copy()
            K[0] *= W / orig_hw[1]
            K[1] *= H / orig_hw[0]
        return dict(rgb_np=rgb, depth=depth, valid=valid, K=K,
                    rgb_path=rec["rgb_path"], depth_path=rec["depth_path"],
                    frame_id=rec["frame_id"])

    def _to_tensors(self, frame: Dict) -> Dict:
        rgb_t   = _train_transform(Image.fromarray(frame["rgb_np"]))
        rgb_raw = torch.from_numpy(frame["rgb_np"]).permute(2, 0, 1).float() / 255.0
        K = frame["K"]
        K_t   = torch.from_numpy(K).float() if K is not None else torch.from_numpy(_DEFAULT_K.copy())
        has_K = torch.tensor(K is not None, dtype=torch.bool)
        out = {"rgb": rgb_t, "rgb_raw": rgb_raw, "K": K_t, "has_K": has_K,
               "rgb_path": frame["rgb_path"]}
        if frame["depth"] is not None:
            out["depth_gt"]   = torch.from_numpy(frame["depth"]).unsqueeze(0).float()
            out["valid_mask"] = torch.from_numpy(frame["valid"].astype(np.float32)).unsqueeze(0)
            out["depth_path"] = frame["depth_path"]
        return out

    def _flip_frame(self, frame: Dict) -> None:
        W = self.target_hw[1]
        frame["rgb_np"] = np.ascontiguousarray(frame["rgb_np"][:, ::-1])
        if frame["depth"] is not None:
            frame["depth"] = np.ascontiguousarray(frame["depth"][:, ::-1])
            frame["valid"] = np.ascontiguousarray(frame["valid"][:, ::-1])
        if frame["K"] is not None:                               # principal point mirrors
            frame["K"][0, 2] = (W - 1) - frame["K"][0, 2]

    def __len__(self):
        return len(self._samples)

    def _aug_params(self) -> Dict:
        if not self.augment:
            return dict(flip=False, brightness=1.0, contrast=1.0, jitter=False)
        rng = np.random.RandomState(int.from_bytes(os.urandom(4), "little"))
        return _draw_aug_params(self.aug_mode, rng)

    def __getitem__(self, idx):
        recs = self._samples[idx]
        frames = {0: self._load_frame(recs[0], load_depth=self.load_gt)}
        for off in self.offsets:
            frames[off] = self._load_frame(recs[off], load_depth=(self.load_gt and off == +1))

        ap = self._aug_params()
        for fr in frames.values():                               # IDENTICAL params for every frame
            if ap["flip"]:
                self._flip_frame(fr)
            if ap["jitter"]:
                fr["rgb_np"] = _apply_color_jitter(fr["rgb_np"], ap["brightness"], ap["contrast"])
        if self.augment:                                         # keep valid ⊂ (depth in range)
            for fr in frames.values():
                if fr["depth"] is not None:
                    fr["valid"] = (fr["depth"] > self.min_depth) & (fr["depth"] < self.max_depth) & fr["valid"]

        result = self._to_tensors(frames[0])
        for off in self.offsets:
            t = self._to_tensors(frames[off])
            result.update({f"{_OFFSET_PREFIX[off]}_{k}": v for k, v in t.items()})
        result["flipped"] = torch.tensor(ap["flip"], dtype=torch.bool)
        return result


def collate_fn(batch):
    from torch.utils.data.dataloader import default_collate
    tensor_keys = [k for k in batch[0] if isinstance(batch[0][k], torch.Tensor)]
    str_keys    = [k for k in batch[0] if isinstance(batch[0][k], str)]
    out = {k: default_collate([b[k] for b in batch]) for k in tensor_keys}
    out.update({k: [b[k] for b in batch] for k in str_keys})
    return out


print("Dataset helpers ready (multi-frame sequences).")
print(f"  KITTISequentialDepthDataset -> (I_t, sources at offsets {SOURCE_OFFSETS})")
print("  flip / colour-jitter are drawn once per sample; K is mirrored on flip.")


Dataset helpers ready (multi-frame sequences).
  KITTISequentialDepthDataset -> (I_t, sources at offsets (-1, 1))
  flip / colour-jitter are drawn once per sample; K is mirrored on flip.


## 3 · GPS / IMU Ground-Truth Pose from OXTS  *(oracle-pose experiments only)*

KITTI records raw GPS/IMU data from an **Oxford Technical Solutions RT3003** unit. Each `oxts/data/NNNNNNNNNN.txt` has 30 fields (`lat, lon, alt, roll, pitch, yaw, …`). Lat/lon are converted to a flat-earth Mercator frame and RPY to SO(3) (Z-Y-X), giving the IMU pose `W_k` (IMU_k → IMU_0).

### Why an oracle pose?
It separates *"is the photometric / geometric objective informative for depth?"* from *"is PoseNet accurate?"*: with a metrically exact pose the only unknown in the warp is depth. **Oracle-pose runs are diagnostics, not self-supervised results** (they use extra sensor data).

### Calibration — v6 correction
OXTS poses live in the **IMU frame** (x forward, y left, z up); `warp_image` needs the **camera-02 frame** (x right, y down, z forward). With `T_ic = T_02_offset · R_rect_00 · T_velo→cam0 · T_imu→velo` (maps IMU coordinates to cam-02 coordinates):

```
T_imu  = inv(W_j) · W_i                       # relative pose, IMU axes
T_cam  = T_ic · T_imu · inv(T_ic)             # conjugation → camera axes; p_cam_j = T_cam · p_cam_i
```

v5 computed `inv(T_ic·W_j) · (T_ic·W_i)`, in which `T_ic` **cancels**, so its "camera-frame" pose was still in IMU axes (a 1 m forward drive gave `t = (−1, 0, 0)` instead of `(0, 0, −1)`). v6 verifies the fix numerically and, at run time, with **check 0a**: warping `I_{t+1}` into `I_t` with LiDAR depth + OXTS pose must beat both the identity pose and the inverse pose on LiDAR pixels.

There is **no silent fallback**: samples without OXTS data or calibration are dropped (and counted), the trainer asserts `has_gt_pose.all()`, and on a horizontal flip the pose is mirrored (`T' = F·T·F`, `F = diag(−1,1,1,1)`).


In [5]:
import math

# =============================================================================
# OXTS / GPS-IMU utilities
# =============================================================================
# Reference: KITTI raw data devkit (A. Geiger, P. Lenz, C. Stiller, R. Urtasun)
# Each oxts/data/NNNNNNNNNN.txt has 30 space-separated floats in the order
# defined by dataformat.txt (lat, lon, alt, roll, pitch, yaw, ...).
# =============================================================================

_EARTH_RADIUS_M = 6378137.0    # WGS-84 semi-major axis


def _latlon_to_mercator(lat_deg: float, lon_deg: float, scale: float) -> Tuple[float, float]:
    """Flat-earth Mercator projection: (lat, lon) in degrees -> (mx, my) in metres."""
    mx = scale * math.radians(lon_deg) * _EARTH_RADIUS_M
    my = scale * _EARTH_RADIUS_M * math.log(
        math.tan(math.pi / 4.0 + math.radians(lat_deg) / 2.0))
    return mx, my


def _rpy_to_rotmat_np(roll: float, pitch: float, yaw: float) -> np.ndarray:
    """Z-Y-X Euler angles -> rotation matrix R = Rz(yaw) @ Ry(pitch) @ Rx(roll)."""
    cr, sr = math.cos(roll),  math.sin(roll)
    cp, sp = math.cos(pitch), math.sin(pitch)
    cy, sy = math.cos(yaw),   math.sin(yaw)
    return np.array([
        [ cy*cp,  cy*sp*sr - sy*cr,  cy*sp*cr + sy*sr],
        [ sy*cp,  sy*sp*sr + cy*cr,  sy*sp*cr - cy*sr],
        [-sp,     cp*sr,             cp*cr            ],
    ], dtype=np.float64)


def _parse_oxts_file(path: str) -> Optional[List[float]]:
    """Parse one OXTS data file -> list of 30 floats, or None on error."""
    try:
        vals = [float(v) for v in Path(path).read_text().strip().split()]
        return vals if len(vals) >= 6 else None
    except Exception:
        return None


def _oxts_packet_to_world_se3(pkt: List[float], scale: float) -> np.ndarray:
    """Convert one OXTS packet to a 4x4 SE(3) in flat-earth Mercator world frame."""
    lat, lon, alt, roll, pitch, yaw = pkt[:6]
    mx, my = _latlon_to_mercator(lat, lon, scale)
    R = _rpy_to_rotmat_np(roll, pitch, yaw)
    T = np.eye(4, dtype=np.float64)
    T[:3, :3] = R
    T[:3,  3] = [mx, my, alt]
    return T


def _load_imu_to_cam02(date_dir: Path) -> Optional[np.ndarray]:
    """
    Compose T_cam02_from_imu from KITTI calibration files found at <date_dir>/:
      calib_imu_to_velo.txt  -> Tr_imu_to_velo  [4x4]
      calib_velo_to_cam.txt  -> Tr_velo_to_cam0 [4x4]
      calib_cam_to_cam.txt   -> R_rect_00 + P_rect_02

    Full chain (maps a point given in IMU coordinates to cam02 coordinates):
      T_cam02_from_imu = T_02_offset @ R_rect_00 @ Tr_velo_to_cam0 @ Tr_imu_to_velo

    Returns None if any file is absent or cannot be parsed.
    """
    def _kv(path: Path) -> Dict[str, np.ndarray]:
        d: Dict[str, np.ndarray] = {}
        for line in path.read_text().splitlines():
            if ":" in line:
                k, v = line.split(":", 1)
                try:
                    d[k.strip()] = np.array(v.split(), dtype=np.float64)
                except Exception:
                    pass
        return d

    try:
        f_iv = date_dir / "calib_imu_to_velo.txt"
        f_vc = date_dir / "calib_velo_to_cam.txt"
        f_cc = date_dir / "calib_cam_to_cam.txt"
        if not all(f.exists() for f in [f_iv, f_vc, f_cc]):
            return None

        kv_iv = _kv(f_iv); kv_vc = _kv(f_vc); kv_cc = _kv(f_cc)

        # Tr_imu->velo [4x4]
        Tr_iv = np.eye(4, dtype=np.float64)
        Tr_iv[:3, :3] = kv_iv["R"].reshape(3, 3)
        Tr_iv[:3,  3] = kv_iv["T"].ravel()

        # Tr_velo->cam0 [4x4]
        Tr_vc = np.eye(4, dtype=np.float64)
        Tr_vc[:3, :3] = kv_vc["R"].reshape(3, 3)
        Tr_vc[:3,  3] = kv_vc["T"].ravel()

        # R_rect_00 (rectification of cam0)
        R_rect = np.eye(4, dtype=np.float64)
        R_rect[:3, :3] = kv_cc["R_rect_00"].reshape(3, 3)

        # T offset from cam0 to cam02 (from P_rect_02 baseline)
        P02 = kv_cc["P_rect_02"].reshape(3, 4)
        tx_offset = P02[0, 3] / (P02[0, 0] + 1e-12)
        T_02_offset = np.eye(4, dtype=np.float64)
        T_02_offset[0, 3] = tx_offset

        return T_02_offset @ R_rect @ Tr_vc @ Tr_iv

    except Exception as e:
        log.debug("[OXTS calib] %s: %s", date_dir, e)
        return None


def _load_oxts_indexed(oxts_data_dir: Path) -> Dict[int, np.ndarray]:
    """
    Load all OXTS frames from <drive>/oxts/data/ and return {frame_id: W_k}
    where W_k is the 4x4 pose of the IMU at frame k in the drive-local world
    frame (IMU_k -> IMU_0, i.e.  p_imu0 = W_k @ p_imu_k).  Everything here is
    in the IMU body frame (x fwd, y left, z up); the conversion to the CAMERA
    frame happens in `_relative_cam_pose`.

    v6 FIX (found while auditing the GT-pose path): v5 multiplied the local
    poses by T_imu_to_cam and then formed inv(P_j) @ P_i.  The calibration
    matrix cancels algebraically in that product, so v5's "camera-frame"
    relative pose was really the relative pose in the IMU frame (x fwd) while
    `warp_image` expects the camera frame (z fwd).  See `_relative_cam_pose`.
    """
    txt_files = sorted(glob.glob(str(oxts_data_dir / "*.txt")))
    if not txt_files:
        return {}
    frame_ids, packets = [], []
    for fpath in txt_files:
        pkt = _parse_oxts_file(fpath)
        if pkt is not None:
            frame_ids.append(int(Path(fpath).stem))
            packets.append(pkt)
    if not packets:
        return {}
    scale = math.cos(math.radians(packets[0][0]))                  # Mercator scale from 1st latitude
    world = [_oxts_packet_to_world_se3(p, scale) for p in packets]
    T0_inv = np.linalg.inv(world[0])
    return {fid: T0_inv @ W for fid, W in zip(frame_ids, world)}


def _relative_cam_pose(W_i: np.ndarray, W_j: np.ndarray, T_imu_to_cam: np.ndarray) -> np.ndarray:
    """
    Relative pose  T_{i->j}  in the CAMERA-02 frame:  p_cam_j = T @ p_cam_i.

        p_imu_j = inv(W_j) @ W_i @ p_imu_i          (IMU frame, from OXTS)
        p_cam_k = T_ic @ p_imu_k                    (T_ic = T_imu_to_cam)
      =>  T_cam = T_ic @ inv(W_j) @ W_i @ inv(T_ic)  (a CONJUGATION, not a product)

    This is exactly the `pose_curr_to_next` convention consumed by
    `Project3D` / `warp_image` (maps coordinates of frame i into frame j).
    """
    T_imu = np.linalg.inv(W_j) @ W_i
    return T_imu_to_cam @ T_imu @ np.linalg.inv(T_imu_to_cam)


_FLIP_X = np.diag([-1.0, 1.0, 1.0, 1.0]).astype(np.float64)


def _flip_pose(T: np.ndarray) -> np.ndarray:
    """Pose of the mirrored scene under a horizontal image flip: T' = F T F, F = diag(-1,1,1,1)."""
    return _FLIP_X @ T @ _FLIP_X


# =============================================================================
# Dataset subclass: KITTISequentialOXTSDataset  (ORACLE-POSE experiments only)
# =============================================================================

class KITTISequentialOXTSDataset(KITTISequentialDepthDataset):
    """
    Adds GPS/IMU ground-truth relative poses (camera-02 frame, `pose_curr_to_src`)
    to every sample of KITTISequentialDepthDataset.

    Extra batch keys
        gt_pose_next : [4,4]  T_{t -> t+1}
        gt_pose_prev : [4,4]  T_{t -> t-1}          (only if -1 in source_offsets)
        has_gt_pose  : bool   (always True when require_gt_pose=True)

    v6 behaviour
      * There is NO silent identity/None fallback.  With `require_gt_pose=True`
        (default) samples whose OXTS pose or calibration is missing are DROPPED
        (and counted in the log) so that every remaining sample has GT pose.
      * The trainer additionally asserts `has_gt_pose.all()`.
      * If a drive has no calibration files, its samples are dropped — the old
        "IMU frame (approx)" behaviour is gone because it was in the wrong axes.
      * On a horizontal flip the pose is mirrored (T' = F T F).
    """

    def __init__(self, *args, oxts_subdir: str = "oxts", require_gt_pose: bool = True, **kwargs):
        self._oxts_subdir     = oxts_subdir
        self._require_gt_pose = require_gt_pose
        self._drive_world: Dict[Tuple, Dict[int, np.ndarray]] = {}
        self._drive_Tic:   Dict[Tuple, np.ndarray] = {}
        self._n_dropped = 0
        super().__init__(*args, **kwargs)          # calls _build_samples via polymorphism
        log.info("[OXTSDataset] %d drives with GT pose | %d samples kept | %d dropped (no GT pose)",
                 len(self._drive_world), len(self._samples), self._n_dropped)

    def _build_drive_index(self) -> Dict:
        by_drive = super()._build_drive_index()
        for (date, drive_name), frames in by_drive.items():
            if not frames:
                continue
            rgb_0 = Path(frames[0]["rgb_path"])
            try:
                parts = list(rgb_0.parts)
                img02_idx = parts.index("image_02")
                drive_dir = Path(*parts[:img02_idx])
                date_dir  = drive_dir.parent
            except (ValueError, IndexError):
                continue
            oxts_data_dir = drive_dir / self._oxts_subdir / "data"
            if not oxts_data_dir.exists():
                log.warning("[OXTS] no oxts dir for %s/%s", date, drive_name)
                continue
            T_ic = _load_imu_to_cam02(date_dir)
            if T_ic is None:
                log.warning("[OXTS] calibration files missing for %s -> drive %s excluded from oracle runs",
                            date, drive_name)
                continue
            try:
                self._drive_world[(date, drive_name)] = _load_oxts_indexed(oxts_data_dir)
                self._drive_Tic[(date, drive_name)]   = T_ic
            except Exception as e:
                log.warning("[OXTS] failed %s/%s: %s", date, drive_name, e)
        return by_drive

    def _build_samples(self):
        for (date, drive_name), frames in self._build_drive_index().items():
            world = self._drive_world.get((date, drive_name), {})
            T_ic  = self._drive_Tic.get((date, drive_name))
            by_id = {r["frame_id"]: r for r in frames}
            for r in frames:
                fid = r["frame_id"]
                if not all((fid + off) in by_id for off in self.offsets):
                    continue
                gt = {}
                ok = (T_ic is not None) and (fid in world)
                if ok:
                    for off in self.offsets:
                        if (fid + off) not in world:
                            ok = False
                            break
                        gt[off] = _relative_cam_pose(world[fid], world[fid + off], T_ic)
                if not ok and self._require_gt_pose:
                    self._n_dropped += 1
                    continue
                s = {0: r}
                for off in self.offsets:
                    s[off] = by_id[fid + off]
                s["gt"] = gt if ok else None
                self._samples.append(s)

    def __getitem__(self, idx: int) -> Dict:
        result = super().__getitem__(idx)
        gt = self._samples[idx].get("gt")
        flipped = bool(result["flipped"])
        if gt is None:
            # only reachable with require_gt_pose=False -> explicit flag, NO fake pose is used by the trainer
            for off in self.offsets:
                result[f"gt_pose_{_OFFSET_PREFIX[off]}"] = torch.eye(4, dtype=torch.float32)
            result["has_gt_pose"] = torch.tensor(False, dtype=torch.bool)
            return result
        for off in self.offsets:
            T = gt[off]
            T = _flip_pose(T) if flipped else T
            result[f"gt_pose_{_OFFSET_PREFIX[off]}"] = torch.from_numpy(T.astype(np.float32))
        result["has_gt_pose"] = torch.tensor(True, dtype=torch.bool)
        return result


def sample_ids(ds) -> List[str]:
    """Canonical id of every sample = RGB path of the CURRENT frame."""
    return [s[0]["rgb_path"] for s in ds._samples]


def assert_same_samples(ds_a, idx_a, ds_b, idx_b, what: str = "") -> None:
    """Oracle-pose and predicted-pose runs are only comparable (B vs C) if they see EXACTLY the same images."""
    ids_a = [sample_ids(ds_a)[i] for i in idx_a]
    ids_b = [sample_ids(ds_b)[i] for i in idx_b]
    assert ids_a == ids_b, (f"{what}: the SSL and oracle-pose datasets do not contain identical samples "
                            f"({len(ids_a)} vs {len(ids_b)}). Set CANONICAL_SAMPLE_SET=True.")


def probe_oxts_sample_ids(train_root: str) -> set:
    """Ids of all samples that have complete OXTS GT pose + calibration (the canonical sample set)."""
    probe = KITTISequentialOXTSDataset(train_root, split="val", aug_mode="none", max_samples=0,
                                       load_gt=False, require_gt_pose=True)
    return set(sample_ids(probe))


print("OXTS utilities ready (v6).")
print("  _relative_cam_pose          -> T_cam = T_ic @ inv(W_j) @ W_i @ inv(T_ic)   (frame-correct)")
print("  KITTISequentialOXTSDataset  -> gt_pose_next / gt_pose_prev, no silent fallback")
print("  probe_oxts_sample_ids / assert_same_samples -> canonical sample set shared with the SSL loaders")


OXTS utilities ready (v6).
  _relative_cam_pose          -> T_cam = T_ic @ inv(W_j) @ W_i @ inv(T_ic)   (frame-correct)
  KITTISequentialOXTSDataset  -> gt_pose_next / gt_pose_prev, no silent fallback
  probe_oxts_sample_ids / assert_same_samples -> canonical sample set shared with the SSL loaders


## 4 · SAM3 Instance Masks & Multi-Object Tracking (Part 3)

### Purpose

The new stationary-object losses (Part 5) need to know **which pixels belong to the same physical object across consecutive frames**. SAM3 gives per-frame instance masks with no correspondence between frames — frame $t$'s "car #3" and frame $t{+}1$'s "car #3" are unrelated indices. This section adds three tracking algorithms of increasing sophistication that solve that correspondence problem, so downstream losses can say "this mask in frame $t$ and that mask in frame $t{+}1$ are the same physical object."

### Mathematical formulation shared by all three trackers

Each tracker solves a **bipartite assignment** between the set of tracks alive from frame $t{-}1$, $\mathcal{T} = \{\tau_1, ..., \tau_N\}$, and the set of new detections in frame $t$, $\mathcal{D} = \{d_1, ..., d_M\}$. Mask IoU is the primary geometric cue:

$$\text{IoU}(\tau_i, d_j) = \frac{|\text{mask}(\tau_i) \cap \text{mask}(d_j)|}{|\text{mask}(\tau_i) \cup \text{mask}(d_j)|}$$

Trackers 1–2 differ only in **how** the assignment is solved (greedy vs. optimal); Tracker 3 additionally changes **which detections are allowed to spawn new tracks**.

### Implementation details

- A cheap bbox-overlap pre-check (`_bbox_iou`) gates the expensive full-resolution `logical_and`/`logical_or` mask IoU computation, since KITTI masks can be ~375×1242 and pairwise IoU across all track×detection pairs would otherwise dominate the frame budget.
- An optional appearance descriptor (`_color_histogram_embedding`, a 48-D HSV histogram inside the mask) is used by Trackers 2–3 when available — SAM3 does not expose a learned Re-ID embedding by default, so this is the practical substitute referenced in the spec ("optional appearance embedding").
- Tracker 3 additionally carries a lightweight constant-velocity **Kalman filter** per track (`_ConstantVelocityKF`) over the 2-D centroid, so the assignment cost is computed against a *predicted* position rather than the last-observed one — this is what lets ByteTrack-style tracking survive a frame or two of partial occlusion.

### Computational complexity

- Tracker 1 (greedy): $O(NM)$ cost construction + $O(NM \log NM)$ sort.
- Tracker 2 (Hungarian): $O(NM)$ cost construction + $O(\min(N,M)^2 \max(N,M))$ Kuhn–Munkres solve (`scipy.optimize.linear_sum_assignment`).
- Tracker 3 (ByteTrack-style): two Hungarian solves per frame (high-score then low-score detections), same asymptotic cost as Tracker 2.

### Limitations

All three are online, appearance-light, nearest-neighbour trackers — none maintains a long-term Re-ID gallery, so identity can still be lost across occlusions longer than `max_age` frames. For the intended use (short-baseline consecutive-frame KITTI pairs, not long video), this is an acceptable trade-off against the cost of a full Re-ID network.

### References
- Bewley et al., *"Simple Online and Realtime Tracking" (SORT)*, ICIP 2016.
- Wojke et al., *"Simple Online and Realtime Tracking with a Deep Association Metric" (DeepSORT)*, ICIP 2017.
- Zhang et al., *"ByteTrack: Multi-Object Tracking by Associating Every Detection Box"*, ECCV 2022.


In [6]:
# =============================================================================
# SAM3 mask loading + Detection/Track data structures (Part 3)
# =============================================================================

@dataclass
class MaskDetection:
    """One SAM3 instance mask, resolved to the geometric quantities every
    tracker needs (bbox / centroid / area are cached at construction time so
    trackers never recompute them from the raw mask on every comparison)."""
    mask: np.ndarray                              # bool [H, W]
    bbox: Tuple[float, float, float, float]        # (x1, y1, x2, y2)
    centroid: Tuple[float, float]                  # (cx, cy) in pixel coords
    area: float                                    # mask pixel count
    class_name: str = "unknown"
    score: float = 1.0
    embedding: Optional[np.ndarray] = None         # optional appearance descriptor


@dataclass
class Track:
    """A tracked object, persisted across frames by all three trackers below."""
    track_id: int
    mask: np.ndarray
    bbox: Tuple[float, float, float, float]
    centroid: Tuple[float, float]
    area: float
    class_name: str
    score: float
    embedding: Optional[np.ndarray] = None
    age: int = 0                                   # frames since track creation
    hits: int = 1                                  # number of matched detections total
    time_since_update: int = 0                     # frames since last matched detection
    history: List[Tuple[float, float]] = field(default_factory=list)


def _bbox_iou(b1: Tuple[float, float, float, float],
              b2: Tuple[float, float, float, float]) -> float:
    """Cheap axis-aligned bbox IoU, used as a quick reject before the
    expensive full-resolution mask IoU below."""
    x1, y1, x2, y2 = b1
    X1, Y1, X2, Y2 = b2
    ix1, iy1 = max(x1, X1), max(y1, Y1)
    ix2, iy2 = min(x2, X2), min(y2, Y2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    a1 = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    a2 = max(0.0, X2 - X1) * max(0.0, Y2 - Y1)
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0.0


def mask_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Exact pixel-wise IoU between two boolean masks."""
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return float(inter) / float(union) if union > 0 else 0.0


def _color_histogram_embedding(rgb_uint8: np.ndarray, mask: np.ndarray,
                                bins: int = 16) -> np.ndarray:
    """Cheap appearance descriptor: normalized per-channel HSV histogram
    inside the mask. Serves as the 'optional appearance embedding' term in
    the Hungarian / ByteTrack cost functions when no learned Re-ID network
    is available (SAM3 does not expose one by default)."""
    hsv = cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2HSV)
    pix = hsv[mask]
    if pix.shape[0] == 0:
        return np.zeros(bins * 3, dtype=np.float32)
    hist = [np.histogram(pix[:, c], bins=bins, range=(0, 256))[0].astype(np.float32)
            for c in range(3)]
    v = np.concatenate(hist)
    return v / (v.sum() + 1e-6)


def _mask_to_detection(mask: np.ndarray, class_name: str = "unknown",
                        score: float = 1.0,
                        rgb: Optional[np.ndarray] = None) -> Optional[MaskDetection]:
    """Builds a MaskDetection from a raw boolean mask. Returns None for an
    empty mask (can happen after resizing/cropping a SAM3 export)."""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    x1, x2 = float(xs.min()), float(xs.max())
    y1, y2 = float(ys.min()), float(ys.max())
    cx, cy = float(xs.mean()), float(ys.mean())
    area = float(mask.sum())
    emb = _color_histogram_embedding(rgb, mask) if rgb is not None else None
    return MaskDetection(mask=mask, bbox=(x1, y1, x2, y2), centroid=(cx, cy),
                         area=area, class_name=class_name, score=float(score),
                         embedding=emb)

# Mask helpers
def get_mask_dir(rgb_path: str, mask_root: str, kitti_root: str) -> str:
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), kitti_root)
    return os.path.join(mask_root, rel_data_dir, frame_stem)

@lru_cache(maxsize=None)          # from 4096 unsure about the size so not caching the masks
def load_masks_for_frame(mask_dir: str) -> List[Dict]:
    p = Path(mask_dir)
    if not p.exists(): return []
    stem = p.name; manifest = p / f"{stem}_manifest.json"
    instances = []
    if manifest.exists():
        for rec in json.loads(manifest.read_text()):
            mf = p / rec["mask_file"]
            if not mf.exists(): continue
            instances.append({
                "uid": rec["uid"], "local_idx": rec["local_idx"],
                "super_cat": rec["super_cat"], "sub_cat": rec["sub_cat"],
                "score": float(rec["score"]), "box": rec.get("box"),
                "mask": np.array(Image.open(mf).convert("L")) > 127, "path": str(mf),
            })
        return instances
    for mf in sorted(p.glob("*_mask.png")):
        s = mf.stem[:-5]
        if not s.startswith(stem+"_"): continue
        parts = s[len(stem)+1:].rsplit("_", 3)
        if len(parts) != 4: continue
        cat_str, idx_str, uid_str, score_str = parts
        cp = cat_str.split("_", 1)
        if len(cp) != 2: continue
        super_cat, sub_cat = cp
        try:
            local_idx = int(idx_str); score = float(score_str)
        except ValueError: continue
        instances.append({
            "uid": uid_str, "local_idx": local_idx, "super_cat": super_cat,
            "sub_cat": sub_cat, "score": score, "box": None,
            "mask": np.array(Image.open(mf).convert("L")) > 127, "path": str(mf),
        })
    instances.sort(key=lambda x: x["local_idx"]); return instances
# ─────────────────────────────────────────────────────────────────────────────

def load_sam3_masks_for_frame(
    rgb_path: str,
    rgb_np: Optional[np.ndarray] = None
) -> List[MaskDetection]:
    """
    Load pre-generated SAM3 instance masks for one KITTI RGB frame and
    convert them into the MaskDetection objects expected by the trackers.

    SAM3 export layout:

        <SAM3_MASK_DIR>/
            train/
                <date>/
                    <drive>/
                        image_02/
                            data/
                                <frame_id>/
                                    <frame_id>_manifest.json
                                    <frame_id>_overview.jpg
                                    <frame_id>_*_mask.png

    Example:

        ~/Downloads/MMDE/outputs/inst_seg_output/train/
            2011_09_26/
                2011_09_26_drive_0001_sync/
                    image_02/
                        data/
                            0000000005/
                                0000000005_manifest.json
                                0000000005_overview.jpg
                                0000000005_road_road_surface_0000_..._mask.png

    The actual on-disk format is handled by:

        get_mask_dir(...)
        load_masks_for_frame(...)

    This function only adapts those loaded instance dictionaries into
    MaskDetection objects used by the tracking code.

    Args:
        rgb_path:
            Path to the corresponding KITTI RGB frame.

        rgb_np:
            Optional RGB uint8 image with shape [H, W, 3].
            When supplied, a HSV color-histogram appearance embedding
            is calculated for each mask.

    Returns:
        List[MaskDetection]:
            One MaskDetection per valid SAM3 instance.
            Returns [] when the mask directory does not exist or no valid
            masks can be loaded.
    """

    try:
        # ------------------------------------------------------------------
        # 1. Resolve the frame-specific SAM3 mask directory.
        #
        # Example result:
        #
        # ~/Downloads/MMDE/outputs/inst_seg_output/train/
        #     2011_09_26/
        #     2011_09_26_drive_0001_sync/
        #     image_02/data/
        #     0000000005/
        # ------------------------------------------------------------------
        mask_root = str(Path(SAM3_MASK_DIR).expanduser())
        kitti_root = str(Path(KITTI_ROOT).expanduser())

        mask_dir = get_mask_dir(
            rgb_path=rgb_path,
            mask_root=mask_root,
            kitti_root=kitti_root,
        )

        # ------------------------------------------------------------------
        # 2. Load the masks using the actual SAM3 directory/manifest format.
        # ------------------------------------------------------------------
        instances = load_masks_for_frame(mask_dir)

        if not instances:
            return []

    except Exception as e:
        log.warning(
            "[SAM3] failed to load masks for frame %s: %s",
            rgb_path,
            e,
        )
        return []

    # ----------------------------------------------------------------------
    # 3. Convert every loaded SAM3 instance into a MaskDetection.
    # ----------------------------------------------------------------------
    detections: List[MaskDetection] = []

    for inst in instances:
        try:
            # Mask returned by load_masks_for_frame() is already boolean,
            # but np.asarray(..., dtype=bool) makes this explicit and safe.
            mask = np.asarray(inst["mask"], dtype=bool)
            if mask.shape[:2] != (TARGET_H, TARGET_W):              # added a check to match and correct the target height and target width of mask and image
                mask = cv2.resize(
                    mask.astype(np.uint8), (TARGET_W, TARGET_H),
                    interpolation=cv2.INTER_NEAREST,
                ).astype(bool)

            # --------------------------------------------------------------
            # Build the tracker-facing class label.
            #
            # Example:
            #   super_cat = "road"
            #   sub_cat   = "road_surface"
            #
            # becomes:
            #   "road/road_surface"
            # --------------------------------------------------------------
            super_cat = str(inst.get("super_cat", "")).strip()
            sub_cat = str(inst.get("sub_cat", "")).strip()

            if super_cat and sub_cat:
                class_name = f"{super_cat}/{sub_cat}"
            elif sub_cat:
                class_name = sub_cat
            elif super_cat:
                class_name = super_cat
            else:
                class_name = "unknown"

            score = float(inst.get("score", 1.0))

            # --------------------------------------------------------------
            # Convert raw mask -> MaskDetection.
            #
            # _mask_to_detection() calculates:
            #   bbox
            #   centroid
            #   area
            #   optional appearance embedding
            # --------------------------------------------------------------
            det = _mask_to_detection(
                mask=mask,
                class_name=class_name,
                score=score,
                rgb=rgb_np,
            )

            # Empty masks are ignored because _mask_to_detection()
            # returns None for them.
            if det is not None:
                detections.append(det)

        except Exception as e:
            log.warning(
                "[SAM3] failed to convert mask instance from %s: %s",
                mask_dir,
                e,
            )
            continue

    return detections

#__________________________________________________________________________________________________________

# =============================================================================
# Tracker 1 — Mask IoU Tracker (greedy)
# =============================================================================

class MaskIoUTracker:
    """
    Greedy frame-to-frame association using a weighted combination of:
      • mask IoU                     (primary geometric cue)
      • normalized centroid distance (handles small-IoU-but-same-object cases
                                       from motion / segmentation jitter)
      • area ratio                   (penalizes matches with drastically
                                       different scale, e.g. mis-segmentation)

    Matching is greedy (highest combined score first), not globally optimal —
    a faithful mask-based adaptation of the association step in SORT
    (Bewley et al., ICRA 2016). Cheapest of the three trackers.

    Limitation: greedy matching can commit to a locally-optimal but globally
    suboptimal assignment when many similar objects are close together
    (e.g. a row of parked cars); Tracker 2 resolves this with an optimal solve.
    """

    def __init__(self, iou_weight: float = 0.6, centroid_weight: float = 0.25,
                 area_weight: float = 0.15, iou_match_threshold: float = TRACK_IOU_MATCH_THRESH,
                 max_age: int = TRACK_MAX_AGE, img_diag: float = 1000.0):
        self.iou_weight, self.centroid_weight, self.area_weight = iou_weight, centroid_weight, area_weight
        self.iou_match_threshold = iou_match_threshold
        self.max_age = max_age
        self.img_diag = img_diag
        self.tracks: List[Track] = []
        self._next_id = 1

    def _score(self, trk: Track, det: MaskDetection) -> float:
        iou = mask_iou(trk.mask, det.mask) if _bbox_iou(trk.bbox, det.bbox) > 0 else 0.0
        cx1, cy1 = trk.centroid
        cx2, cy2 = det.centroid
        centroid_sim = max(0.0, 1.0 - math.hypot(cx1 - cx2, cy1 - cy2) / self.img_diag)
        area_ratio = min(trk.area, det.area) / (max(trk.area, det.area) + 1e-6)
        return (self.iou_weight * iou + self.centroid_weight * centroid_sim +
                self.area_weight * area_ratio)

    def update(self, detections: List[MaskDetection]) -> List[Track]:
        matched_trk, matched_det = set(), set()

        if self.tracks and detections:
            pairs = [(self._score(trk, det), ti, di)
                     for ti, trk in enumerate(self.tracks)
                     for di, det in enumerate(detections)]
            pairs = [p for p in pairs if p[0] > 0]
            pairs.sort(key=lambda x: -x[0])
            for s, ti, di in pairs:
                if ti in matched_trk or di in matched_det or s < self.iou_match_threshold:
                    continue
                self._apply_match(ti, detections[di])
                matched_trk.add(ti)
                matched_det.add(di)

        for ti, trk in enumerate(self.tracks):
            if ti not in matched_trk:
                trk.age += 1
                trk.time_since_update += 1

        for di, det in enumerate(detections):
            if di not in matched_det:
                self._spawn_track(det)

        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]
        return self.tracks

    def _apply_match(self, ti: int, det: MaskDetection) -> None:
        trk = self.tracks[ti]
        trk.mask, trk.bbox, trk.centroid = det.mask, det.bbox, det.centroid
        trk.area, trk.class_name, trk.score, trk.embedding = det.area, det.class_name, det.score, det.embedding
        trk.age += 1
        trk.hits += 1
        trk.time_since_update = 0
        trk.history.append(det.centroid)

    def _spawn_track(self, det: MaskDetection) -> None:
        self.tracks.append(Track(track_id=self._next_id, mask=det.mask, bbox=det.bbox,
                                  centroid=det.centroid, area=det.area, class_name=det.class_name,
                                  score=det.score, embedding=det.embedding, history=[det.centroid]))
        self._next_id += 1


# =============================================================================
# Tracker 2 — Hungarian Assignment Tracker (optimal)
# =============================================================================

class HungarianMaskTracker:
    """
    Formulates frame-to-frame association as a linear assignment problem,
    solved optimally (Kuhn–Munkres) via scipy.optimize.linear_sum_assignment:

        minimize  sum_{i,j} C[i,j] * x[i,j]   s.t. each track <-> at most one detection

    Cost matrix combines four normalized terms in [0, 1]:

        C = w_iou * (1 - IoU) + w_centroid * (dist / img_diag)
            + w_area * (1 - area_ratio) + w_appearance * (1 - cosine_sim(embedding))

    The appearance term is skipped (weight renormalized across the other
    three) when either side has no embedding.

    Unlike Tracker 1's greedy matching, this always finds the assignment that
    minimizes TOTAL cost across all pairs simultaneously — important when
    several similar stationary objects sit next to each other (e.g. parked
    cars), where greedy IoU tracking can swap identities.
    """

    def __init__(self, w_iou: float = 0.5, w_centroid: float = 0.25, w_area: float = 0.15,
                 w_appearance: float = 0.10, max_cost: float = 0.85,
                 max_age: int = TRACK_MAX_AGE, img_diag: float = 1000.0):
        self.w_iou, self.w_centroid, self.w_area, self.w_appearance = w_iou, w_centroid, w_area, w_appearance
        self.max_cost = max_cost
        self.max_age = max_age
        self.img_diag = img_diag
        self.tracks: List[Track] = []
        self._next_id = 1

    def _cost(self, trk: Track, det: MaskDetection) -> float:
        iou = mask_iou(trk.mask, det.mask) if _bbox_iou(trk.bbox, det.bbox) > 0 else 0.0
        cx1, cy1 = trk.centroid
        cx2, cy2 = det.centroid
        dist_n = min(1.0, math.hypot(cx1 - cx2, cy1 - cy2) / self.img_diag)
        area_ratio = min(trk.area, det.area) / (max(trk.area, det.area) + 1e-6)

        w_iou, w_c, w_a = self.w_iou, self.w_centroid, self.w_area
        if trk.embedding is None or det.embedding is None:
            renorm = 1.0 / (w_iou + w_c + w_a + 1e-9)
            w_iou, w_c, w_a = w_iou * renorm, w_c * renorm, w_a * renorm
            appearance_term = 0.0
        else:
            num = float(np.dot(trk.embedding, det.embedding))
            den = float(np.linalg.norm(trk.embedding) * np.linalg.norm(det.embedding) + 1e-9)
            cos_sim = max(-1.0, min(1.0, num / den))
            appearance_term = self.w_appearance * (1.0 - cos_sim)

        return w_iou * (1.0 - iou) + w_c * dist_n + w_a * (1.0 - area_ratio) + appearance_term

    def update(self, detections: List[MaskDetection]) -> List[Track]:
        n_trk, n_det = len(self.tracks), len(detections)
        matched_trk, matched_det = set(), set()

        if n_trk > 0 and n_det > 0:
            C = np.full((n_trk, n_det), 10.0, dtype=np.float64)
            for ti, trk in enumerate(self.tracks):
                for di, det in enumerate(detections):
                    C[ti, di] = self._cost(trk, det)
            row_idx, col_idx = linear_sum_assignment(C)
            for ti, di in zip(row_idx, col_idx):
                if C[ti, di] > self.max_cost:
                    continue
                self._apply_match(ti, detections[di])
                matched_trk.add(ti)
                matched_det.add(di)

        for ti, trk in enumerate(self.tracks):
            if ti not in matched_trk:
                trk.age += 1
                trk.time_since_update += 1

        for di, det in enumerate(detections):
            if di not in matched_det:
                self._spawn_track(det)

        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]
        return self.tracks

    def _apply_match(self, ti: int, det: MaskDetection) -> None:
        trk = self.tracks[ti]
        trk.mask, trk.bbox, trk.centroid = det.mask, det.bbox, det.centroid
        trk.area, trk.class_name, trk.score, trk.embedding = det.area, det.class_name, det.score, det.embedding
        trk.age += 1
        trk.hits += 1
        trk.time_since_update = 0
        trk.history.append(det.centroid)

    def _spawn_track(self, det: MaskDetection) -> None:
        self.tracks.append(Track(track_id=self._next_id, mask=det.mask, bbox=det.bbox,
                                  centroid=det.centroid, area=det.area, class_name=det.class_name,
                                  score=det.score, embedding=det.embedding, history=[det.centroid]))
        self._next_id += 1


# =============================================================================
# Tracker 3 — ByteTrack-compatible Tracker (masks + Kalman-predicted centroid)
# =============================================================================

class _ConstantVelocityKF:
    """Minimal constant-velocity Kalman filter over 2-D centroid (cx, cy, vx, vy).
    Plays the role of the bbox Kalman filter in the original SORT/ByteTrack
    formulation, adapted to mask centroids since masks have no natural
    fixed-size box parametrization to filter."""

    def __init__(self, cx: float, cy: float):
        self.x = np.array([cx, cy, 0.0, 0.0], dtype=np.float64)
        self.P = np.eye(4) * 10.0
        self.F = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]], dtype=np.float64)
        self.H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], dtype=np.float64)
        self.Q = np.eye(4) * 1.0
        self.R = np.eye(2) * 4.0

    def predict(self) -> Tuple[float, float]:
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return float(self.x[0]), float(self.x[1])

    def update(self, cx: float, cy: float) -> None:
        z = np.array([cx, cy], dtype=np.float64)
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P


class ByteTrackMaskTracker:
    """
    Adapts Zhang et al., "ByteTrack: Multi-Object Tracking by Associating
    Every Detection Box" (ECCV 2022), to instance segmentation masks. The
    core idea preserved from the original: don't discard low-confidence
    detections before matching — they are frequently partially-occluded
    TRUE objects, and dropping them (as Trackers 1-2 implicitly do by only
    seeing high-confidence SAM3 output) causes ID switches and fragmented
    tracks.

    Two-stage association per frame:
      Stage 1 (detections with score >= high_thresh):
          Hungarian-matched to ALL existing tracks, cost computed against
          each track's Kalman-PREDICTED centroid (not its last observation).
      Stage 2 (low_thresh <= score < high_thresh):
          Hungarian-matched only to tracks still unmatched after Stage 1 —
          recovers tracks through brief partial occlusion without ever
          letting a low-confidence detection spawn a new track.
      New tracks are created ONLY from unmatched Stage-1 detections.
      Unmatched tracks are kept ("lost") for `max_age` frames, Kalman-predicted
      forward with no measurement update, before deletion.

    Complexity: O(NM) per stage (two Hungarian solves), same order as Tracker 2.
    Limitation: still a nearest-neighbour tracker with no long-term Re-ID
    gallery; identity can be lost across occlusions longer than max_age.
    """

    def __init__(self, high_thresh: float = BYTETRACK_HIGH_THRESH, low_thresh: float = BYTETRACK_LOW_THRESH,
                 w_iou: float = 0.6, w_centroid: float = 0.4, max_cost: float = 0.9,
                 max_age: int = TRACK_MAX_AGE, img_diag: float = 1000.0):
        self.high_thresh, self.low_thresh = high_thresh, low_thresh
        self.w_iou, self.w_centroid = w_iou, w_centroid
        self.max_cost = max_cost
        self.max_age = max_age
        self.img_diag = img_diag
        self.tracks: List[Track] = []
        self._kfs: Dict[int, _ConstantVelocityKF] = {}
        self._next_id = 1

    def _cost(self, trk: Track, pred_centroid: Tuple[float, float], det: MaskDetection) -> float:
        iou = mask_iou(trk.mask, det.mask) if _bbox_iou(trk.bbox, det.bbox) > 0 else 0.0
        dist_n = min(1.0, math.hypot(pred_centroid[0] - det.centroid[0],
                                     pred_centroid[1] - det.centroid[1]) / self.img_diag)
        return self.w_iou * (1.0 - iou) + self.w_centroid * dist_n

    def _hungarian_match(self, track_idxs: List[int], dets: List[MaskDetection],
                          pred_centroids: Dict[int, Tuple[float, float]]):
        if not track_idxs or not dets:
            return [], list(range(len(dets))), list(track_idxs)
        C = np.full((len(track_idxs), len(dets)), 10.0)
        for r, ti in enumerate(track_idxs):
            for c, det in enumerate(dets):
                C[r, c] = self._cost(self.tracks[ti], pred_centroids[ti], det)
        rows, cols = linear_sum_assignment(C)
        matches: List[Tuple[int, int]] = []
        unmatched_dets, unmatched_trks = set(range(len(dets))), set(track_idxs)
        for r, c in zip(rows, cols):
            if C[r, c] <= self.max_cost:
                matches.append((track_idxs[r], c))
                unmatched_dets.discard(c)
                unmatched_trks.discard(track_idxs[r])
        return matches, sorted(unmatched_dets), sorted(unmatched_trks)

    def update(self, detections: List[MaskDetection]) -> List[Track]:
        high = [d for d in detections if d.score >= self.high_thresh]
        low = [d for d in detections if self.low_thresh <= d.score < self.high_thresh]

        pred_centroids: Dict[int, Tuple[float, float]] = {}
        for ti, trk in enumerate(self.tracks):
            kf = self._kfs.get(trk.track_id)
            pred_centroids[ti] = kf.predict() if kf is not None else trk.centroid

        all_track_idx = list(range(len(self.tracks)))
        matched_this_frame = set()

        matches1, unmatched_high, unmatched_trk1 = self._hungarian_match(all_track_idx, high, pred_centroids)
        for ti, di in matches1:
            self._apply_match(ti, high[di])
            matched_this_frame.add(ti)

        matches2, unmatched_low, unmatched_trk2 = self._hungarian_match(unmatched_trk1, low, pred_centroids)
        for ti, di in matches2:
            self._apply_match(ti, low[di])
            matched_this_frame.add(ti)

        for ti, trk in enumerate(self.tracks):
            if ti not in matched_this_frame:
                trk.age += 1
                trk.time_since_update += 1

        for di in unmatched_high:
            self._spawn_track(high[di])

        alive = []
        for trk in self.tracks:
            if trk.time_since_update <= self.max_age:
                alive.append(trk)
            else:
                self._kfs.pop(trk.track_id, None)
        self.tracks = alive
        return self.tracks

    def _apply_match(self, ti: int, det: MaskDetection) -> None:
        trk = self.tracks[ti]
        trk.mask, trk.bbox, trk.centroid = det.mask, det.bbox, det.centroid
        trk.area, trk.class_name, trk.score, trk.embedding = det.area, det.class_name, det.score, det.embedding
        trk.age += 1
        trk.hits += 1
        trk.time_since_update = 0
        trk.history.append(det.centroid)
        kf = self._kfs.get(trk.track_id)
        if kf is not None:
            kf.update(*det.centroid)

    def _spawn_track(self, det: MaskDetection) -> None:
        new_trk = Track(track_id=self._next_id, mask=det.mask, bbox=det.bbox, centroid=det.centroid,
                         area=det.area, class_name=det.class_name, score=det.score,
                         embedding=det.embedding, history=[det.centroid])
        self.tracks.append(new_trk)
        self._kfs[new_trk.track_id] = _ConstantVelocityKF(*det.centroid)
        self._next_id += 1


def build_tracker(tracker_type: str = TRACKER_TYPE, img_diag: float = 1000.0):
    """Factory selecting one of the three trackers by TRACKER_TYPE config."""
    if tracker_type == "iou":
        return MaskIoUTracker(img_diag=img_diag)
    if tracker_type == "hungarian":
        return HungarianMaskTracker(img_diag=img_diag)
    if tracker_type == "bytetrack":
        return ByteTrackMaskTracker(img_diag=img_diag)
    raise ValueError(f"Unknown TRACKER_TYPE: {tracker_type}")


print("SAM3 mask loading + 3 trackers ready.")
print("  load_sam3_masks_for_frame -> List[MaskDetection] per frame (adapter, edit path logic if needed)")
print("  MaskIoUTracker            -> Tracker 1 (greedy)")
print("  HungarianMaskTracker      -> Tracker 2 (optimal assignment)")
print("  ByteTrackMaskTracker      -> Tracker 3 (two-stage + Kalman-predicted centroid)")
print("  build_tracker(TRACKER_TYPE) -> convenience factory used by the training loop")


SAM3 mask loading + 3 trackers ready.
  load_sam3_masks_for_frame -> List[MaskDetection] per frame (adapter, edit path logic if needed)
  MaskIoUTracker            -> Tracker 1 (greedy)
  HungarianMaskTracker      -> Tracker 2 (optimal assignment)
  ByteTrackMaskTracker      -> Tracker 3 (two-stage + Kalman-predicted centroid)
  build_tracker(TRACKER_TYPE) -> convenience factory used by the training loop


## 5 · Stationary vs. Moving Object Filter (Part 4)

### Purpose

The new losses in Part 5 assume every tracked object is **physically stationary**, so that its apparent depth/position change between frames is caused *entirely* by the camera's own ego-motion. If a moving car or pedestrian leaks into that pool, it injects a false ego-motion signal directly into the loss gradient. This section filters `Track` objects down to a trustworthy stationary subset before they reach any loss.

### Two independent filtering signals

**4a — Manual class filtering** (`is_class_dynamic`): SAM3's class label is checked against a curated `DYNAMIC_CLASS_NAMES` set (car, truck, bus, person, bicycle, motorcycle, rider, train, …). Unknown/unlisted classes default to **dynamic** (excluded) — for this use case, treating an unrecognized object as unsafe-to-trust is the conservative choice, since one wrongly-included moving object corrupts every stationary-consistency loss term.

**4b — Optical-flow residual filtering** (`is_object_stationary_by_flow`): Dense Farneback optical flow is computed between the frame pair. A truly static object's apparent flow is caused *only* by ego-motion (the camera moving past it), so once the scene's global ego-flow (estimated from a large static region — the union of class-static masks, or the upper/sky band as a fallback) is subtracted from the object's median flow, a genuinely stationary object should have residual ≈ 0 px. A moving object retains a non-zero residual. This catches the case class labels can't — e.g. a "parked" car that's actually pulling away, or a class label SAM3 got wrong.

### Why require both (`DYNAMIC_FILTER_MODE = "both"`)

Class filtering is fast but brittle to label noise; flow filtering is more direct but can be fooled by textureless surfaces (Farneback flow is unreliable on e.g. blank walls or the sky). Requiring an object to pass **both** checks trades recall for precision — for a loss term whose entire validity rests on the "this object didn't move" assumption, false positives are far more damaging than a few missed true stationary objects.

### Limitations

Optical-flow residual filtering degrades in low-texture regions and at night (KITTI is daytime-only, so this is a minor concern here); the ego-flow estimate is a median-based heuristic, not a solved fundamental-matrix / homography decomposition, so it can be biased when the static-mask union is small or absent (falls back to the image's upper quarter, which is usually sky/background in KITTI's forward-facing camera geometry).

> **v6.2.1 fix (Bug 2):** `is_class_dynamic()` now normalizes SAM3's compound `"super_cat/sub_cat"` labels (e.g. `"vehicle/car"`, `"traffic/traffic_sign"`) before the `STATIC_CLASS_NAMES` / `DYNAMIC_CLASS_NAMES` membership check. An exact match against the raw compound string almost never succeeded, so nearly every real object fell through to the conservative "unknown → dynamic" default regardless of its true class. `STATIC_CLASS_NAMES`, `DYNAMIC_CLASS_NAMES` and that default are unchanged.


In [7]:
# =============================================================================
# Moving-object filter (Part 4)
# =============================================================================

DYNAMIC_CLASS_NAMES = {
    "car", "truck", "bus", "van", "person", "pedestrian", "rider", "riders",
    "bicycle", "bicyclist", "cyclist", "motorcycle", "motorbike", "train", "trailer",
}
STATIC_CLASS_NAMES = {
    "pole", "traffic sign", "traffic light", "building", "wall", "fence",
    "vegetation", "terrain", "sidewalk", "road", "sky", "guard rail",
    "parking", "static", "bridge", "tunnel",
}


def _normalize_class_token(token: str) -> str:
    """v6.2.1 (Bug 2): STATIC_CLASS_NAMES / DYNAMIC_CLASS_NAMES are flat, space-separated
    names (\"traffic sign\", \"guard rail\", \"pole\", \"car\", ...). The actual SAM3 loader
    (load_sam3_masks_for_frame, Part 3) emits class_name as \"super_cat/sub_cat\"
    (e.g. \"vehicle/car\", \"traffic/traffic_sign\", \"road/road_surface\"), with underscores
    inside multi-word sub-categories. An exact match of that compound string against either
    set almost never succeeds, so nearly everything fell through to the conservative
    "unknown -> dynamic" default regardless of its true class -- this is the most likely
    reason the real Group 1c data produced zero usable stationary objects. Normalizing
    underscores/hyphens to spaces is the ONLY change here; no new class names are added."""
    return " ".join(token.strip().lower().replace("_", " ").replace("-", " ").split())


def is_class_dynamic(class_name: str) -> bool:
    """Manual filtering rule (Part 4a). Unknown/unlisted classes default to
    'dynamic' (excluded) — see markdown above for why this default is
    deliberately conservative for this use case.

    v6.2.1: SAM3 emits compound labels ("super_cat/sub_cat", e.g. "vehicle/car",
    "traffic/traffic_sign", "road/road_surface"). We check the whole normalized label
    plus each "/"-separated part -- most specific (sub_cat) first, then super_cat --
    against the SAME two sets, with underscores/hyphens normalized to spaces. The
    STATIC_CLASS_NAMES / DYNAMIC_CLASS_NAMES sets and the "unknown -> dynamic" default
    are unchanged; only which strings get compared against them changed."""
    raw = class_name.strip().lower()
    parts = [p for p in raw.split("/") if p]
    candidates = [_normalize_class_token(p) for p in reversed(parts)]   # sub_cat before super_cat
    whole = _normalize_class_token(raw.replace("/", " "))
    if whole not in candidates:
        candidates.append(whole)
    for name in candidates:
        if name in STATIC_CLASS_NAMES:
            return False
        if name in DYNAMIC_CLASS_NAMES:
            return True
    return True  # unknown class -> conservatively treat as dynamic (exclude)


def compute_dense_flow(rgb_curr_np: np.ndarray, rgb_next_np: np.ndarray) -> np.ndarray:
    """Farneback dense optical flow between two uint8 RGB frames. Returns
    flow [H, W, 2] in pixels (Part 4b)."""
    g1 = cv2.cvtColor(rgb_curr_np, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(rgb_next_np, cv2.COLOR_RGB2GRAY)
    return cv2.calcOpticalFlowFarneback(
        g1, g2, None, pyr_scale=0.5, levels=3, winsize=15,
        iterations=3, poly_n=5, poly_sigma=1.2, flags=0)


def estimate_ego_flow(flow: np.ndarray, static_mask_union: Optional[np.ndarray] = None) -> np.ndarray:
    """Median flow over background pixels as a cheap proxy for the
    ego-motion-induced flow field. Prefers the union of class-static masks;
    falls back to the image's upper quarter (sky/distant background in
    KITTI's forward-facing camera) if no static masks are available."""
    if static_mask_union is not None and static_mask_union.sum() > 100:
        return np.median(flow[static_mask_union], axis=0)
    h = flow.shape[0]
    return np.median(flow[: h // 4, :, :].reshape(-1, 2), axis=0)


def is_object_stationary_by_flow(mask: np.ndarray, flow: np.ndarray,
                                  ego_flow: Optional[np.ndarray] = None,
                                  residual_threshold: float = FLOW_RESIDUAL_PX) -> bool:
    """
    Flags an object as stationary if its median optical flow, AFTER
    subtracting the scene's ego-motion-induced background flow, is below
    `residual_threshold` pixels. See markdown above for the reasoning.
    """
    if mask.sum() < 20:
        return False  # too small a region to trust a flow estimate; exclude conservatively
    obj_median = np.median(flow[mask], axis=0)
    if ego_flow is None:
        ego_flow = np.zeros(2, dtype=np.float32)
    residual = float(np.linalg.norm(obj_median - ego_flow))
    return residual <= residual_threshold


def filter_dynamic_objects(tracks: List[Track],
                            rgb_curr_np: Optional[np.ndarray] = None,
                            rgb_next_np: Optional[np.ndarray] = None,
                            mode: str = DYNAMIC_FILTER_MODE,
                            flow_residual_threshold: float = FLOW_RESIDUAL_PX) -> List[Track]:
    """
    Combined moving-object filter (Part 4). Returns only the STATIONARY
    subset of `tracks`, ready to feed into Part 5's stationary-object losses.

    mode:
      "class" -> manual class-name filtering only (fast, no RGB needed)
      "flow"  -> optical-flow residual filtering only
      "both"  -> object must pass BOTH checks (default; see markdown above)
    """
    if mode not in ("class", "flow", "both"):
        raise ValueError(f"Unknown DYNAMIC_FILTER_MODE: {mode}")

    class_static = [not is_class_dynamic(t.class_name) for t in tracks]

    if mode == "class":
        return [t for t, keep in zip(tracks, class_static) if keep]

    if rgb_curr_np is None or rgb_next_np is None:
        log.warning("[filter_dynamic_objects] mode=%s requires rgb frames; "
                     "falling back to class-only filtering for this pair.", mode)
        return [t for t, keep in zip(tracks, class_static) if keep]

    flow = compute_dense_flow(rgb_curr_np, rgb_next_np)
    static_union = None
    if any(class_static):
        static_union = np.zeros(flow.shape[:2], dtype=bool)
        for t, keep in zip(tracks, class_static):
            if keep and t.mask.shape == static_union.shape:
                static_union |= t.mask
    ego_flow = estimate_ego_flow(flow, static_union)

    flow_static = [is_object_stationary_by_flow(t.mask, flow, ego_flow, flow_residual_threshold)
                   for t in tracks]

    if mode == "flow":
        return [t for t, keep in zip(tracks, flow_static) if keep]

    return [t for t, c_ok, f_ok in zip(tracks, class_static, flow_static) if c_ok and f_ok]


# =============================================================================
# Self-test: verify all 3 trackers preserve identity across a synthetic
# 2-object sequence, and that the dynamic filter separates them correctly.
# =============================================================================

def _make_synthetic_mask(h: int, w: int, cx: float, cy: float, r: float) -> np.ndarray:
    yy, xx = np.mgrid[0:h, 0:w]
    return ((xx - cx) ** 2 + (yy - cy) ** 2) <= r * r


def _self_test_tracking_and_filter() -> None:
    h, w = 128, 256
    img_diag = math.hypot(h, w)

    frame0 = [
        _mask_to_detection(_make_synthetic_mask(h, w, 40, 60, 12), class_name="pole", score=0.95),
        _mask_to_detection(_make_synthetic_mask(h, w, 150, 70, 18), class_name="car", score=0.9),
    ]
    frame1 = [
        _mask_to_detection(_make_synthetic_mask(h, w, 42, 60, 12), class_name="pole", score=0.95),
        _mask_to_detection(_make_synthetic_mask(h, w, 165, 70, 18), class_name="car", score=0.9),
    ]

    for name, trk in [("MaskIoUTracker", MaskIoUTracker(img_diag=img_diag)),
                       ("HungarianMaskTracker", HungarianMaskTracker(img_diag=img_diag)),
                       ("ByteTrackMaskTracker", ByteTrackMaskTracker(img_diag=img_diag))]:
        t1 = trk.update(frame0)
        t2 = trk.update(frame1)
        ids1, ids2 = sorted(t.track_id for t in t1), sorted(t.track_id for t in t2)
        assert ids1 == ids2, f"{name}: track IDs changed across frames ({ids1} -> {ids2})"
        print(f"  {name}: OK  ({len(t2)} tracks, ids={ids2})")

    # Dynamic filter, class-mode only (no synthetic RGB needed)
    tracker = HungarianMaskTracker(img_diag=img_diag)
    tracks = tracker.update(frame0)
    stationary = filter_dynamic_objects(tracks, mode="class")
    assert len(stationary) == 1 and stationary[0].class_name == "pole", \
        f"class filter should keep only the pole, got {[t.class_name for t in stationary]}"
    print(f"  filter_dynamic_objects(mode='class'): OK (kept {[t.class_name for t in stationary]})")

    # v6.2.1 (Bug 2) regression checks: SAM3's actual compound "super_cat/sub_cat" labels
    # must be classified the same way their flat equivalents already are.
    compound_cases = [
        ("vehicle/car", True), ("traffic/traffic_sign", False), ("road/road_surface", False),
        ("structure/pole", False), ("human/pedestrian", True), ("nature/vegetation", False),
        ("Vehicle/Bicycle", True), ("some_never_seen_thing", True),   # still conservative: unknown -> dynamic
    ]
    for name, expect_dynamic in compound_cases:
        got = is_class_dynamic(name)
        assert got == expect_dynamic, f"is_class_dynamic({name!r}) = {got}, expected {expect_dynamic}"
    print(f"  is_class_dynamic() compound-label normalization: OK ({len(compound_cases)} cases)")


_self_test_tracking_and_filter()
print("Tracking + dynamic-filter self-test passed "
      "(synthetic 2-object sequence: all 3 trackers preserve IDs, filter isolates the stationary pole).")


  MaskIoUTracker: OK  (2 tracks, ids=[1, 2])
  HungarianMaskTracker: OK  (2 tracks, ids=[1, 2])
  ByteTrackMaskTracker: OK  (2 tracks, ids=[1, 2])
  filter_dynamic_objects(mode='class'): OK (kept ['pole'])
  is_class_dynamic() compound-label normalization: OK (8 cases)
Tracking + dynamic-filter self-test passed (synthetic 2-object sequence: all 3 trackers preserve IDs, filter isolates the stationary pole).


## 6 · Model wrapper, Pose Network, and Differentiable Warping


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# DA3 model wrapper
# ─────────────────────────────────────────────────────────────────────────────
_DA3_TRANSFORM = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD),
])


def load_da3_model(device: torch.device) -> nn.Module:
    from depth_anything_3.api import DepthAnything3
    log.info("Loading %s ...", MODEL_ID)
    model = DepthAnything3.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR).to(device)
    return model


def _apply_arch_impl(model: nn.Module, arch_mode: str) -> None:
    if arch_mode == "full":
        for p in model.parameters():
            p.requires_grad_(True)
        model.model.train()
        if hasattr(model.model, "gradient_checkpointing_enable"):
            model.model.gradient_checkpointing_enable()
        log.info("  [arch=full] All parameters trainable.")
        return

    if arch_mode == "decoder_only":
        for name, param in model.named_parameters():
            freeze = any(k in name for k in
                         ["encoder", "pretrained", "backbone", "net.net"])
            param.requires_grad_(not freeze)
        model.model.train()
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log.info("  [arch=decoder_only] %.1fM params trainable", trainable/1e6)
        return

    if arch_mode == "last2_blocks":
        for p in model.parameters():
            p.requires_grad_(False)
        decoder_block_names = [
            name for name, _ in model.named_parameters()
            if "decoder" in name.lower() or "head" in name.lower()
               or "depth_head" in name.lower() or "scratch" in name.lower()
        ]
        last_n = max(1, int(0.30 * len(decoder_block_names)))
        for name in decoder_block_names[-last_n:]:
            model.get_parameter(name).requires_grad_(True)
        model.model.train()
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        log.info("  [arch=last2_blocks] %.1fM params trainable", trainable/1e6)
        return

    raise ValueError(f"Unknown arch_mode: {arch_mode}")


def describe_trainable(model: nn.Module, arch_mode: str) -> Dict[str, float]:
    """
    Explicit report of WHICH parts of DA3 are trainable (the name-substring freezing in
    `_apply_arch_impl` is brittle: a renamed module inside DA3 would silently change what trains).
    Logs trainable/total parameters per top-level module of `model.model` and asserts that the
    trainable fraction lies in EXPECTED_TRAINABLE_FRACTION[arch_mode].
    """
    per_top: Dict[str, List[int]] = {}
    for n, p in model.model.named_parameters():
        top = n.split(".")[0]
        t = per_top.setdefault(top, [0, 0])
        t[1] += p.numel()
        t[0] += p.numel() if p.requires_grad else 0
    tot = sum(v[1] for v in per_top.values()); trn = sum(v[0] for v in per_top.values())
    frac = trn / max(tot, 1)
    log.info("  [arch=%s] trainable %.1fM / %.1fM params (%.1f %%)", arch_mode, trn / 1e6, tot / 1e6, 100 * frac)
    for top, (t_, a_) in sorted(per_top.items(), key=lambda kv: -kv[1][1]):
        log.info("      %-24s trainable %7.2fM / %7.2fM", top, t_ / 1e6, a_ / 1e6)
    lo, hi = EXPECTED_TRAINABLE_FRACTION[arch_mode]
    ok = (trn > 0) and (lo <= frac <= hi)
    if not ok:
        msg = (f"arch_mode={arch_mode}: trainable fraction {frac:.4f} outside the expected range "
               f"[{lo}, {hi}] — DA3's parameter names probably changed; inspect the module list above.")
        if STRICT_ARCH_CHECK:
            raise RuntimeError(msg)
        log.warning(msg)
    return {"trainable": trn, "total": tot, "fraction": frac}


def _apply_arch(model: nn.Module, arch_mode: str) -> None:
    _apply_arch_impl(model, arch_mode)
    describe_trainable(model, arch_mode)


def forward_batch(model: nn.Module, rgb_batch: torch.Tensor,
                  K_batch: torch.Tensor, has_K: torch.Tensor) -> torch.Tensor:
    # Differentiable forward pass -> metric depth [B, 1, H, W].
    B, C, H, W = rgb_batch.shape
    inp = rgb_batch.unsqueeze(1)
    preds = model.model(inp, export_feat_layers=[])

    if isinstance(preds, dict):
        raw = preds.get("depth", preds.get("metric_depth", list(preds.values())[0]))
        fx_pred = torch.where(
            has_K.view(B, 1, 1, 1),
            (0.5 * (K_batch[:, 0, 0] + K_batch[:, 1, 1])).view(B, 1, 1, 1),   # v6: avg(fx, fy) as in the DA3 metric recipe
            torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                       device=rgb_batch.device, dtype=rgb_batch.dtype),
        )
    elif isinstance(preds, (tuple, list)):
        raw = preds[0]
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                             device=rgb_batch.device, dtype=rgb_batch.dtype)
    else:
        raw = preds
        fx_pred = torch.full((B, 1, 1, 1), FOCAL_LENGTH_X,
                             device=rgb_batch.device, dtype=rgb_batch.dtype)

    if raw.dim() == 3:
        raw = raw.unsqueeze(1)
    depth = (fx_pred.float() * raw.float()) / ORTHO_NORM      # fp32 from here on (geometry / losses need it)
    if depth.shape[-2:] != (H, W):
        depth = F.interpolate(depth, size=(H, W), mode="bilinear",
                              align_corners=False)
    return depth


@torch.no_grad()
def run_da3_inference(model: nn.Module, image_np: np.ndarray,
                      device: torch.device, fx: float = FOCAL_LENGTH_X) -> np.ndarray:
    # Run DA3METRIC-LARGE inference on one uint8 RGB image (H x W x 3).
    orig_h, orig_w = image_np.shape[:2]
    new_h = int(round(orig_h / 14) * 14)
    new_w = int(round(orig_w / 14) * 14)
    img_pil = Image.fromarray(image_np).resize((new_w, new_h),
                                                Image.Resampling.BICUBIC)
    t_img = _DA3_TRANSFORM(img_pil)
    inp   = t_img.unsqueeze(0).unsqueeze(1).to(device)
    model.model.eval()
    preds = model(inp, export_feat_layers=[])
    if isinstance(preds, dict):
        raw = preds.get("depth", preds.get("metric_depth", list(preds.values())[0]))
    elif isinstance(preds, (tuple, list)):
        raw = preds[0]
    else:
        raw = preds
    focal_used: float = fx
    if isinstance(raw, torch.Tensor):
        if raw.dim() == 3:
            raw = raw.unsqueeze(1)
        depth = (focal_used * raw) / ORTHO_NORM
        depth = F.interpolate(depth, size=(orig_h, orig_w),
                              mode="bilinear", align_corners=False)
    else:
        raise RuntimeError(f"Unexpected model output type {type(raw)}")
    return depth.squeeze().cpu().float().numpy()


# ─────────────────────────────────────────────────────────────────────────────
# Pose Network — SfMLearner / Monodepth2-style ResNet18 with 6-DoF head
# ─────────────────────────────────────────────────────────────────────────────
class PoseNet(nn.Module):
    # Predicts the 6-DoF relative pose T_{curr -> src} (axis-angle + translation)
    # from a pair (I_curr, I_src) stacked along the channel dim.
    #
    # v6 changes (PoseNet bootstrapping):
    #   * NO zero-init of the last layer.  v5 started at exactly R=I, t=0, where
    #     the reprojection is independent of depth (K·I·K^-1·(d·p) = p) so the
    #     photometric loss has ZERO gradient w.r.t. depth.  The default init
    #     gives a small non-degenerate random pose from step 0.
    #   * output scale POSE_OUT_SCALE (0.1) instead of 0.01: with METRIC depth the
    #     translation is in metres (~1 m / frame on KITTI).
    #   * PoseNet is pre-trained with depth frozen (see `warmup_pose_net`).
    def __init__(self, out_scale: float = POSE_OUT_SCALE):
        super().__init__()
        self.out_scale = float(out_scale)
        weights = None
        try:
            weights = tvm.ResNet18_Weights.DEFAULT
        except Exception:
            pass
        try:
            backbone = tvm.resnet18(weights=weights)
        except Exception as e:                       # offline machine
            log.warning("PoseNet: ImageNet weights unavailable (%s) -> random init", e)
            backbone = tvm.resnet18(weights=None)

        old_conv = backbone.conv1
        new_conv = nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv.weight.copy_(torch.cat([old_conv.weight, old_conv.weight], dim=1) / 2.0)
        backbone.conv1 = new_conv
        self.encoder = nn.Sequential(*list(backbone.children())[:-2])   # (B, 512, H/32, W/32)

        self.pose_head = nn.Sequential(
            nn.Conv2d(512, 256, 1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, stride=1, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, stride=1, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 6, 1),
        )
        nn.init.zeros_(self.pose_head[-1].bias)      # bias 0, weights keep the default (non-zero) init

    def forward(self, rgb_curr: torch.Tensor, rgb_src: torch.Tensor) -> torch.Tensor:
        # rgb_* : [B, 3, H, W] (ImageNet-normalised)  ->  pose [B, 6]
        x = torch.cat([rgb_curr, rgb_src], dim=1)
        p = self.pose_head(self.encoder(x))
        return self.out_scale * p.mean(dim=(2, 3))


# ─────────────────────────────────────────────────────────────────────────────
# SE(3) helpers
# ─────────────────────────────────────────────────────────────────────────────
def axisangle_to_rotmat(aa: torch.Tensor) -> torch.Tensor:
    # Rodrigues' formula: [B, 3] -> [B, 3, 3]
    aa    = aa.float()
    theta = aa.norm(dim=1, keepdim=True).clamp(min=EPS)      # [B, 1]
    k     = aa / theta
    cos_t         = torch.cos(theta).unsqueeze(-1)            # [B, 1, 1]
    sin_t         = torch.sin(theta).unsqueeze(-1)
    one_minus_cos = 1.0 - cos_t
    K = torch.zeros(aa.shape[0], 3, 3, device=aa.device, dtype=aa.dtype)
    K[:, 0, 1] = -k[:, 2]; K[:, 0, 2] =  k[:, 1]
    K[:, 1, 0] =  k[:, 2]; K[:, 1, 2] = -k[:, 0]
    K[:, 2, 0] = -k[:, 1]; K[:, 2, 1] =  k[:, 0]
    I = torch.eye(3, device=aa.device, dtype=aa.dtype).unsqueeze(0).expand_as(K)
    return cos_t * I + sin_t * K + one_minus_cos * (k.unsqueeze(2) @ k.unsqueeze(1))


def pose_to_se3(pose: torch.Tensor) -> torch.Tensor:
    # [B, 6] (axisangle[3], translation[3]) -> SE(3) [B, 4, 4]   (always fp32)
    pose = pose.float()
    B = pose.shape[0]
    R = axisangle_to_rotmat(pose[:, :3])
    T_ = torch.eye(4, device=pose.device, dtype=pose.dtype).unsqueeze(0).repeat(B, 1, 1)
    T_[:, :3, :3] = R
    T_[:, :3,  3] = pose[:, 3:]
    return T_


def invert_se3(T_: torch.Tensor) -> torch.Tensor:
    R, t = T_[:, :3, :3], T_[:, :3, 3:]
    Ti = torch.eye(4, device=T_.device, dtype=T_.dtype).unsqueeze(0).repeat(T_.shape[0], 1, 1)
    Ti[:, :3, :3] = R.transpose(1, 2)
    Ti[:, :3, 3:] = -R.transpose(1, 2) @ t
    return Ti


# ─────────────────────────────────────────────────────────────────────────────
# Differentiable inverse-warping  (v6: ALWAYS fp32, autocast disabled)
# ─────────────────────────────────────────────────────────────────────────────
# v5 executed these matmuls under fp16 autocast.  K·X reaches ~u·Z ≈ 1e5 > 65504
# (fp16 max) for far right-hand pixels -> inf -> NaN gradients -> GradScaler
# skipped the optimizer step -> the SSL runs never updated the network.
# Even below the overflow limit fp16 quantises pixel coordinates in [1024, 2048)
# to 1.0 px.  Geometry therefore must not run in half precision.
class BackprojectDepth(nn.Module):
    # Backproject depth to 3-D camera-frame points using inverse intrinsics.
    def __init__(self, height, width):
        super().__init__()
        self.H, self.W = height, width
        u, v = torch.meshgrid(
            torch.arange(0, height, dtype=torch.float32),
            torch.arange(0, width,  dtype=torch.float32),
            indexing="ij")
        pix = torch.stack([v, u], dim=0).view(2, -1)
        ones = torch.ones(1, height * width)
        self.register_buffer("pix_homog", torch.cat([pix, ones], dim=0).unsqueeze(0))   # [1, 3, H*W]

    def forward(self, depth: torch.Tensor, inv_K: torch.Tensor) -> torch.Tensor:
        # depth [B,1,H,W]; inv_K [B,3,3]; returns [B,4,H*W]  (fp32)
        B = depth.shape[0]
        with torch.autocast(device_type=depth.device.type, enabled=False):
            depth = depth.float(); inv_K = inv_K.float()
            pix = self.pix_homog.expand(B, -1, -1)
            cam_pts = torch.matmul(inv_K, pix)
            cam_pts = depth.reshape(B, 1, -1) * cam_pts
            return torch.cat([cam_pts, torch.ones(B, 1, self.H * self.W,
                                                   device=depth.device, dtype=torch.float32)], dim=1)


class Project3D(nn.Module):
    # Project 3-D camera-frame points with intrinsics + pose  ->  normalised grid.
    def __init__(self, height, width):
        super().__init__()
        self.H, self.W = height, width

    def forward(self, points_3d, K, T, return_z: bool = False):
        B = points_3d.shape[0]
        with torch.autocast(device_type=points_3d.device.type, enabled=False):
            P = torch.matmul(K.float(), T.float()[:, :3, :])                # [B,3,4]
            cam_pts = torch.matmul(P, points_3d.float())                    # [B,3,H*W]
            z_raw = cam_pts[:, 2:3]
            pix = cam_pts[:, :2] / z_raw.clamp(min=0.01)
            pix = pix.view(B, 2, self.H, self.W)
            pix_norm = torch.stack([2.0 * pix[:, 0] / (self.W - 1) - 1.0,
                                    2.0 * pix[:, 1] / (self.H - 1) - 1.0], dim=1)
            pix_norm = pix_norm.clamp(-10.0, 10.0).permute(0, 2, 3, 1)
        if return_z:
            return pix_norm, z_raw.view(B, 1, self.H, self.W)
        return pix_norm


# ─── Cached BackprojectDepth / Project3D (avoid per-batch GPU allocation) ─────
_backproj_cache: Dict[Tuple, "BackprojectDepth"] = {}
_project_cache:  Dict[Tuple, "Project3D"]        = {}

def _get_backproj(H: int, W: int, device: torch.device) -> "BackprojectDepth":
    key = (H, W, device)
    if key not in _backproj_cache:
        _backproj_cache[key] = BackprojectDepth(H, W).to(device)
    return _backproj_cache[key]

def _get_project(H: int, W: int, device: torch.device) -> "Project3D":
    key = (H, W, device)
    if key not in _project_cache:
        _project_cache[key] = Project3D(H, W).to(device)
    return _project_cache[key]


def _safe_inv_K(K: torch.Tensor) -> torch.Tensor:
    """Batched K inversion in fp32."""
    B = K.shape[0]
    with torch.autocast(device_type=K.device.type, enabled=False):
        K_f32 = K.float()
        eye = torch.eye(3, device=K.device, dtype=torch.float32).unsqueeze(0).expand(B, -1, -1)
        try:
            return torch.linalg.solve(K_f32, eye)
        except torch.linalg.LinAlgError:
            K_safe = K_f32.clone()
            diag = K_safe.diagonal(dim1=-2, dim2=-1)
            diag[diag.abs() < 1e-6] = 1.0
            return torch.linalg.solve(K_safe, eye)


def warp_image(image_src: torch.Tensor,
               depth_curr: torch.Tensor,
               pose_curr_to_src: torch.Tensor,
               K: torch.Tensor,
               has_K: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    # Inverse-warp `image_src` into the CURRENT frame using the depth of the
    # current frame and T_{curr -> src}.  Returns (warped [B,C,H,W], valid [B,1,H,W]).
    # `valid` = projected pixel inside the source image AND in front of the camera.
    B, C, H, W = image_src.shape
    with torch.autocast(device_type=image_src.device.type, enabled=False):
        image_src = image_src.float()
        inv_K = _safe_inv_K(K)
        pts3d = _get_backproj(H, W, image_src.device)(depth_curr, inv_K)
        pix, z = _get_project(H, W, image_src.device)(pts3d, K, pose_curr_to_src, return_z=True)
        warped = F.grid_sample(image_src, pix, mode="bilinear",
                               padding_mode="zeros", align_corners=True)
        valid = ((pix[..., 0].abs() <= 1.0) & (pix[..., 1].abs() <= 1.0)).float().unsqueeze(1)
        valid = valid * (z > 0.01).float()
        warped = torch.nan_to_num(warped, nan=0.0, posinf=0.0, neginf=0.0)
    return warped, valid


def warp_depth(depth_src: torch.Tensor,
               depth_curr: torch.Tensor,
               pose_curr_to_src: torch.Tensor,
               K: torch.Tensor, has_K: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    # Sample the SOURCE depth map at the locations where the current pixels land.
    return warp_image(depth_src, depth_curr, pose_curr_to_src, K)


print("DA3 model wrapper + PoseNet + fp32 differentiable warping ready (v6).")


DA3 model wrapper + PoseNet + fp32 differentiable warping ready (v6).


## 7 · Batch-Level Stationary Track-Pair Extraction

Bridges Part 3/4 (per-sample tracking + filtering, operating on numpy masks) to Part 5's loss functions (operating on batched `torch` tensors). For one training batch of `B` frame-pairs, this section runs the configured tracker (`TRACKER_TYPE`) across each sample's `(frame_t, frame_{t+1})` SAM3 masks, keeps only objects matched in *both* frames, and applies the Part 4 dynamic-object filter — producing, per sample, a list of `(MaskDetection_t, MaskDetection_{t+1})` pairs that Part 5's stationary-object losses consume directly.

This is computed **once per training step** (not once per loss term) and reused across every stationary-object loss active that step — consistent with Part 6's "avoid duplicate computation" requirement. It runs on CPU (mask I/O + tracker bookkeeping are numpy/Python operations with no GPU tensors involved) and is gated by `ENABLE_TRACKING` and by whether the active loss actually needs it (`NEED_TRACKING_LOSSES`, defined alongside the loss registry below), so trials using the original 13+3 loss variants pay zero extra cost.

> **v6.2.1 fix (Bug 1) — coordinate system:** the optical-flow field returned by `compute_dense_flow(rgb_curr_np, rgb_next_np)` is defined on **frame-t pixels**. `track_and_filter_pair()` previously passed the **frame-(t+1)** mask (`d_next.mask`) into that frame-t-indexed field when building the pseudo-tracks for the flow-stationarity check, sampling flow at the wrong image location and plausibly rejecting genuinely stationary tracked objects — the most likely reason `stationary_depth_motion`, `stationary_uniform_motion`, `full_3d_ego_motion` and `mask_chamfer` all trained to `loss = 0`. The fix uses `det_curr.mask` for that check; `det_next` is still returned and used unchanged by Part 5's losses. A regression test (`_self_test_coordinate_fix`, run automatically at the end of this cell) fails under the old behaviour and passes under the fix. This cell also adds a **real-data** stationary-pair coverage diagnostic (`stationary_pair_coverage_report`, `compare_filter_modes`) — see the Sanity Suite section — that is separate from, and not satisfied by, the generic smoke test's synthetic-mask fallback.


In [9]:
# =============================================================================
# Batch-level stationary track-pair extraction (bridges Part 3/4 -> Part 5)
# =============================================================================

def _flip_detection(det: MaskDetection) -> MaskDetection:
    """Mirror a SAM3 detection horizontally (v6: the dataset flips images on the fly, but the
    SAM3 masks on disk belong to the UNFLIPPED frame — v5 silently mismatched them)."""
    Wn = det.mask.shape[1]
    x1, y1, x2, y2 = det.bbox
    return MaskDetection(mask=np.ascontiguousarray(det.mask[:, ::-1]),
                         bbox=(Wn - 1 - x2, y1, Wn - 1 - x1, y2),
                         centroid=(Wn - 1 - det.centroid[0], det.centroid[1]),
                         area=det.area, class_name=det.class_name, score=det.score,
                         embedding=det.embedding)


def track_and_filter_pair(rgb_curr_np: np.ndarray, rgb_next_np: np.ndarray,
                          rgb_curr_path: str, rgb_next_path: str,
                          tracker_type: str = TRACKER_TYPE,
                          filter_mode: str = DYNAMIC_FILTER_MODE,
                          flipped: bool = False,
                          return_diag: bool = False
                          ):
    """
    Runs SAM3 mask loading + tracking + dynamic filtering for ONE (frame_t,
    frame_{t+1}) training pair. Returns matched (detection_t, detection_{t+1})
    tuples for objects that are BOTH successfully tracked across the pair AND
    judged stationary. This is the single entry point Part 5's losses use —
    they never touch the tracker or filter APIs directly.

    return_diag=True (v6.2.1, Bug 3/4 -- additive, default False, no change to any
    existing caller) additionally returns a small diagnostic dict so the real-data
    coverage report below can distinguish "no SAM3 detections", "no matched tracks"
    and "tracks matched but none stationary" without duplicating this function's logic.
    """
    diag = {"n_dets_curr": 0, "n_dets_next": 0, "class_hist_curr": Counter(),
           "n_matched_pairs": 0, "n_stationary_pairs": 0}
    dets_curr = load_sam3_masks_for_frame(rgb_curr_path, rgb_curr_np)
    dets_next = load_sam3_masks_for_frame(rgb_next_path, rgb_next_np)
    if flipped:
        dets_curr = [_flip_detection(d) for d in dets_curr]
        dets_next = [_flip_detection(d) for d in dets_next]
    diag["n_dets_curr"] = len(dets_curr)
    diag["n_dets_next"] = len(dets_next)
    diag["class_hist_curr"] = Counter(d.class_name for d in dets_curr)
    if not dets_curr or not dets_next:
        return ([], diag) if return_diag else []

    img_diag = math.hypot(rgb_curr_np.shape[0], rgb_curr_np.shape[1])
    tracker = build_tracker(tracker_type, img_diag=img_diag)

    tracker.update(dets_curr)  # seed tracks at frame t
    snapshot_curr: Dict[int, MaskDetection] = {
        t.track_id: MaskDetection(mask=t.mask, bbox=t.bbox, centroid=t.centroid,
                                  area=t.area, class_name=t.class_name,
                                  score=t.score, embedding=t.embedding)
        for t in tracker.tracks
    }

    tracks_next = tracker.update(dets_next)  # match into frame t+1
    paired: List[Tuple[MaskDetection, MaskDetection]] = []
    for t in tracks_next:
        # Keep only tracks that: existed at frame t, were matched (not just
        # aged) this update, and have >=2 total hits (i.e. genuinely tracked
        # across the pair, not a track spawned fresh THIS frame).
        if t.track_id in snapshot_curr and t.time_since_update == 0 and t.hits >= 2:
            det_next = MaskDetection(mask=t.mask, bbox=t.bbox, centroid=t.centroid,
                                     area=t.area, class_name=t.class_name,
                                     score=t.score, embedding=t.embedding)
            paired.append((snapshot_curr[t.track_id], det_next))

    diag["n_matched_pairs"] = len(paired)
    if not paired:
        return ([], diag) if return_diag else []

    # Re-use filter_dynamic_objects by wrapping each pair's CURRENT-FRAME state
    # in a throwaway Track (filter only reads .mask / .class_name).
    #
    # v6.2.1 (Bug 1 -- the primary fix in this patch): the optical-flow field returned by
    # compute_dense_flow(rgb_curr_np, rgb_next_np) is defined on FRAME-t PIXELS (it answers
    # "where did this frame-t pixel move to"). The object mask supplied to the flow-based
    # stationarity test must therefore also be the FRAME-t mask (det_curr.mask). The previous
    # implementation passed det_next.mask -- a frame-(t+1) mask -- into that frame-t-indexed
    # flow field, sampling flow at the wrong image location and plausibly rejecting genuinely
    # stationary tracked objects. The frame-(t+1) mask remains available as det_next.mask and
    # is unchanged for the downstream stationary-object geometric-consistency losses (Part 5),
    # which still receive both det_curr and det_next via `paired`.
    pseudo_tracks = [Track(track_id=i, mask=det_curr.mask, bbox=det_curr.bbox,
                           centroid=det_curr.centroid, area=det_curr.area,
                           class_name=det_curr.class_name, score=det_curr.score)
                     for i, (det_curr, det_next) in enumerate(paired)]
    stationary_pseudo = filter_dynamic_objects(pseudo_tracks, rgb_curr_np, rgb_next_np,
                                               mode=filter_mode)
    keep_ids = {t.track_id for t in stationary_pseudo}   # unique per-pair ids (enumerate), not a shared -1
    result = [pair for i, pair in enumerate(paired) if i in keep_ids]
    diag["n_stationary_pairs"] = len(result)
    return (result, diag) if return_diag else result


def get_batch_stationary_pairs(batch: Dict,
                               tracker_type: str = TRACKER_TYPE,
                               filter_mode: str = DYNAMIC_FILTER_MODE,
                               return_diag: bool = False
                               ):
    """
    Batch-level wrapper: expects `batch["rgb_raw"]` / `batch["next_rgb_raw"]`
    (CPU float tensors in [0, 1], as produced by the dataset before the
    `.to(device)` transfer in the training loop — reused directly here to
    avoid an extra GPU round-trip) and `batch["rgb_path"]` / `batch["next_rgb_path"]`
    (string lists). Returns, for each sample b, its list of stationary
    (det_t, det_{t+1}) pairs (possibly empty).

    return_diag=True (v6.2.1, default False, no change to any existing caller) additionally
    returns a per-sample list of the diagnostic dicts from `track_and_filter_pair`, used by
    the real-data coverage report below.
    """
    rgb_raw = batch.get("rgb_raw")
    rgb_next_raw = batch.get("next_rgb_raw")
    rgb_paths = batch.get("rgb_path")
    rgb_next_paths = batch.get("next_rgb_path")
    if rgb_raw is None or rgb_next_raw is None or rgb_paths is None or rgb_next_paths is None:
        return ([], []) if return_diag else []

    B = rgb_raw.shape[0]
    flipped = batch.get("flipped")
    flipped = flipped.cpu().numpy().astype(bool) if flipped is not None else np.zeros(B, dtype=bool)
    curr_np = (rgb_raw.permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)
    next_np = (rgb_next_raw.permute(0, 2, 3, 1).clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)

    out: List[List[Tuple[MaskDetection, MaskDetection]]] = []
    diags: List[Dict] = []
    for b in range(B):
        try:
            if return_diag:
                pairs, d = track_and_filter_pair(curr_np[b], next_np[b], rgb_paths[b], rgb_next_paths[b],
                                                 tracker_type=tracker_type, filter_mode=filter_mode,
                                                 flipped=bool(flipped[b]), return_diag=True)
                diags.append(d)
            else:
                pairs = track_and_filter_pair(curr_np[b], next_np[b], rgb_paths[b], rgb_next_paths[b],
                                              tracker_type=tracker_type, filter_mode=filter_mode,
                                              flipped=bool(flipped[b]))
        except Exception as e:
            log.warning("[get_batch_stationary_pairs] sample %d failed: %s", b, e)
            pairs = []
            if return_diag:
                diags.append({"n_dets_curr": 0, "n_dets_next": 0, "class_hist_curr": Counter(),
                             "n_matched_pairs": 0, "n_stationary_pairs": 0, "error": str(e)})
        out.append(pairs)
    return (out, diags) if return_diag else out


print("Batch-level stationary track-pair extraction ready: get_batch_stationary_pairs(batch) -> "
      "List[List[Tuple[MaskDetection, MaskDetection]]]")


# ─────────────────────────────────────────────────────────────────────────────
# v6.1 — per-pixel "potentially dynamic" mask for the depth-consistency losses
# ─────────────────────────────────────────────────────────────────────────────
_DYN_STATS = {"frames": 0, "empty": 0, "warned": False}

def get_batch_dynamic_masks(batch: Dict, H: int, W: int,
                            dilate_px: int = DYN_MASK_DILATE_PX) -> torch.Tensor:
    """
    SAM3 instance masks of the CURRENT frame t whose class is *potentially moving*
    (`is_class_dynamic`: cars, pedestrians, cyclists, …), resized to the network resolution, mirrored for
    flipped samples and dilated by `dilate_px`.  Returned as float [B,1,H,W] (1 = exclude from
    temporal / geometric depth-consistency).

    This is a CLASS PRIOR: parked cars are masked too (conservative).  Only the CURRENT-frame mask is used:
    the temporal / geometric losses are evaluated on frame-t pixels, and the t+1 mask lives in frame-(t+1)
    coordinates — uniting the two (v6.1) removed legitimate background pixels wherever the object moved.
    (Warping the t+1 mask into frame t would need predicted geometry/pose and is deliberately not done.)
    If no SAM3 masks exist on disk the mask is all zeros and a warning is logged once.
    """
    B = batch["rgb"].shape[0]
    flipped = batch.get("flipped")
    flipped = flipped.cpu().numpy().astype(bool) if flipped is not None else np.zeros(B, dtype=bool)
    out = np.zeros((B, 1, H, W), dtype=np.float32)
    for b in range(B):
        for key in ("rgb_path",):
            dets = load_sam3_masks_for_frame(batch[key][b])
            _DYN_STATS["frames"] += 1
            _DYN_STATS["empty"] += int(len(dets) == 0)
            if flipped[b]:
                dets = [_flip_detection(d) for d in dets]
            for d in dets:
                if not is_class_dynamic(d.class_name):
                    continue
                m = d.mask.astype(np.uint8)
                if m.shape != (H, W):
                    m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
                out[b, 0] = np.maximum(out[b, 0], m.astype(np.float32))
        if dilate_px > 0 and out[b, 0].any():
            out[b, 0] = cv2.dilate(out[b, 0], np.ones((2 * dilate_px + 1,) * 2, np.uint8))
    if (not _DYN_STATS["warned"]) and _DYN_STATS["frames"] >= 40 and _DYN_STATS["empty"] == _DYN_STATS["frames"]:
        _DYN_STATS["warned"] = True
        log.warning("get_batch_dynamic_masks: NO SAM3 detections found for any of %d frames "
                    "(SAM3_MASK_DIR=%s) -> dynamic mask is all zeros.", _DYN_STATS["frames"], SAM3_MASK_DIR)
    return torch.from_numpy(out)

print("Batch-level tracking + dynamic-mask helpers ready.")

# =============================================================================
# v6.2.1 — Group 1c REAL-DATA stationary-pair coverage diagnostic (Bug 3/4)
# =============================================================================
# The generic gradient smoke test (sanity cell) can fall back to SYNTHETIC stationary
# masks when no real SAM3 tracks are found; that only proves the loss FORMULAS produce
# gradients, not that the real dataset actually yields usable stationary objects (which
# was exactly the failure the user observed: train_loss = val_loss = 0 for all four
# standalone stationary losses). This function scans a small, configurable number of
# REAL batches through the existing SAM3 loading / tracking / filtering machinery
# (get_batch_stationary_pairs -- no second tracker implementation) and reports why
# pairs do or do not survive, so "real" and "synthetic" evidence are never conflated.
def stationary_pair_coverage_report(loader: DataLoader,
                                    max_batches: int = SANITY_STATIONARY_SCAN_BATCHES,
                                    filter_mode: Optional[str] = None,
                                    tracker_type: str = TRACKER_TYPE) -> Dict:
    """
    Scans up to `max_batches` REAL batches from `loader` and reports real-data
    stationary-pair coverage. `filter_mode` defaults to the production
    DYNAMIC_FILTER_MODE; pass "class" / "flow" / "both" explicitly to compare
    which stage of filtering is responsible for a coverage problem (diagnostic
    only -- does not change the production default).
    """
    mode = filter_mode or DYNAMIC_FILTER_MODE
    n_batches = 0
    n_samples = 0
    batches_with_dets = 0
    batches_with_matched = 0
    batches_with_ge1 = 0
    batches_with_ge2 = 0
    per_sample_counts: List[int] = []
    class_hist_before_filter: Counter = Counter()

    it = iter(loader)
    for _ in range(max_batches):
        try:
            batch = next(it)
        except StopIteration:
            break
        n_batches += 1
        _, diags = get_batch_stationary_pairs(batch, tracker_type=tracker_type,
                                              filter_mode=mode, return_diag=True)
        if not diags:
            continue
        n_samples += len(diags)
        b_dets = any(d["n_dets_curr"] > 0 for d in diags)
        b_matched = any(d["n_matched_pairs"] > 0 for d in diags)
        b_ge1 = any(d["n_stationary_pairs"] >= 1 for d in diags)
        b_ge2 = any(d["n_stationary_pairs"] >= 2 for d in diags)
        batches_with_dets += int(b_dets)
        batches_with_matched += int(b_matched)
        batches_with_ge1 += int(b_ge1)
        batches_with_ge2 += int(b_ge2)
        for d in diags:
            per_sample_counts.append(d["n_stationary_pairs"])
            class_hist_before_filter.update(d["class_hist_curr"])

    total_pairs = int(sum(per_sample_counts))
    n_zero = int(sum(1 for c in per_sample_counts if c == 0))
    report = {
        "real_data": True,                      # never conflate with the synthetic smoke-test fallback
        "filter_mode": mode,
        "batches_scanned": n_batches,
        "n_samples_scanned": n_samples,
        "batches_with_detections": batches_with_dets,
        "batches_with_matched_tracks": batches_with_matched,
        "batches_with_ge1_stationary": batches_with_ge1,
        "batches_with_ge2_stationary": batches_with_ge2,
        "total_stationary_pairs": total_pairs,
        "mean_stationary_pairs_per_sample": (total_pairs / n_samples) if n_samples else float("nan"),
        "max_stationary_pairs_per_sample": (max(per_sample_counts) if per_sample_counts else 0),
        "n_samples_zero_stationary": n_zero,
        "coverage_fraction": ((n_samples - n_zero) / n_samples) if n_samples else 0.0,
        "class_histogram_before_filter": dict(class_hist_before_filter),
    }
    return report


def print_stationary_coverage_report(report: Dict, label: str = "") -> None:
    tag = f" [{label}]" if label else ""
    print(f"[Group 1c REAL-DATA coverage{tag}]  filter_mode={report['filter_mode']!r}  "
         f"batches_scanned={report['batches_scanned']}  samples_scanned={report['n_samples_scanned']}")
    print(f"    batches_with_detections={report['batches_with_detections']}  "
         f"batches_with_matched_tracks={report['batches_with_matched_tracks']}  "
         f"batches_with_>=1_stationary={report['batches_with_ge1_stationary']}  "
         f"batches_with_>=2_stationary={report['batches_with_ge2_stationary']}")
    print(f"    total_stationary_pairs={report['total_stationary_pairs']}  "
         f"mean/sample={report['mean_stationary_pairs_per_sample']:.3f}  "
         f"max/sample={report['max_stationary_pairs_per_sample']}  "
         f"samples_with_zero={report['n_samples_zero_stationary']}  "
         f"coverage={100 * report['coverage_fraction']:.1f}%")
    if report["class_histogram_before_filter"]:
        top = sorted(report["class_histogram_before_filter"].items(), key=lambda kv: -kv[1])[:10]
        print(f"    class histogram before filtering (top {len(top)}): {dict(top)}")
    if report["total_stationary_pairs"] == 0:
        print("    *** REAL DATA: ZERO stationary pairs found in this scan. Synthetic masks used elsewhere "
             "(e.g. the gradient smoke test) are NOT evidence that Group 1c works on this dataset. ***")


def compare_filter_modes(loader: DataLoader, max_batches: int = SANITY_STATIONARY_SCAN_BATCHES) -> Dict[str, Dict]:
    """
    Optional diagnostic (not run automatically, does not change DYNAMIC_FILTER_MODE):
    runs `stationary_pair_coverage_report` under "class", "flow" and "both" so a
    coverage problem can be attributed to class filtering, flow filtering, or their
    combination. Call this by hand while investigating; the production default stays
    whatever DYNAMIC_FILTER_MODE is set to in the configuration cell.
    """
    reports = {}
    for mode in ("class", "flow", "both"):
        reports[mode] = stationary_pair_coverage_report(loader, max_batches=max_batches, filter_mode=mode)
        print_stationary_coverage_report(reports[mode], label=f"mode={mode}")
    return reports


# =============================================================================
# v6.2.1 — Bug 1 coordinate-system regression test (SANITY TEST REQUIREMENT D)
# =============================================================================
# Proves that track_and_filter_pair evaluates the flow-stationarity check with the
# CURRENT-frame mask. The synthetic flow field is ego_flow everywhere except inside
# a region that overlaps ONLY the frame-(t+1) mask footprint; a stub tracker (test-only,
# not one of the three production trackers) guarantees the two detections are paired as
# the same object regardless of geometric tracker thresholds, isolating the mask-selection
# line under test. This assertion FAILS if that line reads d_next.mask (the old bug) and
# PASSES if it reads det_curr.mask (the fix) -- verified both ways while developing this patch.
def _self_test_coordinate_fix() -> None:
    h, w = 128, 256
    ego_flow_vec = np.array([2.0, 0.0], dtype=np.float32)

    def _mk_mask(cx, cy, r):
        yy, xx = np.mgrid[0:h, 0:w]
        return ((xx - cx) ** 2 + (yy - cy) ** 2) <= r * r

    # "car" (a DYNAMIC class) keeps class_static=[False] for this single pseudo-track, so
    # estimate_ego_flow() falls back to the upper-quarter-of-the-image reference region
    # (untouched by the residual patch below) instead of self-referencing the tested mask.
    m_curr = _mk_mask(60, 60, 12)     # object's frame-t footprint
    m_next = _mk_mask(150, 60, 12)    # its (tracked) frame-(t+1) footprint, far away
    det_curr = _mask_to_detection(m_curr, class_name="car", score=0.9)
    det_next = _mask_to_detection(m_next, class_name="car", score=0.9)
    assert (np.where(m_curr)[0] < h // 4).sum() == 0 and (np.where(m_next)[0] < h // 4).sum() == 0

    flow = np.tile(ego_flow_vec, (h, w, 1)).astype(np.float32)
    flow[m_next] = ego_flow_vec + np.array([100.0, 0.0], dtype=np.float32)   # residual ONLY at the t+1 footprint

    class _FixedPairTracker:
        """Test-only stub (NOT a production tracker): pairs detections 1:1 by call order,
        so pairing is guaranteed and only the mask-selection line under test matters."""
        def __init__(self):
            self.tracks: List[Track] = []

        def update(self, detections):
            if not self.tracks:
                self.tracks = [Track(track_id=i + 1, mask=d.mask, bbox=d.bbox, centroid=d.centroid,
                                     area=d.area, class_name=d.class_name, score=d.score, hits=1)
                               for i, d in enumerate(detections)]
            else:
                for t, d in zip(self.tracks, detections):
                    t.mask, t.bbox, t.centroid = d.mask, d.bbox, d.centroid
                    t.area, t.class_name, t.score = d.area, d.class_name, d.score
                    t.hits += 1
                    t.time_since_update = 0
            return self.tracks

    g = globals()
    orig_load, orig_flow, orig_build = g["load_sam3_masks_for_frame"], g["compute_dense_flow"], g["build_tracker"]
    g["load_sam3_masks_for_frame"] = lambda path, rgb_np=None: [det_curr] if path == "__curr__" else [det_next]
    g["compute_dense_flow"] = lambda a, b: flow
    g["build_tracker"] = lambda *a, **kw: _FixedPairTracker()
    try:
        dummy_rgb = np.zeros((h, w, 3), dtype=np.uint8)
        pairs = track_and_filter_pair(dummy_rgb, dummy_rgb, "__curr__", "__next__", filter_mode="flow")
    finally:
        g["load_sam3_masks_for_frame"], g["compute_dense_flow"], g["build_tracker"] = orig_load, orig_flow, orig_build

    assert len(pairs) == 1, (
        "Bug 1 coordinate-system regression: track_and_filter_pair rejected a genuinely stationary "
        f"object (got {len(pairs)} pair(s), expected 1). This fails if the flow-stationarity check "
        "is evaluated with det_next.mask instead of det_curr.mask.")
    print("  track_and_filter_pair coordinate-system fix (Bug 1): OK -- current-frame mask used for flow filtering")


_self_test_coordinate_fix()
print("Group 1c real-data coverage diagnostic ready: stationary_pair_coverage_report(), "
     "print_stationary_coverage_report(), compare_filter_modes().")


Batch-level stationary track-pair extraction ready: get_batch_stationary_pairs(batch) -> List[List[Tuple[MaskDetection, MaskDetection]]]
Batch-level tracking + dynamic-mask helpers ready.
  track_and_filter_pair coordinate-system fix (Bug 1): OK -- current-frame mask used for flow filtering
Group 1c real-data coverage diagnostic ready: stationary_pair_coverage_report(), print_stationary_coverage_report(), compare_filter_modes().


## 8 · Self-Supervised Loss Functions (v6 — strict, no GT-depth fallback)

Rules enforced by construction: (1) SSL losses never see GT depth (`gt=None`); (2) missing inputs raise `SSLInputError`; (3) predicted-pose and oracle-pose losses are different experiments routed by `LOSS_SPECS`; (4) all loss maths in fp32. `MinReprojPhotoLoss` covers three behaviours through two flags (`use_min`, `use_automask`): mean-over-sources (Godard 2017), + auto-mask, + per-pixel min over sources (**Monodepth2**, needs `SOURCE_OFFSETS=(-1,+1)`; with a single source it is only "single-source reprojection vs identity").


In [10]:
# =============================================================================
# Self-supervised loss functions for monocular metric depth   (v6 — NO FALLBACKS)
# =============================================================================
#
# DESIGN RULES (v6)
#   1. A self-supervised loss NEVER touches GT depth.  The trainer passes
#      `gt=None, mask=None` to every loss whose LossSpec has uses_gt_depth=False,
#      so any accidental use crashes immediately instead of silently turning
#      into supervised training (this is what invalidated the v5 Group 1).
#   2. A loss that is missing an input it needs (pose, source frame, pred_next…)
#      raises `SSLInputError`.  There is no "if pose is None: return |pred-gt|".
#   3. Predicted-pose and GT-pose (oracle) losses are different experiments and
#      are routed by `LOSS_SPECS` (next cell), never by silent substitution.
#   4. All maths runs in fp32 (the trainer disables autocast around the loss).
#
# Common forward signature (extra keys are absorbed by **kw; `dyn_mask` [B,1,H,W] = 1 on
# potentially-moving-object pixels, used by the temporal / geometric depth-consistency terms):
#   forward(pred, gt=None, mask=None, *, rgb, rgb_srcs, poses, K, has_K,
#           pred_next, pose_next, rgb_next, gt_poses, gt_pose_next, ...)
#     rgb        : [B,3,H,W] current frame, raw [0,1]
#     rgb_srcs   : list of source frames, raw [0,1]  (order = SOURCE_OFFSETS)
#     poses      : list of T_{t->src} (PoseNet)      (same order)
#     gt_poses   : list of T_{t->src} (OXTS)         (oracle runs only)
#     pred_next  : depth predicted on I_{t+1}, pose_next / gt_pose_next : T_{t->t+1}
# =============================================================================

class SSLInputError(RuntimeError):
    """A loss did not receive an input it requires (no supervised fallback exists)."""


def _require(loss_name: str, **named) -> None:
    missing = [k for k, v in named.items() if v is None]
    if missing:
        raise SSLInputError(
            f"{loss_name} requires {missing} but received None. Refusing to fall back to "
            f"GT-depth supervision (see LOSS_SPECS for the inputs this loss declares).")


def _safe_log(x: torch.Tensor) -> torch.Tensor:
    return torch.log(x.clamp(min=EPS))


def _depth(pred: torch.Tensor) -> torch.Tensor:
    return pred.float().clamp(SSL_MIN_DEPTH, SSL_MAX_DEPTH)


def _masked_mean(x: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
    return (x * m).sum() / (m.sum() + EPS)


# ─────────────────────────────────────────────────────────────────────────────
# Building blocks
# ─────────────────────────────────────────────────────────────────────────────
def ssim_loss(pred: torch.Tensor, gt: torch.Tensor,
              mask: torch.Tensor, window_size: int = 7,
              C1: float = 1e-4, C2: float = 9e-4) -> torch.Tensor:
    # Masked SSIM dissimilarity in [0, 1] between two IMAGES (channel-averaged).
    pad = window_size // 2
    kernel = torch.ones(1, 1, window_size, window_size,
                        device=pred.device, dtype=pred.dtype) / (window_size ** 2)

    def lm(x):
        ch = x.shape[1]
        return F.conv2d(x, kernel.expand(ch, 1, window_size, window_size), padding=pad, groups=ch)

    mu_p  = lm(pred); mu_g  = lm(gt)
    mu_pp = lm(pred * pred); mu_gg = lm(gt * gt); mu_pg = lm(pred * gt)
    sigma_p  = (mu_pp - mu_p * mu_p).clamp(min=0)
    sigma_g  = (mu_gg - mu_g * mu_g).clamp(min=0)
    sigma_pg = mu_pg - mu_p * mu_g
    ssim_map = ((2 * mu_p * mu_g + C1) * (2 * sigma_pg + C2)) / (
                (mu_p**2 + mu_g**2 + C1) * (sigma_p + sigma_g + C2))
    loss_map = ((1 - ssim_map) / 2.0).clamp(0, 1).mean(dim=1, keepdim=True)
    return _masked_mean(loss_map, mask)


def photometric_residual(warped: torch.Tensor, target: torch.Tensor,
                         alpha: float = SSIM_ALPHA) -> torch.Tensor:
    # Per-pixel  alpha·SSIM + (1-alpha)·L1   ->  [B,1,H,W]  (Monodepth2).
    ch = warped.shape[1]
    k = (torch.ones(1, 1, 3, 3, device=warped.device, dtype=warped.dtype) / 9.0).expand(ch, 1, 3, 3)
    mu_p  = F.conv2d(warped, k, padding=1, groups=ch)
    mu_g  = F.conv2d(target, k, padding=1, groups=ch)
    mu_pp = F.conv2d(warped * warped, k, padding=1, groups=ch)
    mu_gg = F.conv2d(target * target, k, padding=1, groups=ch)
    mu_pg = F.conv2d(warped * target, k, padding=1, groups=ch)
    sigma_p  = (mu_pp - mu_p * mu_p).clamp(min=0)
    sigma_g  = (mu_gg - mu_g * mu_g).clamp(min=0)
    sigma_pg = mu_pg - mu_p * mu_g
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu_p * mu_g + C1) * (2 * sigma_pg + C2)) / (
                (mu_p**2 + mu_g**2 + C1) * (sigma_p + sigma_g + C2))
    ssim_l = ((1 - ssim_map) / 2.0).clamp(0, 1).mean(dim=1, keepdim=True)
    l1     = (warped - target).abs().mean(dim=1, keepdim=True)
    return alpha * ssim_l + (1 - alpha) * l1


def edge_aware_smoothness_map(disp: torch.Tensor, image: torch.Tensor,
                              decay: float = SMOOTHNESS_DECAY) -> torch.Tensor:
    # Edge-aware smoothness of the mean-normalised disparity (Godard 2017).
    mean_disp = disp.mean(dim=(2, 3), keepdim=True).clamp(min=EPS)
    dn = disp / mean_disp
    gdx = (dn[:, :, :, :-1] - dn[:, :, :, 1:]).abs()
    gdy = (dn[:, :, :-1, :] - dn[:, :, 1:, :]).abs()
    gix = (image[:, :, :, :-1] - image[:, :, :, 1:]).abs().mean(1, keepdim=True)
    giy = (image[:, :, :-1, :] - image[:, :, 1:, :]).abs().mean(1, keepdim=True)
    return (gdx * torch.exp(-decay * gix)).mean() + (gdy * torch.exp(-decay * giy)).mean()


def _warp_sources(rgb_srcs, poses, depth, K, has_K=None):
    # -> lists of (warped_i, valid_i) for each source frame
    out = []
    for src, T_ in zip(rgb_srcs, poses):
        out.append(warp_image(src, depth, T_, K, has_K=has_K))
    return out


def _check_src_pose_lists(name, rgb_srcs, poses):
    _require(name, rgb_srcs=rgb_srcs, poses=poses)
    if len(rgb_srcs) != len(poses) or len(rgb_srcs) == 0:
        raise SSLInputError(f"{name}: got {len(rgb_srcs)} source frames but {len(poses)} poses")


# ─────────────────────────────────────────────────────────────────────────────
# 1. photo_l1 — masked L1 photometric reprojection (Garg et al., ECCV 2016)
# ─────────────────────────────────────────────────────────────────────────────
class PhotoL1Loss(nn.Module):
    # Reference: Garg et al. ECCV 2016 (arXiv:1603.04992).
    # v6: per-pixel channel-mean L1 averaged over VALID pixels only (v5 averaged over all pixels).
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, poses=None,
                K=None, has_K=None, **kw):
        _require("photo_l1", rgb=rgb, K=K)
        _check_src_pose_lists("photo_l1", rgb_srcs, poses)
        d = _depth(pred)
        ls, vf = [], []
        for w, v in _warp_sources(rgb_srcs, poses, d, K, has_K):
            l1 = (w - rgb).abs().mean(dim=1, keepdim=True)
            ls.append(_masked_mean(l1, v)); vf.append(v.mean())
        l = torch.stack(ls).mean()
        return l, {"total": l, "photo_l1": l, "valid_frac": torch.stack(vf).mean()}


# ─────────────────────────────────────────────────────────────────────────────
# 2. photo_ssim — SSIM-only photometric reprojection
# ─────────────────────────────────────────────────────────────────────────────
class PhotoSSIMLoss(nn.Module):
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, poses=None,
                K=None, has_K=None, **kw):
        _require("photo_ssim", rgb=rgb, K=K)
        _check_src_pose_lists("photo_ssim", rgb_srcs, poses)
        d = _depth(pred)
        ls, vf = [], []
        for w, v in _warp_sources(rgb_srcs, poses, d, K, has_K):
            ls.append(ssim_loss(w, rgb, v)); vf.append(v.mean())
        l = torch.stack(ls).mean()
        return l, {"total": l, "ssim": l, "valid_frac": torch.stack(vf).mean()}


# ─────────────────────────────────────────────────────────────────────────────
# 5 (+3). MinReprojPhotoLoss — one class, three behaviours
# ─────────────────────────────────────────────────────────────────────────────
class MinReprojPhotoLoss(nn.Module):
    # alpha·SSIM + (1-alpha)·L1 photometric loss over one or more source frames.
    #   use_min=False, use_automask=False : mean over sources           (Godard 2017 / "photo_ssim_l1")
    #   use_min=False, use_automask=True  : + Monodepth2 auto-mask
    #   use_min=True,  use_automask=True  : per-pixel MIN over sources + auto-mask
    #                                       = Monodepth2 (Godard et al., ICCV 2019)
    # With SOURCE_OFFSETS=(-1,+1) this is the canonical 3-frame Monodepth2 loss.
    # With a single source (+1 only) it degenerates to
    #   "single-source reprojection vs. identity auto-mask" and is NOT full Monodepth2.
    # Each identity error gets ~1e-5 noise BEFORE the min to break ties (as in the Monodepth2 reference code).
    def __init__(self, alpha: float = SSIM_ALPHA,
                 auto_mask_eps: float = MIN_REPROJ_AUTO_MASK_THRESHOLD,
                 use_min: bool = True, use_automask: bool = True):
        super().__init__()
        self.alpha, self.eps = alpha, auto_mask_eps
        self.use_min, self.use_automask = use_min, use_automask

    def extra_repr(self) -> str:
        return f"alpha={self.alpha}, eps={self.eps}, use_min={self.use_min}, use_automask={self.use_automask}"

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, poses=None,
                K=None, has_K=None, **kw):
        _require("MinReprojPhotoLoss", rgb=rgb, K=K)
        _check_src_pose_lists("MinReprojPhotoLoss", rgb_srcs, poses)
        d = _depth(pred)
        errs, valids, idents = [], [], []
        for src, (w, v) in zip(rgb_srcs, _warp_sources(rgb_srcs, poses, d, K, has_K)):
            errs.append(photometric_residual(w, rgb, self.alpha)); valids.append(v)
            if self.use_automask:
                idents.append(photometric_residual(src, rgb, self.alpha))
        E, V = torch.cat(errs, 1), torch.cat(valids, 1)                  # [B,S,H,W]
        any_valid = V.max(dim=1, keepdim=True).values
        if self.use_min:
            reproj = (E + (1.0 - V) * 1e3).min(dim=1, keepdim=True).values
        else:
            reproj = (E * V).sum(1, keepdim=True) / (V.sum(1, keepdim=True) + EPS)
        if self.use_automask:
            Id = torch.cat(idents, 1)
            Id = Id + torch.randn_like(Id) * 1e-5                     # per-source noise BEFORE the min (Monodepth2)
            ident = Id.min(dim=1, keepdim=True).values if self.use_min else Id.mean(dim=1, keepdim=True)
            auto = (reproj < ident + self.eps).float()
        else:
            auto = torch.ones_like(any_valid)
        final = any_valid * auto
        l = _masked_mean(reproj, final)
        return l, {"total": l, "photo": l, "valid_frac": any_valid.mean(),
                   "auto_mask_frac": (auto * any_valid).sum() / (any_valid.sum() + EPS)}


class PhotoSSIML1Loss(MinReprojPhotoLoss):
    # 3. photo_ssim_l1 — alpha·SSIM + (1-alpha)·L1, mean over sources, no min / no auto-mask
    # (Godard et al., CVPR 2017; the de-facto photometric loss in MDE)
    def __init__(self, alpha: float = SSIM_ALPHA):
        super().__init__(alpha=alpha, use_min=False, use_automask=False)


# ─────────────────────────────────────────────────────────────────────────────
# 4. edge_aware_smooth — regulariser ONLY (degenerate constant depth on its own)
# ─────────────────────────────────────────────────────────────────────────────
class EdgeAwareSmoothLoss(nn.Module):
    def __init__(self, decay: float = SMOOTHNESS_DECAY):
        super().__init__()
        self.decay = decay

    def forward(self, pred, gt=None, mask=None, *, rgb=None, **kw):
        _require("edge_aware_smooth", rgb=rgb)
        l = edge_aware_smoothness_map(1.0 / _depth(pred), rgb, self.decay)
        return l, {"total": l, "smooth": l}


# ─────────────────────────────────────────────────────────────────────────────
# 6. multi_scale_photo — multi-RESOLUTION (image-pyramid) Monodepth2 loss
# ─────────────────────────────────────────────────────────────────────────────
class MultiScalePhotoLoss(nn.Module):
    # DA3 emits a single depth map, so Monodepth2's multi-scale-decoder trick is
    # not available.  v5 merely blurred the depth (not a real multi-scale loss).
    # v6 evaluates the min-reprojection loss on an IMAGE PYRAMID: images and depth are
    # area-downsampled by 2^s, K is rescaled, and the per-scale losses are averaged.
    def __init__(self, alpha: float = SSIM_ALPHA, n_scales: int = 4):
        super().__init__()
        self.n_scales = n_scales
        self.base = MinReprojPhotoLoss(alpha=alpha)

    def extra_repr(self) -> str:
        return f"n_scales={self.n_scales}"

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, poses=None,
                K=None, has_K=None, **kw):
        _require("multi_scale_photo", rgb=rgb, K=K)
        _check_src_pose_lists("multi_scale_photo", rgb_srcs, poses)
        H, W = pred.shape[-2:]
        total, parts = 0.0, {}
        for s in range(self.n_scales):
            f = 2 ** s
            if f == 1:
                p_s, r_s, src_s, K_s = pred, rgb, rgb_srcs, K
            else:
                size = (max(H // f, 8), max(W // f, 8))
                p_s   = F.interpolate(pred, size=size, mode="area")
                r_s   = F.interpolate(rgb,  size=size, mode="area")
                src_s = [F.interpolate(x, size=size, mode="area") for x in rgb_srcs]
                K_s   = K.clone()
                K_s[:, 0, :] = K_s[:, 0, :] * (size[1] / W)          # exact width / height ratios
                K_s[:, 1, :] = K_s[:, 1, :] * (size[0] / H)          # (H, W are not divisible by 2^s)
            l_s, _ = self.base(p_s, rgb=r_s, rgb_srcs=src_s, poses=poses, K=K_s, has_K=has_K)
            total = total + l_s / self.n_scales
            parts[f"scale{s}"] = l_s
        return total, {"total": total, **parts}


# ─────────────────────────────────────────────────────────────────────────────
# Shared: depth-consistency residual between t and t+1  (used by 7 and 10)
# ─────────────────────────────────────────────────────────────────────────────
def _depth_consistency_terms(d_curr, d_next, pose_next, K):
    # d_proj    : depth of frame t+1 sampled where the current pixels land
    # d_compute : z-coordinate of the current 3-D points expressed in frame t+1
    # valid     : in-bounds & in front of the camera & both depths positive
    d_proj, vmask = warp_depth(d_next, d_curr, pose_next, K)
    B, _, H, W = d_curr.shape
    pts = _get_backproj(H, W, d_curr.device)(d_curr, _safe_inv_K(K))              # [B,4,HW]
    d_compute = torch.matmul(pose_next.float()[:, :3, :], pts)[:, 2:3].reshape(B, 1, H, W)
    valid = vmask * (d_proj > EPS).float() * (d_compute > EPS).float()
    diff = (d_proj - d_compute).abs() / (d_proj + d_compute + EPS)
    return diff, valid


# ─────────────────────────────────────────────────────────────────────────────
# 7. geometric_consistency — SC-SfMLearner depth-consistency loss (Bian, NeurIPS 2019)
# ─────────────────────────────────────────────────────────────────────────────
class GeometricConsistencyLoss(nn.Module):
    #   L = mean_p |D_proj - D_compute| / (D_proj + D_compute)
    def forward(self, pred, gt=None, mask=None, *, pred_next=None, pose_next=None,
                K=None, has_K=None, dyn_mask=None, **kw):
        _require("geometric_consistency", pred_next=pred_next, pose_next=pose_next, K=K)
        diff, valid = _depth_consistency_terms(_depth(pred), _depth(pred_next), pose_next, K)
        if dyn_mask is not None:                                 # potentially-moving-object pixels
            valid = valid * (1.0 - dyn_mask.float())
        l = _masked_mean(diff, valid)
        return l, {"total": l, "geom_cons": l, "valid_frac": valid.mean()}


# ─────────────────────────────────────────────────────────────────────────────
# 8. feature_metric — feature-space photometric loss (FeatDepth-style, simplified)
# ─────────────────────────────────────────────────────────────────────────────
class FeatureMetricLoss(nn.Module):
    # Compares frozen ImageNet ResNet18 (conv1..layer1) features at 1/4 resolution
    # + a small raw-pixel L1 term.  NOT the learned auto-encoder features of the
    # original FeatDepth (Shu et al., ECCV 2020) — a simplified variant.
    def __init__(self, alpha: float = SSIM_ALPHA):
        super().__init__()
        self.alpha = alpha
        self.fe = None
        self.register_buffer("_mean", torch.tensor(_IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("_std",  torch.tensor(_IMAGENET_STD).view(1, 3, 1, 1))

    def _build_extractor(self, device):
        try:
            try:
                weights = tvm.ResNet18_Weights.DEFAULT
            except Exception:
                weights = None
            rn = tvm.resnet18(weights=weights).to(device)
        except Exception:
            rn = tvm.resnet18(weights=None).to(device)
        for p in rn.parameters():
            p.requires_grad_(False)
        self.fe = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool, rn.layer1).eval()

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_next=None, pose_next=None,
                K=None, has_K=None, **kw):
        _require("feature_metric", rgb=rgb, rgb_next=rgb_next, pose_next=pose_next, K=K)
        if self.fe is None:
            self._build_extractor(pred.device)
        d = _depth(pred)
        warped, vmask = warp_image(rgb_next, d, pose_next, K)
        l_pix = _masked_mean((warped - rgb).abs().mean(1, keepdim=True), vmask)

        with torch.no_grad():
            f_t = self.fe((rgb - self._mean) / self._std)
        f_n = self.fe((rgb_next - self._mean) / self._std).detach()
        hf, wf = f_t.shape[-2:]
        d_f = F.interpolate(d, size=(hf, wf), mode="area")
        K_f = K.clone()
        K_f[:, 0, :] = K_f[:, 0, :] * (wf / rgb.shape[-1])          # fx, cx scale with the WIDTH ratio
        K_f[:, 1, :] = K_f[:, 1, :] * (hf / rgb.shape[-2])          # fy, cy scale with the HEIGHT ratio
        f_w, v_f = warp_image(f_n, d_f, pose_next, K_f)
        l_feat = _masked_mean((f_w - f_t).abs().mean(1, keepdim=True), v_f)
        l = 0.5 * l_feat + 0.5 * l_pix
        return l, {"total": l, "feature": l_feat, "pixel": l_pix, "valid_frac": vmask.mean()}


# ─────────────────────────────────────────────────────────────────────────────
# 9. rigid_flow_consistency — depth+pose induced ("rigid") flow, photometric + smooth
#    (v5 name: flow_depth_consistency, described as GeoNet.  It is NOT GeoNet: GeoNet
#     learns a separate optical-flow branch and enforces rigid/full-flow consistency.)
# ─────────────────────────────────────────────────────────────────────────────
class RigidFlowConsistencyLoss(nn.Module):
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_next=None, pose_next=None,
                K=None, has_K=None, **kw):
        _require("rigid_flow_consistency", rgb=rgb, rgb_next=rgb_next, pose_next=pose_next, K=K)
        d = _depth(pred)
        B, _, H, W = d.shape
        pts  = _get_backproj(H, W, d.device)(d, _safe_inv_K(K))
        pix_in_next = _get_project(H, W, d.device)(pts, K, pose_next)                    # [B,H,W,2] normalised
        u, v = torch.meshgrid(torch.arange(W, dtype=torch.float32, device=d.device),
                              torch.arange(H, dtype=torch.float32, device=d.device), indexing="xy")
        pix0 = torch.stack([2.0 * u / (W - 1) - 1.0, 2.0 * v / (H - 1) - 1.0], dim=-1)
        flow = pix_in_next - pix0.unsqueeze(0)
        warped, vmask = warp_image(rgb_next, d, pose_next, K)
        photo = _masked_mean(photometric_residual(warped, rgb, SSIM_ALPHA), vmask)
        flow_mag = flow.permute(0, 3, 1, 2).pow(2).sum(dim=1, keepdim=True).add(EPS).sqrt()
        flow_smooth = edge_aware_smoothness_map(flow_mag, rgb)
        l = photo + 0.1 * flow_smooth
        return l, {"total": l, "photo": photo, "flow_smooth": flow_smooth, "valid_frac": vmask.mean()}


# ─────────────────────────────────────────────────────────────────────────────
# 10. temporal_warp_consistency — warped-depth consistency (+ small photometric anchor)
# ─────────────────────────────────────────────────────────────────────────────
class TemporalWarpConsistencyLoss(nn.Module):
    # Same residual as SC-SfM's geometric loss, plus an optional photometric term.
    #
    # v6 MASKING FIX.  v5 built its "static" mask from the SAME residual it then
    # minimised (keep pixels with rel_diff < 0.5), i.e. it ignored exactly the
    # pixels that needed correcting.  v6 separates:
    #    visibility mask   : in-bounds, in front of camera, positive depths
    #    dynamic-object mask (optional `dyn_mask`, 1 = moving object)
    #    residual handling : bounded L1 (rel. diff is in [0,1]) — robust by
    #                        construction, no residual-based pixel selection.
    def __init__(self, photo_weight: float = 0.5):
        super().__init__()
        self.photo_weight = photo_weight

    def extra_repr(self) -> str:
        return f"photo_weight={self.photo_weight}"

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_next=None,
                pred_next=None, pose_next=None, K=None, has_K=None, dyn_mask=None, **kw):
        _require("temporal_warp_consistency", pred_next=pred_next, pose_next=pose_next, K=K)
        d_curr, d_next = _depth(pred), _depth(pred_next)
        diff, valid = _depth_consistency_terms(d_curr, d_next, pose_next, K)
        if dyn_mask is not None:
            valid = valid * (1.0 - dyn_mask.float())
        l_tc = _masked_mean(diff, valid)
        with torch.no_grad():                                   # diagnostic only, never used in the loss
            large_frac = ((diff > 0.5).float() * valid).sum() / (valid.sum() + EPS)

        if self.photo_weight > 0:
            _require("temporal_warp_consistency(photo term)", rgb=rgb, rgb_next=rgb_next)
            warped, vm = warp_image(rgb_next, d_curr, pose_next, K)
            l_p = _masked_mean(photometric_residual(warped, rgb, SSIM_ALPHA), vm)
        else:
            l_p = torch.zeros((), device=pred.device)
        l = l_tc + self.photo_weight * l_p
        return l, {"total": l, "temporal_consistency": l_tc, "photo": l_p,
                   "valid_frac": valid.mean(), "large_resid_frac": large_frac}


# ─────────────────────────────────────────────────────────────────────────────
# 11. naive_temporal_delta — HEURISTIC "constant Δd" regulariser (negative control)
# ─────────────────────────────────────────────────────────────────────────────
class NaiveTemporalDeltaLoss(nn.Module):
    def forward(self, pred, gt=None, mask=None, *, pred_next=None, **kw):
        _require("naive_temporal_delta", pred_next=pred_next)
        p_c, p_n = _depth(pred), _depth(pred_next)
        delta = p_n - p_c
        mm = torch.ones_like(delta)                             # no GT-derived mask in an SSL loss
        mean_delta = _masked_mean(delta, mm)
        l = _masked_mean((delta - mean_delta).abs(), mm)
        return l, {"total": l, "naive_delta": l, "batch_mean_delta": mean_delta}


# ─────────────────────────────────────────────────────────────────────────────
# 12. combined_monodepth2 — Monodepth2 recipe (multi-resolution min-reproj + smoothness)
# ─────────────────────────────────────────────────────────────────────────────
class CombinedMonodepth2Loss(nn.Module):
    def __init__(self, w_photo: float = 1.0, w_smooth: float = 1e-3,
                 alpha: float = SSIM_ALPHA, n_scales: int = 4):
        super().__init__()
        self.w_photo, self.w_smooth = w_photo, w_smooth
        self.photo  = MultiScalePhotoLoss(alpha=alpha, n_scales=n_scales)
        self.smooth = EdgeAwareSmoothLoss()

    def extra_repr(self) -> str:
        return f"w_photo={self.w_photo}, w_smooth={self.w_smooth}"

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, poses=None,
                K=None, has_K=None, **kw):
        l_p, d_p = self.photo(pred, rgb=rgb, rgb_srcs=rgb_srcs, poses=poses, K=K, has_K=has_K)
        l_s, _   = self.smooth(pred, rgb=rgb)
        l = self.w_photo * l_p + self.w_smooth * l_s
        return l, {"total": l, "photo": l_p, "smooth": l_s}


# ─────────────────────────────────────────────────────────────────────────────
# SILog helper + supervised control
# ─────────────────────────────────────────────────────────────────────────────
def _silog(pred, gt, mask, lam: float = 0.85):
    _require("SILog anchor", gt=gt, mask=mask)
    p = pred.float().clamp(EPS, MAX_DEPTH).squeeze(1)
    g = gt.squeeze(1).clamp(EPS)
    m = mask.squeeze(1).bool()
    if not m.any():
        return p.sum() * 0.0
    d = _safe_log(p[m]) - _safe_log(g[m])
    return torch.sqrt((d ** 2).mean() - lam * d.mean() ** 2 + EPS)


class LiDARAnchorOnlyLoss(nn.Module):
    # SUPERVISED control: SILog on sparse LiDAR.  Not a self-supervised loss.
    def __init__(self, lam: float = 0.85):
        super().__init__()
        self.lam = lam

    def forward(self, pred, gt=None, mask=None, **kw):
        l = _silog(pred, gt, mask, self.lam)
        return l, {"total": l, "silog": l}


# ─────────────────────────────────────────────────────────────────────────────
# 13. combined_recommended — HYBRID (SSL terms + sparse-LiDAR SILog anchor)
#     w_anchor = 0  ->  pure SSL variant ("combined_recommended_ssl")
# ─────────────────────────────────────────────────────────────────────────────
class CombinedRecommendedLoss(nn.Module):
    #   L = w_photo·L_min_reproj + w_smooth·L_smooth + w_temporal·L_temporal + w_anchor·L_SILog(GT)
    def __init__(self, w_photo=DEFAULT_W_PHOTO, w_smooth=DEFAULT_W_SMOOTH,
                 w_temporal=DEFAULT_W_TEMPORAL, w_anchor=DEFAULT_W_ANCHOR,
                 alpha=SSIM_ALPHA, lam_silog=0.85):
        super().__init__()
        self.w_photo, self.w_smooth = w_photo, w_smooth
        self.w_temporal, self.w_anchor = w_temporal, w_anchor
        self.lam_silog = lam_silog
        self.photo    = MinReprojPhotoLoss(alpha=alpha)
        self.smooth   = EdgeAwareSmoothLoss()
        self.temporal = TemporalWarpConsistencyLoss(photo_weight=0.0)

    @property
    def uses_gt_depth(self) -> bool:
        return self.w_anchor > 0

    def extra_repr(self) -> str:
        return (f"w_photo={self.w_photo}, w_smooth={self.w_smooth}, w_temporal={self.w_temporal}, "
                f"w_anchor={self.w_anchor}, lam_silog={self.lam_silog}")

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, poses=None,
                pred_next=None, pose_next=None, K=None, has_K=None, dyn_mask=None, **kw):
        l_p, _ = self.photo(pred, rgb=rgb, rgb_srcs=rgb_srcs, poses=poses, K=K, has_K=has_K)
        l_s, _ = self.smooth(pred, rgb=rgb)
        total = self.w_photo * l_p + self.w_smooth * l_s
        terms = {"photo": l_p, "smooth": l_s}
        if self.w_temporal > 0:
            l_t, _ = self.temporal(pred, pred_next=pred_next, pose_next=pose_next, K=K, dyn_mask=dyn_mask)
            total = total + self.w_temporal * l_t; terms["temporal"] = l_t
        if self.w_anchor > 0:
            l_a = _silog(pred, gt, mask, self.lam_silog)
            total = total + self.w_anchor * l_a; terms["anchor"] = l_a
        terms["total"] = total
        return total, terms


# =============================================================================
# ORACLE-POSE variants (GT OXTS pose + learned depth) — NOT self-supervised
# =============================================================================
# Strict wrappers: they require `gt_poses` / `gt_pose_next` and never look at the
# PoseNet output.  If OXTS pose is missing they raise (no silent substitution).
class GTPosePhotoReprojLoss(MinReprojPhotoLoss):
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None,
                gt_poses=None, K=None, has_K=None, **kw):
        _require("gt_photo_reproj", gt_poses=gt_poses)
        return super().forward(pred, rgb=rgb, rgb_srcs=rgb_srcs, poses=gt_poses, K=K, has_K=has_K)


class GTPoseRigidFlowLoss(RigidFlowConsistencyLoss):
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_next=None,
                gt_pose_next=None, K=None, has_K=None, **kw):
        _require("gt_rigid_flow_consistency", gt_pose_next=gt_pose_next)
        return super().forward(pred, rgb=rgb, rgb_next=rgb_next, pose_next=gt_pose_next, K=K, has_K=has_K)


class GTPoseTemporalWarpLoss(TemporalWarpConsistencyLoss):
    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_next=None, pred_next=None,
                gt_pose_next=None, K=None, has_K=None, dyn_mask=None, **kw):
        _require("gt_temporal_warp_consistency", gt_pose_next=gt_pose_next)
        return super().forward(pred, rgb=rgb, rgb_next=rgb_next, pred_next=pred_next,
                               pose_next=gt_pose_next, K=K, has_K=has_K, dyn_mask=dyn_mask)


# =============================================================================
# Registry  (the routing requirements live in LOSS_SPECS — next-but-one cell)
# =============================================================================
LOSS_REGISTRY: Dict[str, nn.Module] = {
    # ── self-supervised, predicted pose ───────────────────────────────────
    "photo_l1":                  PhotoL1Loss(),
    "photo_ssim":                PhotoSSIMLoss(),
    "photo_ssim_l1":             PhotoSSIML1Loss(),
    "min_reproj_photo":          MinReprojPhotoLoss(),
    "multi_scale_photo":         MultiScalePhotoLoss(),
    "geometric_consistency":     GeometricConsistencyLoss(),
    "feature_metric":            FeatureMetricLoss(),
    "rigid_flow_consistency":    RigidFlowConsistencyLoss(),
    "temporal_warp_consistency": TemporalWarpConsistencyLoss(),
    "combined_monodepth2":       CombinedMonodepth2Loss(),
    "combined_recommended_ssl":  CombinedRecommendedLoss(w_anchor=0.0),
    # ── regulariser / negative control (no pose) ──────────────────────────
    "edge_aware_smooth":         EdgeAwareSmoothLoss(),
    "naive_temporal_delta":      NaiveTemporalDeltaLoss(),
    # ── hybrid + supervised control (use sparse GT depth ON PURPOSE) ──────
    "combined_recommended":      CombinedRecommendedLoss(),
    "lidar_anchor_only":         LiDARAnchorOnlyLoss(),
    # ── oracle pose (GT OXTS pose, learned depth) ─────────────────────────
    "gt_photo_reproj":              GTPosePhotoReprojLoss(),
    "gt_rigid_flow_consistency":    GTPoseRigidFlowLoss(),
    "gt_temporal_warp_consistency": GTPoseTemporalWarpLoss(),
}

# Pre-build FeatureMetricLoss extractor once (avoids a mid-training download hang)
try:
    LOSS_REGISTRY["feature_metric"]._build_extractor(torch.device("cpu"))
    log.info("FeatureMetricLoss extractor pre-loaded on CPU.")
except Exception as _e:
    log.warning("FeatureMetricLoss extractor pre-load failed (%s) — built lazily instead.", _e)

print(f"Loss registry (v6): {len(LOSS_REGISTRY)} variants, all strict / no GT-depth fallback")


22:39:34  INFO      FeatureMetricLoss extractor pre-loaded on CPU.


Loss registry (v6): 18 variants, all strict / no GT-depth fallback


## 9 · Stationary-Object Ego-Motion Losses (5 variants, Part 5)

### Purpose

Every loss in the original photometric/geometric bank (Section 8) supervises depth through **appearance matching**: photometric residuals, SSIM, feature-metric distance. None of them ever check whether the model's depth predictions are **metrically self-consistent** with the camera's own known motion. These four new losses do exactly that, using the tracked-and-filtered stationary objects from Parts 3–4 as anchors, then a fifth loss combines them with the existing photometric/temporal terms into one configurable recipe.

### Shared pose convention (read this before Loss 1)

All four losses transform points using `pose_curr_to_next`, the **same** SE(3) convention already used by `Project3D`/`warp_image` above: it maps a point's *coordinates* from the current camera's frame into the next camera's frame, $p_{\text{next}} = R\,p_{\text{curr}} + t$. `ENABLE_GT_POSE` (Part 7) selects the OXTS `gt_pose` over the PoseNet-predicted `pose` when both are available — see the corrected `_build_pairs` cell above for why getting this direction right matters (an earlier version of `gt_pose` had it backwards).

### Loss 1 — Stationary Object Depth Motion Loss

$$\mathcal{L}_1 = \frac{1}{|\mathcal{O}|}\sum_{o \in \mathcal{O}} \left| \hat{d}_{t+1}^{(o)} - \big[R\,P_t^{(o)} + t\big]_z \right|$$

where $P_t^{(o)}$ is object $o$'s frame-$t$ 3D point (backprojected from its **robust median depth** inside the mask), and $\hat{d}_{t+1}^{(o)}$ is the model's own median depth at $o$'s frame-$(t{+}1)$ mask. Ties the depth *scale* at two different image locations together through a physically known transform, rather than only through appearance.

### Loss 2 — Stationary Uniform Motion Loss

$$\mathcal{L}_2 = \frac{1}{|\mathcal{B}|}\sum_{b \in \mathcal{B}} \mathrm{Var}_{o \in \mathcal{O}_b}\!\left(\hat{d}_{t+1}^{(o)} - \big[R\,P_t^{(o)} + t\big]_z\right)$$

Every stationary object in the *same* frame pair experiences the *same* ego-motion, so their **signed** residuals should agree with each other, independent of whether that shared residual is exactly zero. This is a strictly local consistency signal (per sample, across objects) that a global mean-toward-zero loss like Loss 1 cannot enforce by itself — it can be satisfied by a systematic bias that merely averages to zero. Needs $\geq 2$ stationary objects per sample; sparser samples are skipped.

### Loss 3 — Full 3D Ego Motion Loss (Part 9: the principled upgrade of Loss 1)

$$\mathcal{L}_3 = \frac{1}{|\mathcal{O}|}\sum_{o \in \mathcal{O}} \left\| P_{t+1,\text{obs}}^{(o)} - \big(R\,P_{t}^{(o)} + t\big) \right\|_2$$

Compares the **full 3D centroid**, not just its $Z$-component. A depth-only comparison (Loss 1) is blind to $X/Y$ errors that the pixel location + intrinsics encode — e.g. a tracking/segmentation registration error between the mask in frame $t$ and frame $t{+}1$ is invisible to Loss 1 but directly penalized here. This is the formulation Part 9 asks to prefer when in doubt; both are kept as separate configurable losses so the ablation study can quantify the difference empirically rather than assume it.

### Loss 4 — Mask Point Cloud Chamfer Loss

$$\mathcal{L}_4 = \frac{1}{|\mathcal{O}|}\sum_{o \in \mathcal{O}} \mathrm{Chamfer}\big(R\,\mathrm{PC}_t^{(o)} + t,\ \mathrm{PC}_{t+1}^{(o)}\big)$$

Losses 1–3 reduce each object to one representative point; this loss instead compares **shape**: up to `CHAMFER_MAX_POINTS` pixels are sampled from the mask, each backprojected with its *own* per-pixel predicted depth (not the median), transformed, and compared via bidirectional nearest-neighbour distance against an equivalently-sampled point cloud from frame $t{+}1$. Catches cases where the median depth matches but the object's internal relief is wrong. $O(N_a N_b)$ per object — gated behind `ENABLE_CHAMFER` (default off) for this reason.

### Loss 5 — Combined Stationary Consistency Loss

$$\mathcal{L} = w_p\mathcal{L}_{\text{photo}} + w_s\mathcal{L}_{\text{smooth}} + w_\tau\mathcal{L}_{\text{temporal}} + w_1\mathcal{L}_1 + w_2\mathcal{L}_2 + w_3\mathcal{L}_3 + w_4\mathcal{L}_4$$

Reuses the existing `MinReprojPhotoLoss` / `EdgeAwareSmoothLoss` / `TemporalWarpConsistencyLoss` instances (no duplicated math) plus Losses 1–4 above, each independently weighted; $\mathcal{L}_2$ and $\mathcal{L}_4$ are individually gate-able (`ENABLE_UNIFORMITY`, `ENABLE_CHAMFER`) since they are the priciest/most experimental terms.

### Computational complexity

Losses 1–3: $O(|\mathcal{O}|)$ per batch (one median + one 3×3 transform per object). Loss 4: $O(|\mathcal{O}| \cdot \text{CHAMFER\_MAX\_POINTS}^2)$. All four are Python-level loops over a small number of tracked objects per sample (typically single digits to low tens on KITTI), not vectorized across objects — acceptable for this batch size but the dominant remaining cost is the CPU-side tracking pass in Part 4a, not the loss math itself.

### Limitations

All four losses are only as good as the upstream tracking + dynamic filter: a moving object misclassified as stationary injects a false ego-motion constraint into every one of them. Median-depth robustness (Losses 1–3) helps against segmentation noise but not against a systematically wrong depth prediction inside the whole mask. Loss 2's variance signal is undefined (skipped) for scenes with fewer than 2 trustworthy stationary objects — common on KITTI's more rural sequences.

### References
- Zhou et al., *"Unsupervised Learning of Depth and Ego-Motion from Video"*, CVPR 2017 (the ego-motion-as-supervision idea this section generalizes from full-frame photometric consistency to per-object geometric consistency).
- Fan et al., *"A Point Set Generation Network for 3D Object Reconstruction from a Single Image"*, CVPR 2017 (Chamfer distance, Loss 4).

> **v6.2.1:** the "all four standalone losses received zero objects" failure mode described in the Limitations above (Part 7's tracking/filtering pipeline) was traced to two upstream bugs — a frame-t/frame-(t+1) coordinate mismatch in the flow-stationarity check and an unnormalized SAM3 class-label comparison — both fixed in Part 4 / Part 7. The `_empty()` zero-loss-with-gradient behavior in this cell is unchanged: a genuinely empty stationary-object set still contributes nothing, by design.


In [11]:
# =============================================================================
# Stationary-object ego-motion losses — Part 5, Losses 1-5
# (v6.1: the Part 9 improvements — erosion, area weighting, MAD dispersion — are merged INTO these
#  classes; the former monkey-patch cell is gone.)
# =============================================================================

def _erode_mask(mask_np: np.ndarray, px: int = MASK_ERODE_PX) -> np.ndarray:
    """Morphological erosion (Part 9, Finding 3): strip boundary pixels before any depth statistic is
    computed. No-op for px<=0 or masks too small to survive erosion (falls back to the original mask)."""
    if px <= 0:
        return mask_np
    kernel = np.ones((2 * px + 1, 2 * px + 1), dtype=np.uint8)
    eroded = cv2.erode(mask_np.astype(np.uint8), kernel, iterations=1).astype(bool)
    return eroded if eroded.sum() >= 5 else mask_np


def _weighted_mean(values: List[torch.Tensor], weights: List[float]) -> torch.Tensor:
    w = torch.tensor(weights, dtype=values[0].dtype, device=values[0].device)
    w = w / w.sum()
    return (torch.stack(values) * w).sum()


class _StationaryObjectLossBase(nn.Module):
    """
    Shared utilities for Losses 1-4 (not used directly). See markdown above
    for the shared pose convention and per-object recipe.
    """

    @staticmethod
    def _pick_pose(pose: Optional[torch.Tensor], gt_pose: Optional[torch.Tensor]) -> Optional[torch.Tensor]:
        if ENABLE_GT_POSE and gt_pose is not None:
            return gt_pose
        return pose

    @staticmethod
    def _resize_mask(mask_np: np.ndarray, H: int, W: int) -> torch.Tensor:
        mask_np = _erode_mask(mask_np)                       # Part 9, Finding 3 (merged from the former patch cell)
        if mask_np.shape != (H, W):
            mask_np = cv2.resize(mask_np.astype(np.uint8), (W, H),
                                 interpolation=cv2.INTER_NEAREST).astype(bool)
        return torch.from_numpy(mask_np)

    @staticmethod
    def _scale_centroid(cx: float, cy: float, native_hw: Tuple[int, int],
                        target_hw: Tuple[int, int]) -> Tuple[float, float]:
        nh, nw = native_hw
        th, tw = target_hw
        return cx * (tw / max(nw, 1)), cy * (th / max(nh, 1))

    @staticmethod
    def _median_depth_in_mask(depth_map: torch.Tensor, mask_t: torch.Tensor) -> Optional[torch.Tensor]:
        # depth_map: [H, W] single sample; mask_t: bool [H, W], same device.
        vals = depth_map[mask_t]
        if vals.numel() < 5:
            return None
        return torch.median(vals)

    @staticmethod
    def _backproject(u: float, v: float, depth_scalar: torch.Tensor, K_row: torch.Tensor) -> torch.Tensor:
        fx, fy = K_row[0, 0], K_row[1, 1]
        cx_p, cy_p = K_row[0, 2], K_row[1, 2]
        X = (u - cx_p) / fx * depth_scalar
        Y = (v - cy_p) / fy * depth_scalar
        return torch.stack([X, Y, depth_scalar])

    def _iter_batch_objects(self, pred: torch.Tensor, pred_next: torch.Tensor,
                            K: torch.Tensor, eff_pose: torch.Tensor,
                            stationary_pairs: List[List[Tuple[MaskDetection, MaskDetection]]]):
        """
        Yields (P_curr, P_next_expected, P_next_observed, d_curr, d_next,
        sample_idx, weight) for every stationary tracked object in the batch whose
        median depth is extractable in both frames.  weight = sqrt(mean mask area)
        (Part 9, Finding 2: tiny noisy masks must not dominate the average).
        """
        B = pred.shape[0]
        H, W = pred.shape[-2], pred.shape[-1]
        for b in range(min(B, len(stationary_pairs))):
            pairs_b = stationary_pairs[b]
            if not pairs_b:
                continue
            depth_curr = pred[b, 0]
            depth_next = pred_next[b, 0]
            K_b, T_b = K[b], eff_pose[b]
            for det_curr, det_next in pairs_b:
                mask_c = self._resize_mask(det_curr.mask, H, W).to(depth_curr.device)
                mask_n = self._resize_mask(det_next.mask, H, W).to(depth_next.device)
                d_curr = self._median_depth_in_mask(depth_curr, mask_c)
                d_next = self._median_depth_in_mask(depth_next, mask_n)
                if d_curr is None or d_next is None:
                    continue
                u_c, v_c = self._scale_centroid(det_curr.centroid[0], det_curr.centroid[1],
                                                det_curr.mask.shape, (H, W))
                u_n, v_n = self._scale_centroid(det_next.centroid[0], det_next.centroid[1],
                                                det_next.mask.shape, (H, W))
                P_curr = self._backproject(u_c, v_c, d_curr, K_b)
                P_next_obs = self._backproject(u_n, v_n, d_next, K_b)
                P_next_exp = T_b[:3, :3] @ P_curr + T_b[:3, 3]
                weight = math.sqrt(max((det_curr.area + det_next.area) / 2.0, 1.0))
                yield P_curr, P_next_exp, P_next_obs, d_curr, d_next, b, weight

    @staticmethod
    def _empty(pred: torch.Tensor, extra: Dict) -> Tuple[torch.Tensor, Dict]:
        l = pred.sum() * 0.0
        return l, {"total": l, **extra}


class StationaryObjectDepthMotionLoss(_StationaryObjectLossBase):
    """Part 5, Loss 1 — area-weighted mean of |d_next − z_expected| over stationary objects."""

    def forward(self, pred, gt, mask, pred_next=None, K=None, pose=None,
                gt_pose=None, stationary_pairs=None, **kw):
        eff_pose = self._pick_pose(pose, gt_pose)
        _require(type(self).__name__, pred_next=pred_next, pose=eff_pose, stationary_pairs=stationary_pairs)
        residuals, weights = [], []
        for P_curr, P_next_exp, P_next_obs, d_curr, d_next, b, w in \
                self._iter_batch_objects(pred, pred_next, K, eff_pose, stationary_pairs):
            residuals.append((d_next - P_next_exp[2]).abs())
            weights.append(w)
        if not residuals:
            return self._empty(pred, {"n_objects": 0})
        l = _weighted_mean(residuals, weights)
        return l, {"total": l, "stationary_depth_motion": l, "n_objects": len(residuals)}


class StationaryUniformMotionLoss(_StationaryObjectLossBase):
    """Part 5, Loss 2 — dispersion (MAD by default, Part 9 Finding 4; variance via UNIFORMITY_METRIC)
    of the signed per-object residuals within a sample."""

    def forward(self, pred, gt, mask, pred_next=None, K=None, pose=None,
                gt_pose=None, stationary_pairs=None, **kw):
        eff_pose = self._pick_pose(pose, gt_pose)
        _require(type(self).__name__, pred_next=pred_next, pose=eff_pose, stationary_pairs=stationary_pairs)

        per_sample: Dict[int, List[torch.Tensor]] = {}
        for P_curr, P_next_exp, P_next_obs, d_curr, d_next, b, w in \
                self._iter_batch_objects(pred, pred_next, K, eff_pose, stationary_pairs):
            per_sample.setdefault(b, []).append(d_next - P_next_exp[2])  # signed, not abs

        dispersions = []
        for resids in per_sample.values():
            if len(resids) < 2:
                continue
            r = torch.stack(resids)
            if UNIFORMITY_METRIC == "mad":
                dispersions.append((r - r.median()).abs().mean())
            else:  # "variance" — original Stage 2 behaviour, kept for ablation comparison
                dispersions.append(r.var(unbiased=False))
        if not dispersions:
            return self._empty(pred, {"n_samples_used": 0})
        l = torch.stack(dispersions).mean()
        return l, {"total": l, "stationary_uniformity": l, "n_samples_used": len(dispersions)}


class Full3DEgoMotionLoss(_StationaryObjectLossBase):
    """Part 5, Loss 3 — area-weighted mean of ‖P_next_obs − P_next_exp‖₂ (Part 9 preferred over Loss 1)."""

    def forward(self, pred, gt, mask, pred_next=None, K=None, pose=None,
                gt_pose=None, stationary_pairs=None, **kw):
        eff_pose = self._pick_pose(pose, gt_pose)
        _require(type(self).__name__, pred_next=pred_next, pose=eff_pose, stationary_pairs=stationary_pairs)
        residuals, weights = [], []
        for P_curr, P_next_exp, P_next_obs, d_curr, d_next, b, w in \
                self._iter_batch_objects(pred, pred_next, K, eff_pose, stationary_pairs):
            residuals.append((P_next_obs - P_next_exp).norm(p=2))
            weights.append(w)
        if not residuals:
            return self._empty(pred, {"n_objects": 0})
        l = _weighted_mean(residuals, weights)
        return l, {"total": l, "full_3d_ego_motion": l, "n_objects": len(residuals)}


def chamfer_distance(pc_a: torch.Tensor, pc_b: torch.Tensor) -> torch.Tensor:
    """Bidirectional mean squared nearest-neighbour distance between point
    clouds [Na, 3] / [Nb, 3] (Fan et al., CVPR 2017), via `torch.cdist` —
    adequate since each cloud is capped at CHAMFER_MAX_POINTS points."""
    d = torch.cdist(pc_a.unsqueeze(0), pc_b.unsqueeze(0)).squeeze(0)
    return d.min(dim=1).values.pow(2).mean() + d.min(dim=0).values.pow(2).mean()


class MaskPointCloudChamferLoss(_StationaryObjectLossBase):
    """Part 5, Loss 4 — see markdown above. Gated behind ENABLE_CHAMFER."""

    def __init__(self, max_points: int = CHAMFER_MAX_POINTS):
        super().__init__()
        self.max_points = max_points

    def _sample_pointcloud(self, depth_map: torch.Tensor, mask_t: torch.Tensor,
                           K_row: torch.Tensor) -> Optional[torch.Tensor]:
        ys, xs = torch.where(mask_t)
        n = ys.shape[0]
        if n < 5:
            return None
        if n > self.max_points:
            idx = torch.randperm(n, device=ys.device)[: self.max_points]
            ys, xs = ys[idx], xs[idx]
        depths = depth_map[ys, xs]
        fx, fy = K_row[0, 0], K_row[1, 1]
        cx_p, cy_p = K_row[0, 2], K_row[1, 2]
        X = (xs.float() - cx_p) / fx * depths
        Y = (ys.float() - cy_p) / fy * depths
        return torch.stack([X, Y, depths], dim=1)

    def forward(self, pred, gt, mask, pred_next=None, K=None, pose=None,
                gt_pose=None, stationary_pairs=None, **kw):
        eff_pose = self._pick_pose(pose, gt_pose)
        _require(type(self).__name__, pred_next=pred_next, pose=eff_pose, stationary_pairs=stationary_pairs)

        B = pred.shape[0]
        H, W = pred.shape[-2], pred.shape[-1]
        residuals = []
        for b in range(min(B, len(stationary_pairs))):
            pairs_b = stationary_pairs[b]
            if not pairs_b:
                continue
            depth_curr, depth_next = pred[b, 0], pred_next[b, 0]
            K_b, T_b = K[b], eff_pose[b]
            for det_curr, det_next in pairs_b:
                mask_c = self._resize_mask(det_curr.mask, H, W).to(depth_curr.device)
                mask_n = self._resize_mask(det_next.mask, H, W).to(depth_next.device)
                pc_curr = self._sample_pointcloud(depth_curr, mask_c, K_b)
                pc_next = self._sample_pointcloud(depth_next, mask_n, K_b)
                if pc_curr is None or pc_next is None:
                    continue
                pc_curr_t = (T_b[:3, :3] @ pc_curr.T).T + T_b[:3, 3]
                residuals.append(chamfer_distance(pc_curr_t, pc_next))

        if not residuals:
            return self._empty(pred, {"n_objects": 0})
        l = torch.stack(residuals).mean()
        return l, {"total": l, "chamfer": l, "n_objects": len(residuals)}


class CombinedStationaryConsistencyLoss(nn.Module):
    """Part 5, Loss 5 — see markdown above."""

    def extra_repr(self) -> str:
        return self._w_repr

    def __init__(self,
                 w_photo: float = DEFAULT_W_PHOTO, w_smooth: float = DEFAULT_W_SMOOTH,
                 w_temporal: float = DEFAULT_W_TEMPORAL, w_stationary: float = DEFAULT_W_STATIONARY,
                 w_uniformity: float = DEFAULT_W_UNIFORMITY, w_3d_motion: float = DEFAULT_W_3D_MOTION,
                 w_chamfer: float = DEFAULT_W_CHAMFER, alpha: float = SSIM_ALPHA):
        super().__init__()
        self.w_photo, self.w_smooth, self.w_temporal = w_photo, w_smooth, w_temporal
        self.w_stationary, self.w_uniformity = w_stationary, w_uniformity
        self.w_3d_motion, self.w_chamfer = w_3d_motion, w_chamfer
        self._w_repr = (f"w_photo={w_photo}, w_smooth={w_smooth}, w_temporal={w_temporal}, w_stationary={w_stationary}, "
                        f"w_uniformity={w_uniformity}, w_3d_motion={w_3d_motion}, w_chamfer={w_chamfer}")
        self.photo = MinReprojPhotoLoss(alpha=alpha)
        self.smooth = EdgeAwareSmoothLoss()
        self.temporal = TemporalWarpConsistencyLoss(photo_weight=0.0)
        self.stationary_depth = StationaryObjectDepthMotionLoss()
        self.uniformity = StationaryUniformMotionLoss()
        self.full_3d = Full3DEgoMotionLoss()
        self.chamfer = MaskPointCloudChamferLoss()

    def forward(self, pred, gt=None, mask=None, rgb=None, rgb_srcs=None, poses=None,
                pred_next=None, pose=None, gt_pose=None, K=None, has_K=None,
                stationary_pairs=None, **kw):
        l_p, _ = self.photo(pred, rgb=rgb, rgb_srcs=rgb_srcs, poses=poses, K=K, has_K=has_K)
        l_s, _ = self.smooth(pred, rgb=rgb)
        l_t, _ = self.temporal(pred, rgb=rgb, pred_next=pred_next, pose_next=pose, K=K, dyn_mask=kw.get("dyn_mask"))
        l_st, d_st = self.stationary_depth(pred, gt, mask, pred_next=pred_next, K=K,
                                           pose=pose, gt_pose=gt_pose, stationary_pairs=stationary_pairs)
        l_3d, d_3d = self.full_3d(pred, gt, mask, pred_next=pred_next, K=K,
                                  pose=pose, gt_pose=gt_pose, stationary_pairs=stationary_pairs)

        total = (self.w_photo * l_p + self.w_smooth * l_s + self.w_temporal * l_t +
                 self.w_stationary * l_st + self.w_3d_motion * l_3d)
        terms = {"total": total, "photo": l_p, "smooth": l_s, "temporal": l_t,
                 "stationary_depth_motion": l_st, "full_3d_ego_motion": l_3d,
                 "n_objects": d_st.get("n_objects", 0)}

        if ENABLE_UNIFORMITY:
            l_u, d_u = self.uniformity(pred, gt, mask, pred_next=pred_next, K=K,
                                       pose=pose, gt_pose=gt_pose, stationary_pairs=stationary_pairs)
            total = total + self.w_uniformity * l_u
            terms["stationary_uniformity"] = l_u
            terms["n_samples_used"] = d_u.get("n_samples_used", 0)

        if ENABLE_CHAMFER:
            l_c, d_c = self.chamfer(pred, gt, mask, pred_next=pred_next, K=K,
                                    pose=pose, gt_pose=gt_pose, stationary_pairs=stationary_pairs)
            total = total + self.w_chamfer * l_c
            terms["chamfer"] = l_c

        terms["total"] = total
        return total, terms


# ─── Register the 5 stationary-object losses ──────────────────────────────────────────────
LOSS_REGISTRY.update({
    "stationary_depth_motion":     StationaryObjectDepthMotionLoss(),
    "stationary_uniform_motion":   StationaryUniformMotionLoss(),
    "full_3d_ego_motion":          Full3DEgoMotionLoss(),
    "mask_chamfer":                MaskPointCloudChamferLoss(),
    "combined_stationary":         CombinedStationaryConsistencyLoss(),
})

NEED_TRACKING_LOSSES = {
    "stationary_depth_motion", "stationary_uniform_motion",
    "full_3d_ego_motion", "mask_chamfer", "combined_stationary",
}
# Routing requirements (pose / pred_next / tracking) are declared in LOSS_SPECS (next-but-one cell).
print(f"Loss registry updated: {len(LOSS_REGISTRY)} total loss variants "
      f"({len(NEED_TRACKING_LOSSES)} Part 5 stationary-object variants added)")


Loss registry updated: 23 total loss variants (5 Part 5 stationary-object variants added)


## 9a · Part 9 — Critical Evaluation & Principled Improvements

Reviewing Stages 1–2 against alternatives, rather than assuming the first working formulation was the best one.

### Finding 1 — the tracker's multi-frame machinery is architecturally inert in this pipeline (no tracker is validated as optimal)

`track_and_filter_pair` (Section 7) constructs a **fresh tracker instance per training pair** and calls `.update()` on it exactly twice — once to seed tracks from frame $t$'s detections, once to match frame $t{+}1$'s. This is a direct consequence of the dataset yielding independent `(t, t{+}1)` pairs rather than persistent multi-frame sequences (a structural property of `KITTISequentialDataset` I did not change — restructuring it into a sequence-yielding dataset would touch nearly every downstream cell and was judged out of scope for this pass).

The consequence: **ByteTrackMaskTracker's Kalman filter never has track history to estimate velocity from** — every track is freshly spawned with zero initial velocity, so its `predict()` call on the only update it ever sees returns the *last observed position unchanged*. The two-stage high/low-confidence matching still functions (and is a real advantage of Tracker 3), but the constant-velocity motion model that is ByteTrack's other core contribution provides **zero benefit** in this specific 2-frame-only integration. Tracker 1 similarly gets no advantage from being "online" across many frames — it only ever sees one match decision.

**Conclusion (v6.1, neutral):** the tracker is a *configuration choice* (`TRACKER_TYPE`, currently `"bytetrack"`). Hungarian assignment is the exact solver for a single two-frame bipartite match, while ByteTrack's Kalman/velocity machinery is inert with fresh per-pair trackers; **no held-out tracking-quality evaluation exists**, so no tracker is claimed to be optimal. The three trackers remain available for an empirical comparison.

### Finding 2 — unweighted object averaging in Losses 1 & 3 lets tiny, noisy masks dominate

Losses 1 and 3 previously averaged residuals across all stationary objects with equal weight. A 6-pixel sliver of a distant pole and a 400-pixel parked car contribute equally — but the pole's median-depth estimate is far noisier (fewer samples, more sensitive to a single mis-segmented boundary pixel). **Fixed:** both losses now use an area-weighted mean, weight $= \sqrt{\text{mean object area in pixels}}$ (square-root rather than linear weighting, so one very large object can't fully dominate the batch either — a standard compromise between "trust bigger samples more" and "don't let one object erase all others").

### Finding 3 — boundary pixels contaminate the median depth estimate

SAM3 masks, like any segmentation output, are noisiest at object boundaries — a handful of background pixels routinely leak into the mask along silhouette edges. The median is already robust to a *few* such outliers, but on small objects (where boundary pixels are a larger fraction of the total) this can meaningfully bias the estimate. **Fixed:** masks are now morphologically eroded by `MASK_ERODE_PX` pixels before any depth statistic is computed (Losses 1–4 and the new geometric filter below all share this through `_resize_mask`), trading a small amount of usable mask area for a cleaner depth sample. Masks that erode to near-empty are simply skipped (already handled by the existing `< 5 valid pixels` guard).

### Finding 4 — variance is a poor dispersion metric for Loss 2

Variance is quadratic in the residual, so a single badly-tracked or partially-dynamic object dominates $\mathcal{L}_2$'s gradient and can drown out the signal from every well-behaved object in the same sample — exactly the failure mode Loss 2 exists to catch becomes the thing that breaks it. **Fixed:** `UNIFORMITY_METRIC = "mad"` (default) switches Loss 2 to mean absolute deviation from the per-sample median residual, a linear (not quadratic) and outlier-resistant dispersion measure. `"variance"` remains available for the ablation study to quantify the difference empirically rather than assume it.

### Finding 5 — a more robust stationary-object filter is possible, with an honest caveat

Part 4's class + optical-flow filtering is upstream of, and independent from, the depth model — it can't use anything the model itself believes about the scene. A natural additional signal: an object whose Loss-1-style residual is a strong *outlier relative to the other stationary objects in the same sample* is more likely mis-tracked or actually moving (despite passing class+flow filtering) than to be a genuine ego-motion violation the model should learn to fix. **Implemented as `geometric_consistency_filter`** (below): per-sample, MAD-based rejection of objects whose residual exceeds `GEOMETRIC_FILTER_MAD_THRESHOLD` MADs from that sample's median residual, computed under `torch.no_grad()` so it never receives gradient.

This is deliberately **off by default** (`ENABLE_GEOMETRIC_FILTER = False`) and gated by a warmup period (`GEOMETRIC_FILTER_WARMUP_EPOCHS`) when enabled, for a reason worth stating plainly rather than glossing over: **this filter uses the very depth predictions the loss is meant to improve.** Early in training, when depth is close to random, this can't distinguish "genuinely moving object" from "the model is currently wrong about everything" — enabling it from epoch 1 risks the model's own early mistakes being treated as confirmed exclusions, a self-reinforcing loop. The warmup delay and MAD-relative (rather than absolute) threshold both mitigate this, but do not eliminate it; this is offered as an ablation option (Group 6 territory) for you to validate empirically, not asserted as a strict improvement.

### What I deliberately did *not* change

- **Loss 3's preference over Loss 1** (already established in Section 9's original markdown) stands — nothing in this review reversed it.
- **The dataset's pair-based structure** — restructuring to persistent sequences would be the "correct" fix for Finding 1's root cause, but is a much larger architectural change than this review's scope; flagged as future work instead of attempted here.
- **Chamfer distance's algorithm** (Fan et al. 2017 bidirectional nearest-neighbour) — this is already the standard formulation; no more principled alternative was identified worth the added complexity (e.g. Earth Mover's Distance is more geometrically faithful but $O(N^3)$ and not justified at `CHAMFER_MAX_POINTS`-scale point clouds).


In [12]:
# =============================================================================
# Part 9, Finding 5 — optional geometric-consistency outlier rejection
# (Findings 2-4 — area weighting, erosion, MAD dispersion — now live inside the loss classes above.)
# =============================================================================

@torch.no_grad()
def geometric_consistency_filter(stationary_pairs: List[List[Tuple[MaskDetection, MaskDetection]]],
                                 pred: torch.Tensor, pred_next: torch.Tensor,
                                 K: torch.Tensor, eff_pose: torch.Tensor,
                                 mad_threshold: float = GEOMETRIC_FILTER_MAD_THRESHOLD,
                                 min_objects: int = GEOMETRIC_FILTER_MIN_OBJECTS
                                 ) -> List[List[Tuple[MaskDetection, MaskDetection]]]:
    """
    Per-sample MAD-based rejection of stationary-object candidates whose
    Loss-1-style residual is an outlier relative to the other objects in the
    same sample. Runs under no_grad — never contributes gradient itself, only
    decides which objects Losses 1-4 get to see this step. See Part 9 markdown
    (Finding 5) for the chicken-and-egg caveat this is deliberately gated
    against (ENABLE_GEOMETRIC_FILTER default False, warmup epochs).
    """
    base = _StationaryObjectLossBase()
    H, W = pred.shape[-2], pred.shape[-1]
    out: List[List[Tuple[MaskDetection, MaskDetection]]] = []
    for b, pairs_b in enumerate(stationary_pairs):
        if len(pairs_b) < min_objects:
            out.append(pairs_b)
            continue
        depth_curr, depth_next = pred[b, 0], pred_next[b, 0]
        K_b, T_b = K[b], eff_pose[b]
        residuals: List[Optional[float]] = []
        for det_curr, det_next in pairs_b:
            mask_c = base._resize_mask(det_curr.mask, H, W).to(depth_curr.device)
            mask_n = base._resize_mask(det_next.mask, H, W).to(depth_next.device)
            d_curr = base._median_depth_in_mask(depth_curr, mask_c)
            d_next = base._median_depth_in_mask(depth_next, mask_n)
            if d_curr is None or d_next is None:
                residuals.append(None)
                continue
            u_c, v_c = base._scale_centroid(det_curr.centroid[0], det_curr.centroid[1],
                                            det_curr.mask.shape, (H, W))
            P_curr = base._backproject(u_c, v_c, d_curr, K_b)
            P_next_exp = T_b[:3, :3] @ P_curr + T_b[:3, 3]
            residuals.append(float((d_next - P_next_exp[2]).abs().item()))

        valid = [r for r in residuals if r is not None]
        if len(valid) < min_objects:
            out.append(pairs_b)
            continue
        med = float(np.median(valid))
        mad = float(np.median([abs(r - med) for r in valid])) + 1e-6
        kept = [pair for pair, r in zip(pairs_b, residuals)
                if r is not None and abs(r - med) <= mad_threshold * mad]
        out.append(kept)
    return out


print("Part 9 improvements in place:")
print("  Finding 1: tracker is configurable (TRACKER_TYPE=%r); no held-out tracking evaluation exists, so no tracker is claimed optimal" % TRACKER_TYPE)
print("  Finding 2: Loss 1 & Loss 3 are area-weighted (sqrt(pixel_count))  [inside the classes]")
print(f"  Finding 3: masks eroded by MASK_ERODE_PX={MASK_ERODE_PX}px before all depth statistics  [inside the classes]")
print(f"  Finding 4: Loss 2 dispersion metric = {UNIFORMITY_METRIC!r}  [inside the class]")
print(f"  Finding 5: geometric_consistency_filter() ready, ENABLE_GEOMETRIC_FILTER={ENABLE_GEOMETRIC_FILTER} (off by default)")


Part 9 improvements in place:
  Finding 1: tracker is configurable (TRACKER_TYPE='hungarian'); no held-out tracking evaluation exists, so no tracker is claimed optimal
  Finding 2: Loss 1 & Loss 3 are area-weighted (sqrt(pixel_count))  [inside the classes]
  Finding 3: masks eroded by MASK_ERODE_PX=2px before all depth statistics  [inside the classes]
  Finding 4: Loss 2 dispersion metric = 'mad'  [inside the class]
  Finding 5: geometric_consistency_filter() ready, ENABLE_GEOMETRIC_FILTER=False (off by default)


## 9b · Loss specification (single source of truth) + unified dispatcher

`LOSS_SPECS` declares, per loss, which inputs the trainer must build: PoseNet poses, OXTS poses, the second DA3 forward on `I_{t+1}`, SAM3 tracking, and whether the loss legitimately reads sparse LiDAR (`uses_gt_depth`). The table is asserted equal to `LOSS_REGISTRY`, and every `NEED_*` set is *derived* from it — the drift between three hand-maintained sets is what silently disabled the v5 `gt_*` losses.


In [13]:
# =============================================================================
# LOSS_SPECS — the SINGLE source of truth for what each loss needs
# =============================================================================
# v5 kept three independent sets (NEED_POSE_LOSSES, NEED_NEXT_DEPTH_LOSSES,
# NEED_GT_POSE_LOSSES) that silently drifted apart: the gt_* losses were in none of
# them, so no pose was built and every gt_* loss fell back to GT-depth L1.
# v6 derives ALL routing from this table and validates it against the registry.
@dataclass(frozen=True)
class LossSpec:
    needs_pred_pose: bool = False     # build PoseNet poses  T_{t->src}
    needs_gt_pose:   bool = False     # use OXTS poses (needs OXTS dataset; asserted per batch)
    needs_pred_next: bool = False     # second DA3 forward on I_{t+1}
    needs_tracking:  bool = False     # SAM3 masks + tracker (CPU)
    uses_gt_depth:   bool = False     # loss legitimately reads sparse LiDAR (supervised / hybrid)
    needs_dyn_mask:  bool = False     # temporal/geometric depth-consistency: exclude potentially-moving objects
    category:        str  = "ssl"     # ssl | regulariser | stationary_ssl | hybrid | supervised_control | oracle_pose

_P  = dict(needs_pred_pose=True)
_PN = dict(needs_pred_pose=True, needs_pred_next=True, needs_dyn_mask=True)
_G  = dict(needs_gt_pose=True)
_GN = dict(needs_gt_pose=True, needs_pred_next=True, needs_dyn_mask=True)
_TR  = dict(needs_pred_pose=True, needs_pred_next=True, needs_tracking=True, category="stationary_ssl")
_TRD = dict(_TR, needs_dyn_mask=True)      # only combined_stationary has a temporal term that consumes the dynamic mask

LOSS_SPECS: Dict[str, LossSpec] = {
    # self-supervised, PoseNet pose
    "photo_l1":                  LossSpec(**_P),
    "photo_ssim":                LossSpec(**_P),
    "photo_ssim_l1":             LossSpec(**_P),
    "min_reproj_photo":          LossSpec(**_P),
    "multi_scale_photo":         LossSpec(**_P),
    "geometric_consistency":     LossSpec(**_PN),
    "feature_metric":            LossSpec(**_P),
    "rigid_flow_consistency":    LossSpec(**_P),
    "temporal_warp_consistency": LossSpec(**_PN),
    "combined_monodepth2":       LossSpec(**_P),
    "combined_recommended_ssl":  LossSpec(**_PN),
    # regulariser / negative control
    "edge_aware_smooth":         LossSpec(category="regulariser"),
    "naive_temporal_delta":      LossSpec(needs_pred_next=True),
    # hybrid + supervised control
    "combined_recommended":      LossSpec(**_PN, uses_gt_depth=True, category="hybrid"),
    "lidar_anchor_only":         LossSpec(uses_gt_depth=True, category="supervised_control"),
    # oracle pose
    "gt_photo_reproj":              LossSpec(**_G,  category="oracle_pose"),
    "gt_rigid_flow_consistency":    LossSpec(**_G,  category="oracle_pose"),
    "gt_temporal_warp_consistency": LossSpec(**_GN, category="oracle_pose"),
    # stationary-object (Part 5) — PoseNet pose + tracking
    "stationary_depth_motion":   LossSpec(**_TR),
    "stationary_uniform_motion": LossSpec(**_TR),
    "full_3d_ego_motion":        LossSpec(**_TR),
    "mask_chamfer":              LossSpec(**_TR),
    "combined_stationary":       LossSpec(**_TRD),
}
# rigid_flow / gt_rigid_flow use I_{t+1} only through the source list; they need the +1 pose.
assert set(LOSS_SPECS) == set(LOSS_REGISTRY), (
    f"LOSS_SPECS and LOSS_REGISTRY are out of sync: {set(LOSS_SPECS) ^ set(LOSS_REGISTRY)}")
for _n, _s in LOSS_SPECS.items():
    assert not (_s.needs_pred_pose and _s.needs_gt_pose), f"{_n}: pred-pose and GT-pose are separate experiments"

# Back-compat views (derived — never edit by hand)
NEED_POSE_LOSSES       = {n for n, s in LOSS_SPECS.items() if s.needs_pred_pose}
NEED_GT_POSE_LOSSES    = {n for n, s in LOSS_SPECS.items() if s.needs_gt_pose}
NEED_NEXT_DEPTH_LOSSES = {n for n, s in LOSS_SPECS.items() if s.needs_pred_next}
NEED_TRACKING_LOSSES   = {n for n, s in LOSS_SPECS.items() if s.needs_tracking}
NEED_DYN_MASK_LOSSES   = {n for n, s in LOSS_SPECS.items() if s.needs_dyn_mask}

# Named groupings used by the ablation groups
SSL_LOSS_NAMES    = [n for n, s in LOSS_SPECS.items() if s.category in ("ssl", "regulariser")]
STATIONARY_LOSS_NAMES = [n for n, s in LOSS_SPECS.items() if s.category == "stationary_ssl"]
ORACLE_LOSS_NAMES = [n for n, s in LOSS_SPECS.items() if s.category == "oracle_pose"]
CONTROL_LOSS_NAMES = [n for n, s in LOSS_SPECS.items() if s.category in ("hybrid", "supervised_control")]

def spec_for(loss_name: str, override: Optional[LossSpec] = None) -> LossSpec:
    return override if override is not None else LOSS_SPECS[loss_name]

# =============================================================================
# Unified dispatcher — every loss gets the same keyword context
# =============================================================================
def _unified_forward(criterion, pred, gt=None, mask=None, **ctx):
    """
    Calls `criterion(pred, gt, mask, **ctx)` with autocast DISABLED and fp32 inputs.
    Each loss picks the keys it needs from ctx and ignores the rest (**kw).  Missing
    required inputs raise SSLInputError inside the loss — there is no fallback path.
    Stationary losses use the aliases ctx['pose'] (=pose_next), ctx['gt_pose'] (=gt_pose_next)
    and ctx['rgb_next'].
    """
    with torch.autocast(device_type=pred.device.type, enabled=False):
        out = criterion(pred.float(), gt, mask, **ctx)
    if not isinstance(out, tuple):
        out = (out, {"total": out})
    return out


print(f"LOSS_SPECS validated: {len(LOSS_SPECS)} losses "
      f"({len(SSL_LOSS_NAMES)} SSL/regulariser, {len(STATIONARY_LOSS_NAMES)} stationary-object, "
      f"{len(ORACLE_LOSS_NAMES)} oracle-pose, {len(CONTROL_LOSS_NAMES)} hybrid/supervised controls)")
print(f"  {'loss':32s} {'pred_pose':>9s} {'gt_pose':>7s} {'pred_next':>9s} {'track':>5s} {'dyn':>5s} {'uses_GT_depth':>13s}  category")
for _n, _s in LOSS_SPECS.items():
    print(f"  {_n:32s} {str(_s.needs_pred_pose):>9s} {str(_s.needs_gt_pose):>7s} {str(_s.needs_pred_next):>9s} "
          f"{str(_s.needs_tracking):>5s} {str(_s.needs_dyn_mask):>5s} {str(_s.uses_gt_depth):>13s}  {_s.category}")


LOSS_SPECS validated: 23 losses (13 SSL/regulariser, 5 stationary-object, 3 oracle-pose, 2 hybrid/supervised controls)
  loss                             pred_pose gt_pose pred_next track   dyn uses_GT_depth  category
  photo_l1                              True   False     False False False         False  ssl
  photo_ssim                            True   False     False False False         False  ssl
  photo_ssim_l1                         True   False     False False False         False  ssl
  min_reproj_photo                      True   False     False False False         False  ssl
  multi_scale_photo                     True   False     False False False         False  ssl
  geometric_consistency                 True   False      True False  True         False  ssl
  feature_metric                        True   False     False False False         False  ssl
  rigid_flow_consistency                True   False     False False False         False  ssl
  temporal_warp_consistency   

## 10 · Metrics (4 scale-alignment strategies)


In [14]:
def eigen_crop_mask(shape: Tuple[int, int]) -> np.ndarray:
    h, w = shape[:2]
    mask = np.zeros((h, w), dtype=bool)
    top = 44; bot = min(top + 352, h)
    mask[top:bot, 0:w] = True
    return mask


def compute_depth_errors(gt: np.ndarray, pred: np.ndarray,
                         mask: np.ndarray) -> Dict[str, float]:
    gt_v   = gt[mask].astype(np.float64)
    pred_v = pred[mask].astype(np.float64)
    pred_v = np.maximum(pred_v, 1e-8)
    thresh   = np.maximum(gt_v / pred_v, pred_v / gt_v)
    rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2)))
    mae      = float(np.mean(np.abs(gt_v - pred_v)))
    abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v))
    sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / gt_v))
    rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2)))
    log10    = float(np.mean(np.abs(np.log10(gt_v) - np.log10(pred_v))))
    delta1   = float(np.mean(thresh < 1.25))
    delta2   = float(np.mean(thresh < 1.25 ** 2))
    delta3   = float(np.mean(thresh < 1.25 ** 3))
    mean_err = float(np.mean(gt_v - pred_v))
    n_pts    = int(mask.sum())
    return dict(n_pts=n_pts, rmse=rmse, mae=mae, abs_rel=abs_rel,
                sq_rel=sq_rel, rmse_log=rmse_log, log10=log10,
                delta1=delta1, delta2=delta2, delta3=delta3,
                mean_error=mean_err,
                lidar_mean=float(np.mean(gt_v)), pred_mean=float(np.mean(pred_v)))


def compute_all_scales(gt_depth: np.ndarray,
                       pred_raw: np.ndarray,
                       mask: np.ndarray) -> Dict[str, Dict]:
    # Compute metrics under 4 scale-alignment strategies.
    gt_v   = gt_depth[mask]
    pred_c = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
    results = {}

    # Unscaled
    m0 = compute_depth_errors(gt_depth, pred_c, mask)
    results["unscaled"] = {"scale": 1.0, "shift": 0.0, **m0}

    # Median scale
    s_med = float(np.median(gt_v) / max(np.median(pred_c[mask]), EPS))
    pred_med = np.clip(pred_c * s_med, MIN_DEPTH, MAX_DEPTH)
    m1 = compute_depth_errors(gt_depth, pred_med, mask)
    results["median"] = {"scale": s_med, "shift": 0.0, **m1}

    # Mean scale
    s_mean = float(np.mean(gt_v) / max(np.mean(pred_c[mask]), EPS))
    pred_mean = np.clip(pred_c * s_mean, MIN_DEPTH, MAX_DEPTH)
    m2 = compute_depth_errors(gt_depth, pred_mean, mask)
    results["mean"] = {"scale": s_mean, "shift": 0.0, **m2}

    # Affine (OLS)
    pf = pred_c[mask].ravel()
    A  = np.vstack([pf, np.ones_like(pf)]).T
    params, _, _, _ = np.linalg.lstsq(A, gt_v.ravel(), rcond=None)
    s_aff = float(params[0]); sh_aff = float(params[1])
    pred_aff = np.clip(s_aff * pred_c + sh_aff, MIN_DEPTH, MAX_DEPTH)
    m3 = compute_depth_errors(gt_depth, pred_aff, mask)
    results["affine"] = {"scale": s_aff, "shift": sh_aff, **m3}

    return results


def true_rmse(df_sub: pd.DataFrame, rmse_col: str = "rmse") -> float:
    valid = df_sub[[rmse_col, "n_pts"]].dropna()
    if valid.empty or valid["n_pts"].sum() == 0:
        return float("nan")
    total_sse = (valid[rmse_col] ** 2 * valid["n_pts"]).sum()
    total_pts = valid["n_pts"].sum()
    return float(np.sqrt(total_sse / total_pts))


def aggregate_results(frame_rows: List[Dict], label: str) -> Dict[str, float]:
    df = pd.DataFrame(frame_rows)
    if df.empty:
        return {}
    row: Dict[str, float] = {"config": label}
    scale_variants = ["unscaled", "median", "mean", "affine"]
    for sv in scale_variants:
        for col in ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log",
                    "log10", "delta1", "delta2", "delta3", "mean_error"]:
            full_col = f"{sv}_{col}"
            if full_col not in df.columns:
                continue
            if col == "rmse":
                n_col = f"{sv}_n_pts"
                if n_col in df.columns:
                    tmp = df[[full_col, n_col]].rename(
                        columns={full_col: "rmse", n_col: "n_pts"})
                    row[full_col] = true_rmse(tmp)
                else:
                    row[full_col] = float(df[full_col].mean())
            else:
                row[full_col] = float(df[full_col].dropna().mean())
    return row


def compute_torch_metrics(pred, gt, mask) -> Dict[str, float]:
    pred_c = pred.clamp(EPS, MAX_DEPTH)
    mask_b = mask.bool()
    if not mask_b.any():
        return {k: float("nan") for k in
                ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}
    p = pred_c[mask_b].float()
    g = gt[mask_b].float().clamp(EPS)
    thresh = torch.maximum(g / p, p / g)
    return {
        "abs_rel":  float(((g - p).abs() / g).mean()),
        "sq_rel":   float(((g - p)**2 / g).mean()),
        "rmse":     float(torch.sqrt(((g - p)**2).mean())),
        "rmse_log": float(torch.sqrt((_safe_log(p) - _safe_log(g)).pow(2).mean())),
        "d1":       float((thresh < 1.25).float().mean()),
        "d2":       float((thresh < 1.25**2).float().mean()),
        "d3":       float((thresh < 1.25**3).float().mean()),
    }

print("Metric helpers ready.")


Metric helpers ready.


## 11 · Training & Validation Loops


In [15]:
def build_optimizer(param_groups, optim_mode: str = "adamw",
                    lr: float = LEARNING_RATE) -> torch.optim.Optimizer:
    # `param_groups` is a list of {"params": [...], "lr": ...} dicts (v6: depth group + pose group).
    if optim_mode == "adam":
        return torch.optim.Adam(param_groups, lr=lr)
    if optim_mode == "sgd":
        return torch.optim.SGD(param_groups, lr=lr, momentum=0.9,
                               weight_decay=WEIGHT_DECAY, nesterov=True)
    return torch.optim.AdamW(param_groups, lr=lr, weight_decay=WEIGHT_DECAY)


def updates_per_epoch_for(n_batches: int) -> int:
    # v6 FIX: the scheduler is stepped once per OPTIMIZER UPDATE, and an update happens
    # every ACCUM_STEPS batches (plus a final partial one).  v5 sized the schedule with
    # len(train_loader), making warm-up/cosine ~ACCUM_STEPS x too long.
    return max(1, math.ceil(n_batches / ACCUM_STEPS))


def build_scheduler(optimizer, updates_per_epoch: int,
                    sched_mode: str = "cosine_warmup",
                    lr: float = LEARNING_RATE):
    from torch.optim.lr_scheduler import LambdaLR, OneCycleLR, StepLR
    if sched_mode == "onecycle":
        return OneCycleLR(optimizer, max_lr=[g["lr"] for g in optimizer.param_groups],
                          steps_per_epoch=updates_per_epoch, epochs=ABLATION_EPOCHS, pct_start=0.1)
    if sched_mode == "step":
        return StepLR(optimizer, step_size=max(1, ABLATION_EPOCHS // 2), gamma=0.5)
    warm  = max(1, WARMUP_EPOCHS * updates_per_epoch)
    total = max(warm + 1, ABLATION_EPOCHS * updates_per_epoch)

    def factor(s: int) -> float:                       # multiplies EVERY group's base lr
        if s < warm:
            return (s + 1) / warm
        prog = min(1.0, (s - warm) / max(total - warm, 1))
        return 0.01 + 0.99 * 0.5 * (1.0 + math.cos(math.pi * prog))
    return LambdaLR(optimizer, lr_lambda=factor)


def _grad_norm(params) -> Optional[float]:
    # None = none of the parameters received a gradient; nan/inf = non-finite gradient
    sq = None
    for p in params:
        if p.grad is not None:
            g = p.grad.detach().float().pow(2).sum()
            sq = g if sq is None else sq + g
    return None if sq is None else float(sq.sqrt())


def _clip(params, max_norm):
    params = [p for p in params if p.grad is not None]
    if params and max_norm > 0:
        nn.utils.clip_grad_norm_(params, max_norm)


def _to(x, device):
    return x.to(device, non_blocking=True)


# ─────────────────────────────────────────────────────────────────────────────
# One forward pass shared by train / validate / warm-up / sanity checks
# ─────────────────────────────────────────────────────────────────────────────
def prepare_step(batch, model, pose_net, spec: "LossSpec", device,
                 epoch: int = 0, train_depth: bool = True, stationary_pairs=None, dyn_mask=None):
    """
    Runs DA3 (+ PoseNet / GT-pose / pred_next as declared by `spec`) and returns
        pred [B,1,H,W] fp32, gt, mask (evaluation targets), loss_kwargs (dict), loss_gt, loss_mask
    `loss_gt / loss_mask` are None unless spec.uses_gt_depth -> a pure-SSL loss cannot
    read LiDAR even by accident.  `gt` / `mask` themselves are None for load_gt=False (SSL-training) loaders.
    """
    rgb   = _to(batch["rgb"], device);      rgb_r = _to(batch["rgb_raw"], device)
    K     = _to(batch["K"], device);        has_K = _to(batch["has_K"], device)
    gt    = _to(batch["depth_gt"], device)   if "depth_gt"   in batch else None   # absent when load_gt=False
    mask  = _to(batch["valid_mask"], device) if "valid_mask" in batch else None
    if spec.uses_gt_depth and (gt is None or mask is None):
        raise SSLInputError("this loss uses sparse GT depth but the loader was built with load_gt=False")

    pref     = [_OFFSET_PREFIX[o] for o in SOURCE_OFFSETS]
    src_norm = [_to(batch[f"{p}_rgb"], device) for p in pref]
    src_raw  = [_to(batch[f"{p}_rgb_raw"], device) for p in pref]
    ni       = SOURCE_OFFSETS.index(+1)
    K_next, has_K_next = _to(batch["next_K"], device), _to(batch["next_has_K"], device)

    use_amp = MIXED_PREC and torch.cuda.is_available()
    with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
        with (nullcontext() if train_depth else torch.no_grad()):
            pred = forward_batch(model, rgb, K, has_K)
            pred_next = (forward_batch(model, src_norm[ni], K_next, has_K_next)
                         if spec.needs_pred_next else None)
        poses = None
        if spec.needs_pred_pose:
            if pose_net is None:
                raise SSLInputError("spec.needs_pred_pose=True but no PoseNet was built")
            poses = [pose_to_se3(pose_net(rgb, s)) for s in src_norm]
    pred = pred.float()
    pred_next = pred_next.float() if pred_next is not None else None

    gt_poses = None
    if spec.needs_gt_pose:
        if "has_gt_pose" not in batch:
            raise SSLInputError("this loss needs OXTS GT pose: use loaders_for(spec) which builds the OXTS loaders")
        assert bool(batch["has_gt_pose"].all()), "batch contains samples WITHOUT GT pose — refusing to continue"
        gt_poses = [_to(batch[f"gt_pose_{p}"], device) for p in pref]

    pose_next    = poses[ni] if poses is not None else None
    gt_pose_next = gt_poses[ni] if gt_poses is not None else None

    if (ENABLE_GEOMETRIC_FILTER and stationary_pairs is not None and pred_next is not None
            and epoch > GEOMETRIC_FILTER_WARMUP_EPOCHS and pose_next is not None):
        stationary_pairs = geometric_consistency_filter(stationary_pairs, pred.detach(),
                                                        pred_next.detach(), K, pose_next.detach())
    ctx = dict(rgb=rgb_r, rgb_srcs=src_raw, rgb_next=src_raw[ni], K=K, has_K=has_K,
               poses=poses, gt_poses=gt_poses, pred_next=pred_next,
               pose_next=pose_next, gt_pose_next=gt_pose_next,
               pose=pose_next, gt_pose=gt_pose_next,            # aliases used by the stationary losses
               stationary_pairs=stationary_pairs,
               dyn_mask=(_to(dyn_mask, device) if (dyn_mask is not None and USE_DYNAMIC_MASK) else None))
    return pred, gt, mask, ctx, (gt if spec.uses_gt_depth else None), (mask if spec.uses_gt_depth else None)


def _iter_batches(loader, needs_tracking: bool, needs_dyn: bool = False,
                  max_batches: Optional[int] = None):
    """Yields (index, batch, stationary_pairs, dyn_mask).  SAM3-based work (tracking pairs / dynamic masks) for
    batch i+1 runs in worker threads while the GPU processes batch i.  (v5 submitted the future ONCE and then
    re-read the same result for every batch, so every step used the first batch's masks.)"""
    it = iter(loader)
    want_dyn = bool(needs_dyn and USE_DYNAMIC_MASK)
    want_trk = bool(needs_tracking and ENABLE_TRACKING)

    def _next():
        try:
            b = next(it)
        except StopIteration:
            return None
        H, W = b["rgb"].shape[-2:]
        f_trk = executor.submit(get_batch_stationary_pairs, b) if want_trk else None
        f_dyn = executor.submit(get_batch_dynamic_masks, b, H, W) if want_dyn else None
        return b, f_trk, f_dyn

    cur, n = _next(), 0
    while cur is not None and (max_batches is None or n < max_batches):
        nxt = _next() if (max_batches is None or n + 1 < max_batches) else None
        yield (n, cur[0],
               cur[1].result() if cur[1] is not None else None,
               cur[2].result() if cur[2] is not None else None)
        cur, n = nxt, n + 1


def train_one_epoch(model, pose_net, loader, criterion, optimizer, scaler,
                    scheduler, device, epoch, global_step, spec: "LossSpec",
                    sched_mode: str = "cosine_warmup",
                    gpu_duty_cycle: float = 1.0,
                    train_depth: bool = True,
                    depth_params: Optional[List] = None,
                    pose_params: Optional[List] = None,
                    max_batches: Optional[int] = None) -> Tuple[int, Dict]:
    """
    One epoch.  Returns (global_step, averaged_terms).  The dict also carries health diagnostics
    (keys prefixed `diag_`): depth/pose gradient norms, skipped (non-finite) updates …
    These exist because v5 only looked at validation RMSE and never noticed that no update
    was ever applied.
    """
    depth_params = depth_params if depth_params is not None else [p for p in model.parameters() if p.requires_grad]
    pose_params  = pose_params  if pose_params  is not None else (list(pose_net.parameters()) if pose_net is not None else [])
    if train_depth:
        model.model.train()
    else:
        model.model.eval()
    if pose_net is not None and spec.needs_pred_pose:
        pose_net.train()

    n_batches = len(loader) if max_batches is None else min(len(loader), max_batches)
    running: Dict[str, float] = {}
    diag = dict(n_updates=0, n_skipped=0, n_bad_loss=0, dg_sum=0.0, pg_sum=0.0, n_gn=0)
    n_ok = 0
    valid_in_window = 0                       # finite-loss batches that contributed gradient to the CURRENT window
    optimizer.zero_grad(set_to_none=True)

    for bi, batch, stationary_pairs, dyn_mask in _iter_batches(loader, spec.needs_tracking, spec.needs_dyn_mask, max_batches):
        if bi % 50 == 0:
            log.info("    batch %d/%d", bi, n_batches)
        t0 = time.perf_counter()

        pred, gt, mask, ctx, l_gt, l_mask = prepare_step(
            batch, model, pose_net, spec, device, epoch=epoch,
            train_depth=train_depth, stationary_pairs=stationary_pairs, dyn_mask=dyn_mask)
        total, losses = _unified_forward(criterion, pred, l_gt, l_mask, **ctx)

        # size of THIS accumulation window (the last one can be shorter)
        window_start = (bi // ACCUM_STEPS) * ACCUM_STEPS
        window_size  = min(ACCUM_STEPS, n_batches - window_start)
        if torch.isfinite(total):
            scaler.scale(total / window_size).backward()     # nominal 1/window_size; corrected below if batches were skipped
            valid_in_window += 1
            n_ok += 1
            for k, v in losses.items():
                running[k] = running.get(k, 0.0) + float(v.detach() if torch.is_tensor(v) else v)
        else:
            diag["n_bad_loss"] += 1                       # non-finite loss -> skip this batch entirely

        is_last = (bi + 1 == n_batches)
        if (bi + 1) % ACCUM_STEPS == 0 or is_last:
            has_grads = any(p.grad is not None for p in (depth_params + pose_params))
            if has_grads and USE_GRAD_SCALER:
                scaler.unscale_(optimizer)
            dn = _grad_norm(depth_params) if (train_depth and has_grads) else None
            pn = _grad_norm(pose_params) if has_grads else None
            finite = has_grads and all(math.isfinite(x) for x in (dn, pn) if x is not None)
            if finite and 0 < valid_in_window < window_size:
                # v6.2: some batches of this window had a non-finite loss and contributed nothing; the gradient is
                # the sum of `valid_in_window` terms each divided by `window_size` -> rescale to a true mean.
                # (grads are already unscaled here when the GradScaler is active, so this is scale-safe)
                _corr = window_size / valid_in_window
                for _p in depth_params + pose_params:
                    if _p.grad is not None:
                        _p.grad.mul_(_corr)
                dn = None if dn is None else dn * _corr
                pn = None if pn is None else pn * _corr
            if finite:
                if train_depth: _clip(depth_params, MAX_GRAD_NORM)     # clipped per group: PoseNet gradients
                _clip(pose_params, MAX_GRAD_NORM)                      # can no longer squash the depth update
                scaler.step(optimizer); scaler.update()
                diag["n_updates"] += 1
                diag["dg_sum"] += dn or 0.0
                diag["pg_sum"] += pn or 0.0
                diag["n_gn"] += 1
            else:
                if has_grads and USE_GRAD_SCALER:
                    scaler.step(optimizer); scaler.update()            # scaler skips + lowers its scale
                diag["n_skipped"] += 1
            optimizer.zero_grad(set_to_none=True)
            valid_in_window = 0
            if scheduler is not None and sched_mode in ("cosine_warmup", "onecycle"):
                scheduler.step()
            global_step += 1

        if gpu_duty_cycle < 1.0:
            torch.cuda.synchronize(device)
            t_sleep = (time.perf_counter() - t0) * (1.0 / gpu_duty_cycle - 1.0)
            if t_sleep > 0:
                time.sleep(t_sleep)

    if scheduler is not None and sched_mode == "step":
        scheduler.step()

    avg = {k: v / max(n_ok, 1) for k, v in running.items()}
    tot_upd = diag["n_updates"] + diag["n_skipped"]
    avg.update({
        "diag_updates_applied":  diag["n_updates"],
        "diag_updates_skipped":  diag["n_skipped"],
        "diag_skip_frac":        diag["n_skipped"] / max(tot_upd, 1),
        "diag_bad_loss_batches": diag["n_bad_loss"],
        "diag_depth_grad_norm":  diag["dg_sum"] / max(diag["n_gn"], 1) if train_depth else float("nan"),
        "diag_pose_grad_norm":   diag["pg_sum"] / max(diag["n_gn"], 1) if pose_params else float("nan"),
    })
    if diag["n_skipped"] or diag["n_bad_loss"]:
        log.warning("    [health] %d/%d optimizer updates SKIPPED (non-finite grads), %d batches with non-finite loss",
                    diag["n_skipped"], tot_upd, diag["n_bad_loss"])
    return global_step, avg


@torch.no_grad()
def validate(model, pose_net, loader, criterion, device, epoch,
             spec: "LossSpec", max_batches: Optional[int] = None) -> Dict[str, float]:
    model.model.eval()
    if pose_net is not None:
        pose_net.eval()
    all_m: List[Dict] = []
    total_loss, n_batches = 0.0, 0
    for bi, batch, stationary_pairs, dyn_mask in _iter_batches(loader, spec.needs_tracking, spec.needs_dyn_mask, max_batches):
        pred, gt, mask, ctx, l_gt, l_mask = prepare_step(
            batch, model, pose_net, spec, device, epoch=epoch,
            train_depth=False, stationary_pairs=stationary_pairs, dyn_mask=dyn_mask)
        assert gt is not None and mask is not None, "validation loader must be built with load_gt=True"
        loss, _ = _unified_forward(criterion, pred, l_gt, l_mask, **ctx)
        total_loss += float(loss); n_batches += 1
        all_m.append(compute_torch_metrics(pred, gt, mask))       # evaluation only — GT is used HERE, not in the loss

    def _avg(k):
        return float(np.nanmean([x[k] for x in all_m]))

    agg = {k: _avg(k) for k in ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}
    agg["val_loss"] = total_loss / max(n_batches, 1)
    return agg


print("Training / validation loops ready (v6: spec-driven routing, fp32 losses, gradient diagnostics).")


Training / validation loops ready (v6: spec-driven routing, fp32 losses, gradient diagnostics).


## 12 · Test-Set Evaluation (KITTI Eigen split)


In [16]:
def index_kitti_test(test_root: str) -> pd.DataFrame:
    rows = []
    pattern = os.path.join(test_root, "*", "*_sync",
                            "proj_depth", "groundtruth", "image_02", "*.png")
    gt_paths = glob.glob(pattern)
    if not gt_paths:
        pattern = os.path.join(test_root, "*", "*",
                               "proj_depth", "groundtruth", "image_02", "*.png")
        gt_paths = glob.glob(pattern)
    for gt_path in tqdm(gt_paths, desc="Indexing test split"):
        parts = gt_path.split(os.sep)
        try:
            gt_idx = parts.index("proj_depth")
            drive  = parts[gt_idx - 1]; date = parts[gt_idx - 2]; fname = parts[-1]
        except (ValueError, IndexError):
            continue
        rgb_path = os.path.join(test_root, date, drive, "image_02", "data", fname)
        if not os.path.exists(rgb_path):
            continue
        rows.append(dict(date=date, drive=drive, filename=fname,
                         gt_path=gt_path, rgb_path=rgb_path))
    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Test index: %d frames, %d drives", len(df), df["drive"].nunique())
    return df


def evaluate_model(model: nn.Module, test_df: pd.DataFrame,
                   device: torch.device,
                   config_label: str) -> List[Dict]:
    model.model.eval()
    frame_rows = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df),
                       desc=f"Eval: {config_label}"):
        try:
            image_np = np.array(Image.open(row["rgb_path"]).convert("RGB"))
            gt_depth = _read_kitti_depth(row["gt_path"])
            crop_mask = eigen_crop_mask(gt_depth.shape)
            valid_mask = ((gt_depth > MIN_DEPTH) & (gt_depth < MAX_DEPTH) & crop_mask)
            if not np.any(valid_mask):
                continue
            pred_raw = run_da3_inference(model, image_np, device)
            scale_res = compute_all_scales(gt_depth, pred_raw, valid_mask)
            frame_row = {"config": config_label}
            for sv, res in scale_res.items():
                for k, v in res.items():
                    if k not in ("scale", "shift"):
                        frame_row[f"{sv}_{k}"] = v
            frame_rows.append(frame_row)
        except Exception as e:
            log.error("Error evaluating %s: %s", row["rgb_path"], e)
    return frame_rows

print("Evaluation helpers ready.")


Evaluation helpers ready.


## 13 · HPT-Enabled Ablation Runner


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info("Running on: %s", device)

_base_model = load_da3_model(device)
_base_state = copy.deepcopy(_base_model.state_dict())

def load_da3_model_fast(device):
    _base_model.load_state_dict(copy.deepcopy(_base_state))
    return _base_model


# ─── Temp-checkpoint path helpers ───────────────────────────────────────────
def _ckpt_path(ablation_label: str, hpt_tag: str, trial_hash: str = "") -> str:
    safe = hpt_tag.replace("/", "-").replace(" ", "_")
    suffix = f"__{trial_hash}" if trial_hash else ""
    return os.path.join(TEMP_CKPT_DIR, f"temp_best_{ablation_label}__{safe}{suffix}.pt")


def _stable_hash(obj) -> str:
    import hashlib, json
    return hashlib.md5(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()[:10]


def _trial_hash(hpt_cfg: Dict, loss_name: str, arch_mode: str, aug_mode: str, optim_mode: str,
                sched_mode: str, spec: "LossSpec", criterion: nn.Module, train_loader: DataLoader) -> str:
    """Identity of one trial's experiment: everything that changes what the trial computes.  A checkpoint whose
    hash differs is never resumed (v6 keyed only on label + LR).  `repr(criterion)` includes the loss weights via
    the modules' extra_repr."""
    return _stable_hash({"hpt": hpt_cfg, "loss": loss_name, "arch": arch_mode, "aug": aug_mode,
                         "optim": optim_mode, "sched": sched_mode, "spec": repr(spec),
                         "criterion": repr(criterion), "train_samples": _loader_fingerprint(train_loader),
                         "experiment": EXPERIMENT_HASH})

def _hpt_tag(hpt_cfg: Dict) -> str:
    return "__".join(f"{k}={v}" for k, v in sorted(hpt_cfg.items()))

def _param_fingerprint(params) -> float:
    # float64 sum(|w|): detects "the network never changed" (v5's silent failure)
    return float(sum(p.detach().double().abs().sum().item() for p in params))


def _loader_fingerprint(loader: DataLoader) -> str:
    """Stable short hash of the exact samples a loader iterates (+ dataset class, aug mode, root, offsets)."""
    import hashlib
    ds = loader.dataset
    idx = list(ds.indices) if hasattr(ds, "indices") else None
    base = ds.dataset if hasattr(ds, "dataset") else ds
    paths = [s[0]["rgb_path"] for s in base._samples]
    if idx is not None:
        paths = [paths[i] for i in idx]
    h = hashlib.md5("|".join(paths).encode())
    h.update(f"{type(base).__name__}|{getattr(base, 'aug_mode', '')}|{getattr(base, 'offsets', '')}|{TRAIN_ROOT}".encode())
    return h.hexdigest()[:12]


# ── Pretrained-baseline validation metrics: computed ONCE per validation set (v6 recomputed them per HPT trial) ──
_BASELINE_CACHE: Dict[str, Dict[str, float]] = {}

@torch.no_grad()
def get_baseline_metrics(val_loader: DataLoader, device: torch.device) -> Dict[str, float]:
    import json as _json
    fp = _loader_fingerprint(val_loader)
    if fp in _BASELINE_CACHE:
        return _BASELINE_CACHE[fp]
    path = os.path.join(TEMP_CKPT_DIR, f"baseline_{MODEL_ID.replace('/', '-')}_{fp}.json")
    if os.path.exists(path):
        _BASELINE_CACHE[fp] = _json.load(open(path))
        log.info("[baseline] loaded cached pretrained-baseline metrics (rmse=%.4f)", _BASELINE_CACHE[fp]["rmse"])
        return _BASELINE_CACHE[fp]
    model = load_da3_model_fast(device)
    model.model.eval()
    use_amp = MIXED_PREC and torch.cuda.is_available()
    all_m: List[Dict] = []
    for batch in val_loader:
        rgb, K, has_K = _to(batch["rgb"], device), _to(batch["K"], device), _to(batch["has_K"], device)
        gt, mask = _to(batch["depth_gt"], device), _to(batch["valid_mask"], device)
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=use_amp):
            pred = forward_batch(model, rgb, K, has_K)
        all_m.append(compute_torch_metrics(pred.float(), gt, mask))
    agg = {k: float(np.nanmean([x[k] for x in all_m])) for k in ["abs_rel", "sq_rel", "rmse", "rmse_log", "d1", "d2", "d3"]}
    agg["val_loss"] = float("nan")
    _BASELINE_CACHE[fp] = agg
    _json.dump(agg, open(path, "w"))
    log.info("[baseline] pretrained DA3 on validation set: rmse=%.4f  d1=%.4f  (computed once, cached)", agg["rmse"], agg["d1"])
    return agg


# ══════════════════════════════════════════════════════════════════════════════
# PoseNet warm-up (depth FROZEN, photometric loss only, no GT depth) — computed once, cached
# ══════════════════════════════════════════════════════════════════════════════
_POSE_WARM_CACHE: Dict[str, Dict] = {}

def get_warm_pose_state(train_loader: DataLoader, device: torch.device, tag_extra: str = "") -> Dict:
    """
    v5 started PoseNet at R=I,t=0 where the warp does not depend on depth => no depth gradient.
    Here PoseNet is first trained for POSE_WARMUP_EPOCHS with the (pretrained, metric) DA3 depth
    frozen and the Monodepth2 photometric loss, so joint training starts with a pose that
    already carries real motion.  The result is identical for every HPT trial of the same
    data/offsets, so it is computed once and cached on disk.
    """
    ds = train_loader.dataset
    ds = getattr(ds, "dataset", ds)
    # v6.2 cache key: hash of a canonical dict with EVERY setting that influences the warm-up
    warm_cfg = {"code_version": POSE_WARMUP_CODE_VERSION, "experiment": EXPERIMENT_HASH, "model_id": MODEL_ID,
                "dataset_class": type(ds).__name__, "samples_aug_root_offsets": _loader_fingerprint(train_loader),
                "pose_lr": POSE_LR, "warmup_epochs": POSE_WARMUP_EPOCHS, "warmup_max_batches": POSE_WARMUP_MAX_BATCHES,
                "pose_out_scale": POSE_OUT_SCALE, "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE,
                "accum_steps": ACCUM_STEPS, "max_grad_norm": MAX_GRAD_NORM, "amp_mode": AMP_DTYPE_MODE,
                "use_grad_scaler": USE_GRAD_SCALER, "target_hw": (TARGET_H, TARGET_W),
                "source_offsets": SOURCE_OFFSETS, "tag_extra": tag_extra}
    tag = _stable_hash(warm_cfg)
    if tag in _POSE_WARM_CACHE:
        return _POSE_WARM_CACHE[tag]
    path = os.path.join(TEMP_CKPT_DIR, f"pose_warmup__{tag}.pt")   # tag = hash of warm_cfg
    if os.path.exists(path):
        try:
            _POSE_WARM_CACHE[tag] = torch.load(path, map_location="cpu", weights_only=False)
            log.info("[pose warm-up] loaded cached state %s", os.path.basename(path))
            return _POSE_WARM_CACHE[tag]
        except Exception as e:
            log.warning("[pose warm-up] cache unreadable (%s) — recomputing", e)

    log.info("[pose warm-up] training PoseNet for %d epoch(s), depth frozen (lr=%.1e)", POSE_WARMUP_EPOCHS, POSE_LR)
    model = load_da3_model_fast(device)
    for p in model.parameters():
        p.requires_grad_(False)
    pose_net = PoseNet().to(device)
    pose_params = list(pose_net.parameters())
    opt = torch.optim.AdamW(pose_params, lr=POSE_LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler("cuda" if torch.cuda.is_available() else "cpu", enabled=USE_GRAD_SCALER)
    crit = MinReprojPhotoLoss().to(device)
    wspec = LossSpec(needs_pred_pose=True)
    gstep = 0
    for ep in range(1, POSE_WARMUP_EPOCHS + 1):
        gstep, tl = train_one_epoch(model, pose_net, train_loader, crit, opt, scaler, None, device, ep, gstep,
                                    wspec, train_depth=False, depth_params=[], pose_params=pose_params,
                                    max_batches=(POSE_WARMUP_MAX_BATCHES or None))
        log.info("[pose warm-up] epoch %d  photo=%.4f  pose_grad=%.3e  skipped=%d",
                 ep, tl.get("total", float("nan")), tl["diag_pose_grad_norm"], tl["diag_updates_skipped"])
    state = {k: v.detach().cpu() for k, v in pose_net.state_dict().items()}
    torch.save(state, path)
    _POSE_WARM_CACHE[tag] = state
    del pose_net, opt, scaler, crit
    torch.cuda.empty_cache(); gc.collect()
    return state


# ══════════════════════════════════════════════════════════════════════════════
# Core HPT trial — train + validate + early-stop + checkpoint
# ══════════════════════════════════════════════════════════════════════════════
FAIL_FAST_NO_UPDATES = True      # abort a trial whose first epoch applied zero optimizer updates

def _train_one_hpt_trial(
    ablation_label: str,
    hpt_cfg:        Dict,
    loss_name:      str,
    arch_mode:      str,
    aug_mode:       str,
    optim_mode:     str,
    sched_mode:     str,
    device:         torch.device,
    train_loader:   DataLoader,
    val_loader:     DataLoader,
    custom_criterion: Optional[nn.Module] = None,
    spec:           Optional["LossSpec"] = None,
) -> Tuple[float, str, List[Dict], float]:
    spec = spec_for(loss_name, spec)
    tag  = _hpt_tag(hpt_cfg)
    if custom_criterion is not None:
        criterion = copy.deepcopy(custom_criterion).to(device)
    else:
        criterion = copy.deepcopy(LOSS_REGISTRY[loss_name]).to(device)
    thash = _trial_hash(hpt_cfg, loss_name, arch_mode, aug_mode, optim_mode, sched_mode, spec, criterion, train_loader)
    ckpt = _ckpt_path(ablation_label, tag, thash)
    latest_ckpt = ckpt.replace("temp_best_", "temp_latest_")

    lr = float(hpt_cfg.get("lr", LEARNING_RATE))
    log.info("  [HPT trial] ablation=%s  tag=%s  depth_lr=%.2e  pose_lr=%.2e  spec=%s",
             ablation_label, tag, lr, POSE_LR, spec)

    # PoseNet warm-up FIRST (it borrows the shared DA3 instance, the trial re-loads it afterwards)
    warm_state = None
    if spec.needs_pred_pose and POSE_WARMUP_EPOCHS > 0:
        warm_state = get_warm_pose_state(train_loader, device)

    model = load_da3_model_fast(device)
    _apply_arch(model, arch_mode)

    pose_net = None
    if spec.needs_pred_pose:
        pose_net = PoseNet().to(device)
        if warm_state is not None:
            pose_net.load_state_dict(warm_state)

    depth_params = [p for p in model.parameters() if p.requires_grad]
    pose_params  = list(pose_net.parameters()) if pose_net is not None else []
    groups = [{"params": depth_params, "lr": lr}]
    if pose_params:
        groups.append({"params": pose_params, "lr": POSE_LR})          # separate depth / pose learning rates
    optimizer = build_optimizer(groups, optim_mode, lr)

    upe       = updates_per_epoch_for(len(train_loader))
    scheduler = build_scheduler(optimizer, upe, sched_mode, lr)
    scaler    = GradScaler("cuda" if torch.cuda.is_available() else "cpu", enabled=USE_GRAD_SCALER)

    best_val_rmse = float("inf")
    baseline_rmse = float("nan")
    no_improve    = 0
    global_step   = 0
    start_epoch   = 1
    history: List[Dict] = []

    # ── Resumability ─────────────────────────────────────────────────────
    resumed = False
    if os.path.exists(latest_ckpt):
        log.info("  [RESUME] Resuming from %s", os.path.basename(latest_ckpt))
        try:
            checkpoint = torch.load(latest_ckpt, map_location=device, weights_only=False)
            if checkpoint.get("trial_hash") != thash or checkpoint.get("experiment_hash") != EXPERIMENT_HASH:
                raise RuntimeError("checkpoint was produced by a different experiment configuration "
                                   f"(trial_hash {checkpoint.get('trial_hash')} vs {thash})")
            model.model.load_state_dict(checkpoint["model_state"])
            if pose_net is not None and "pose_state" in checkpoint:
                pose_net.load_state_dict(checkpoint["pose_state"])
            optimizer.load_state_dict(checkpoint["optimizer_state"])
            if scheduler and checkpoint.get("scheduler_state"):
                scheduler.load_state_dict(checkpoint["scheduler_state"])
            scaler.load_state_dict(checkpoint["scaler_state"])
            start_epoch   = checkpoint["epoch"] + 1
            best_val_rmse = checkpoint["best_val_rmse"]
            baseline_rmse = checkpoint.get("baseline_rmse", float("nan"))
            no_improve    = checkpoint["no_improve"]
            global_step   = checkpoint["global_step"]
            history       = checkpoint.get("history", [])
            resumed = True
            log.info("  [RESUME] From epoch %d  best_val_rmse=%.4f", start_epoch, best_val_rmse)
        except Exception as e:
            log.warning("  [RESUME] Failed: %s — starting fresh.", e)
            best_val_rmse = float("inf"); no_improve = 0
            global_step = 0; start_epoch = 1; history = []

    # ── Epoch 0: pretrained baseline (cached per validation set — NOT recomputed per trial) ──
    if not resumed:
        v0 = dict(get_baseline_metrics(val_loader, device))
        baseline_rmse = v0["rmse"]
        log.info("  Ep00 (pretrained DA3 baseline)  val_rmse=%.4f  d1=%.4f", v0["rmse"], v0["d1"])
        history.append({"config": ablation_label, "hpt_tag": tag, "epoch": 0, "lr": lr,
                        "train_loss": float("nan"), **{f"val_{k}": v for k, v in v0.items()}})

    if not math.isfinite(baseline_rmse):
        baseline_rmse = get_baseline_metrics(val_loader, device)["rmse"]
    fp0 = _param_fingerprint(depth_params)

    # ── Training ─────────────────────────────────────────────────────────
    for epoch in range(start_epoch, ABLATION_EPOCHS + 1):
        t0 = time.time()
        global_step, train_losses = train_one_epoch(
            model, pose_net, train_loader, criterion, optimizer,
            scaler, scheduler, device, epoch, global_step, spec,
            sched_mode=sched_mode, gpu_duty_cycle=gpu_duty_cycle,
            depth_params=depth_params, pose_params=pose_params)
        val_m = validate(model, pose_net, val_loader, criterion, device, epoch, spec)
        elapsed = time.time() - t0

        val_rmse = val_m["rmse"]
        improved = val_rmse < best_val_rmse
        log.info("  Ep%02d/%02d  lr=%.2e  train=%.4f  val_loss=%.4f  val_rmse=%.4f (base %.4f)  d1=%.4f  "
                 "|g_depth|=%.2e |g_pose|=%.2e skip=%.0f%%  (%.0fs)%s",
                 epoch, ABLATION_EPOCHS, lr, train_losses.get("total", float("nan")),
                 val_m["val_loss"], val_rmse, baseline_rmse, val_m["d1"],
                 train_losses["diag_depth_grad_norm"], train_losses["diag_pose_grad_norm"],
                 100 * train_losses["diag_skip_frac"], elapsed, "  IMPROVED" if improved else "")

        rec = {"config": ablation_label, "hpt_tag": tag, "epoch": epoch, "lr": lr,
               "train_loss": train_losses.get("total", float("nan")),
               **{f"val_{k}": v for k, v in val_m.items()},
               **{f"train_{k}": v for k, v in train_losses.items()
                   if k.startswith("diag_") or k in ("n_objects", "n_samples_used")}}  # v6.2.1: surface Group 1c health metrics (Bug 5); no other loss produces these keys, so this is a no-op for every other trial (pandas fills the column with NaN for them)
        history.append(rec)

        if epoch == start_epoch and FAIL_FAST_NO_UPDATES:
            if train_losses["diag_updates_applied"] == 0:
                raise RuntimeError(
                    f"[{ablation_label}/{tag}] epoch {epoch}: ZERO optimizer updates were applied "
                    f"({train_losses['diag_updates_skipped']} skipped, "
                    f"{train_losses['diag_bad_loss_batches']} non-finite losses). "
                    f"The pipeline is not training — refusing to burn GPU time.")
            fp1 = _param_fingerprint(depth_params)
            log.info("  [health] depth-parameter fingerprint change after epoch %d: %.3e (relative %.2e)",
                     epoch, fp1 - fp0, abs(fp1 - fp0) / max(fp0, 1e-12))

        def _blob(with_opt: bool) -> Dict:
            b = {"epoch": epoch, "model_state": model.model.state_dict(), "val_rmse": val_rmse,
                 "hpt_cfg": hpt_cfg, "config_label": ablation_label,
                 "trial_hash": thash, "experiment_hash": EXPERIMENT_HASH,
                 "loss_name": loss_name, "arch_mode": arch_mode}
            if pose_net is not None:
                b["pose_state"] = pose_net.state_dict()
            if with_opt:
                b.update({"optimizer_state": optimizer.state_dict(),
                          "scheduler_state": scheduler.state_dict() if scheduler else None,
                          "scaler_state": scaler.state_dict(), "best_val_rmse": best_val_rmse,
                          "baseline_rmse": baseline_rmse, "no_improve": no_improve,
                          "global_step": global_step, "history": history})
            return b

        if improved:
            best_val_rmse = val_rmse
            no_improve = 0
            torch.save(_blob(False), ckpt)
        else:
            no_improve += 1
            if no_improve >= EARLY_STOP_PATIENCE:
                log.info("  Early stop @ epoch %d (no improvement for %d ep)", epoch, EARLY_STOP_PATIENCE)
                torch.save(_blob(True), latest_ckpt)
                break

        torch.save(_blob(True), latest_ckpt)
        if SAVE_EVERY_N_EPOCHS > 0 and epoch % SAVE_EVERY_N_EPOCHS == 0:
            torch.save(_blob(False), ckpt.replace("temp_best_", f"temp_ep{epoch:03d}_"))

    log.info("  Done.  best_val_rmse=%.4f  baseline=%.4f  (ckpt: %s)",
             best_val_rmse, baseline_rmse, os.path.basename(ckpt))
    if best_val_rmse >= baseline_rmse - 1e-4:
        log.warning("  [health] this trial did NOT improve on the pretrained baseline (%.4f vs %.4f)",
                    best_val_rmse, baseline_rmse)

    del model, criterion, optimizer, scaler
    if pose_net is not None:
        del pose_net
    torch.cuda.empty_cache()
    gc.collect()
    return best_val_rmse, ckpt, history, baseline_rmse


# ══════════════════════════════════════════════════════════════════════════════
# HPT sweep orchestrator for one ablation label
# ══════════════════════════════════════════════════════════════════════════════
def run_ablation_with_hpt(
    ablation_label:   str,
    hpt_grid:         List[Dict],
    loss_name:        str            = "combined_recommended",
    arch_mode:        str            = "decoder_only",
    aug_mode:         str            = "base",
    optim_mode:       str            = "adamw",
    sched_mode:       str            = "cosine_warmup",
    device:           torch.device   = None,
    train_loader:     DataLoader     = None,
    val_loader:       DataLoader     = None,
    test_df:          pd.DataFrame   = None,
    custom_criterion: Optional[nn.Module] = None,
    spec:             Optional["LossSpec"] = None,
) -> Dict[str, float]:
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    spec = spec_for(loss_name, spec)

    log.info("")
    log.info("=" * 70)
    log.info("HPT SWEEP  ablation=%s", ablation_label)
    log.info("  loss=%s [%s]  arch=%s  aug=%s  optim=%s", loss_name, spec.category, arch_mode, aug_mode, optim_mode)
    log.info("  routing: pred_pose=%s gt_pose=%s pred_next=%s tracking=%s uses_GT_depth=%s",
             spec.needs_pred_pose, spec.needs_gt_pose, spec.needs_pred_next, spec.needs_tracking, spec.uses_gt_depth)
    log.info("  %d trial(s) in grid x up to %d epochs each (patience=%d)",
             len(hpt_grid), ABLATION_EPOCHS, EARLY_STOP_PATIENCE)
    log.info("=" * 70)

    # Phase 1: train every HPT trial
    trial_results: List[Dict] = []
    all_history:   List[Dict] = []
    for hpt_cfg in hpt_grid:
        best_rmse, ckpt, history, base_rmse = _train_one_hpt_trial(
            ablation_label=ablation_label, hpt_cfg=hpt_cfg,
            loss_name=loss_name, arch_mode=arch_mode, aug_mode=aug_mode,
            optim_mode=optim_mode, sched_mode=sched_mode, device=device,
            train_loader=train_loader, val_loader=val_loader,
            custom_criterion=custom_criterion, spec=spec,
        )
        trial_results.append(dict(tag=_hpt_tag(hpt_cfg), hpt_cfg=hpt_cfg,
                                  best_val_rmse=best_rmse, baseline_val_rmse=base_rmse, ckpt=ckpt))
        all_history.extend(history)

    # Phase 2: model selection
    winner = min(trial_results, key=lambda x: x["best_val_rmse"])
    log.info("")
    log.info("HPT results for [%s]:", ablation_label)
    for t in sorted(trial_results, key=lambda x: x["best_val_rmse"]):
        marker = " <- WINNER" if t["tag"] == winner["tag"] else ""
        log.info("  %-30s  best_val_rmse=%.4f%s",
                 t["tag"], t["best_val_rmse"], marker)

    # Phase 3: test eval (ONCE)
    log.info("Loading winner ckpt -> test evaluation (ONCE) ...")
    model = load_da3_model_fast(device)
    _apply_arch(model, arch_mode)
    ckpt_data = torch.load(winner["ckpt"], map_location=device, weights_only=False)
    model.model.load_state_dict(ckpt_data["model_state"])
    log.info("Winner ckpt: epoch=%d  val_rmse=%.4f",
             ckpt_data["epoch"], ckpt_data["val_rmse"])

    frame_rows = evaluate_model(model, test_df, device, ablation_label)
    summary    = aggregate_results(frame_rows, ablation_label)
    summary.update({
        "loss_fn":         loss_name,
        "experiment_hash": EXPERIMENT_HASH,
        "arch":            arch_mode,
        "aug":             aug_mode,
        "optim":           optim_mode,
        "sched":           sched_mode,
        "winner_hpt_tag":  winner["tag"],
        "winner_val_rmse": winner["best_val_rmse"],
        "baseline_val_rmse": winner.get("baseline_val_rmse", float("nan")),
        "loss_category":   (spec.category if spec is not None else "n/a"),
        "winner_epoch":    int(ckpt_data["epoch"]),
        "n_hpt_trials":    len(hpt_grid),
        "n_frames":        len(frame_rows),
    })

    # Phase 4: persist
    header = not os.path.exists(SUMMARY_CSV)
    pd.DataFrame([summary]).to_csv(SUMMARY_CSV, mode="a", header=header, index=False)
    hist_df   = pd.DataFrame(all_history)
    hist_path = os.path.join(RESULTS_DIR, f"history_{ablation_label}.csv")
    hist_df.to_csv(hist_path, index=False)
    hpt_rows = [{
        "ablation":      ablation_label,
        "tag":           t["tag"],
        "best_val_rmse": t["best_val_rmse"],
        "baseline_val_rmse": t.get("baseline_val_rmse", float("nan")),
        "is_winner":     t["tag"] == winner["tag"],
        **t["hpt_cfg"],
    } for t in trial_results]
    hpt_path = os.path.join(RESULTS_DIR, f"hpt_sweep_{ablation_label}.csv")
    pd.DataFrame(hpt_rows).to_csv(hpt_path, index=False)
    log.info("  CSVs written: summary, history, hpt_sweep")

    # Phase 5: cleanup
    for t in trial_results:
        for _pe in glob.glob(os.path.join(TEMP_CKPT_DIR, f"temp_ep*_{ablation_label}__*.pt")):
            try: os.remove(_pe)
            except OSError: pass
        for prefix in ["temp_best_", "temp_latest_"]:
            path = t["ckpt"].replace("temp_best_", prefix)
            try:
                os.remove(path)
            except FileNotFoundError:
                pass

    del model
    torch.cuda.empty_cache()
    gc.collect()
    log.info("  -- ablation=%s complete --", ablation_label)
    return summary


print("HPT-enabled ablation runner ready.")
print("  _train_one_hpt_trial   -> train + early-stop + temp ckpt")
print("  run_ablation_with_hpt  -> full sweep + selection + test ONCE + cleanup")


22:39:35  INFO      Running on: cuda
22:39:36  INFO      Loading depth-anything/DA3METRIC-LARGE ...


[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


22:39:36  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
22:39:36  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/depth-anything/DA3METRIC-LARGE/4010e39f3634a45bc60553321fb49fb760bd594e/config.json "HTTP/1.1 200 OK"


[INFO ] using MLP layer as FFN


22:39:39  INFO      HTTP Request: HEAD https://huggingface.co/depth-anything/DA3METRIC-LARGE/resolve/main/model.safetensors "HTTP/1.1 302 Found"


HPT-enabled ablation runner ready.
  _train_one_hpt_trial   -> train + early-stop + temp ckpt
  run_ablation_with_hpt  -> full sweep + selection + test ONCE + cleanup


## 13a · Data loaders (train / val / test)

* **Canonical sample set** — with `CANONICAL_SAMPLE_SET=True` every loader (SSL and oracle-pose) uses exactly the samples that have complete OXTS pose + calibration and the *same* train/val index split (asserted), so oracle-vs-predicted-pose comparisons use the same images ("Design B": *every* group then runs on this OXTS-complete subset).
* **`load_gt`** — pure-SSL training loaders read no LiDAR; validation loaders (and supervised / hybrid controls) do.
* `loaders_for(spec)` picks the right (train, val) pair for a loss; oracle (OXTS) loaders are created lazily.


In [18]:
# ════════════════════════════════════════════════════════════════════════════════
# Build train / val / test data (multi-frame sequences)
# ════════════════════════════════════════════════════════════════════════════════
# v6.1 design
#   * ONE canonical list of samples.  With CANONICAL_SAMPLE_SET=True, only samples that have complete OXTS
#     pose + calibration are used — by the SSL loaders AND the oracle-pose loaders — and the train/val
#     index split is shared, so "oracle pose (B) vs predicted pose (C)" differ ONLY in where the pose comes
#     from (same images; augmentation is random but identically distributed).
#   * Pure-SSL TRAINING loaders are built with load_gt=False (no LiDAR read / decoded / transferred);
#     validation loaders and supervised / hybrid controls use load_gt=True.
_canonical_ids = None
if CANONICAL_SAMPLE_SET and (RUN_ORACLE_POSE_ABLATION or RUN_SANITY_EXPERIMENTS):
    try:
        _canonical_ids = probe_oxts_sample_ids(TRAIN_ROOT) or None
        log.info("Canonical sample set: %s samples with complete OXTS pose + calibration",
                 f"{len(_canonical_ids):,}" if _canonical_ids else "0 (-> falling back to all samples)")
    except Exception as e:
        log.warning("Could not probe the OXTS sample set (%s) — SSL and oracle loaders will NOT be forced identical.", e)

_val_ds = KITTISequentialDepthDataset(
    TRAIN_ROOT, split="val", aug_mode="none", max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED,
    load_gt=True, allowed_paths=_canonical_ids)
n_total = len(_val_ds)
_rng    = np.random.RandomState(VAL_SPLIT_SEED)
all_idx = _rng.permutation(n_total)
n_val   = max(1, int(n_total * VAL_SPLIT_FRAC))
_val_idx = sorted(all_idx[:n_val].tolist())
_tr_idx  = sorted(all_idx[n_val:].tolist())

_val_kw   = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
                 prefetch_factor=2, collate_fn=collate_fn, timeout=120)
_train_kw = dict(_val_kw, persistent_workers=False)      # several train loaders may exist -> don't pin idle workers
_loader_kw = _train_kw                                    # (alias used by older cells)
_val_loader = DataLoader(Subset(_val_ds, _val_idx), batch_size=BATCH_SIZE, shuffle=False, **_val_kw)

def _check_same_samples(ds_other, what: str) -> None:
    same = sample_ids(_val_ds) == sample_ids(ds_other)
    if CANONICAL_SAMPLE_SET and _canonical_ids is not None:
        assert same, f"{what}: sample list differs from the SSL sample list — B-vs-C comparison would be invalid"
    elif not same:
        log.warning("%s: sample list differs from the SSL sample list (CANONICAL_SAMPLE_SET off / probe failed) — "
                    "oracle-vs-predicted comparisons are NOT like-for-like.", what)

_train_loader_cache: Dict[Tuple, DataLoader] = {}
def make_train_loader(aug_mode: str = "base", load_gt: bool = False, oracle: bool = False) -> DataLoader:
    key = (aug_mode, load_gt, oracle)
    if key not in _train_loader_cache:
        cls_kw = dict(split="train", aug_mode=aug_mode, max_samples=MAX_TRAIN, seed=VAL_SPLIT_SEED,
                      load_gt=load_gt, allowed_paths=_canonical_ids)
        ds = (KITTISequentialOXTSDataset(TRAIN_ROOT, require_gt_pose=True, **cls_kw) if oracle
              else KITTISequentialDepthDataset(TRAIN_ROOT, **cls_kw))
        assert len(ds) == n_total, f"train dataset has {len(ds)} samples but the val dataset {n_total}"
        _check_same_samples(ds, f"train loader {key}")
        _train_loader_cache[key] = DataLoader(Subset(ds, _tr_idx), batch_size=BATCH_SIZE, shuffle=True,
                                              drop_last=True, **_train_kw)
    return _train_loader_cache[key]

_oracle_val_cache: Dict[str, DataLoader] = {}
def get_oracle_val_loader() -> DataLoader:
    if "v" not in _oracle_val_cache:
        ds = KITTISequentialOXTSDataset(TRAIN_ROOT, split="val", aug_mode="none", max_samples=MAX_TRAIN,
                                        seed=VAL_SPLIT_SEED, load_gt=True, require_gt_pose=True,
                                        allowed_paths=_canonical_ids)
        assert len(ds) == n_total, f"oracle val dataset has {len(ds)} samples but the SSL val dataset {n_total}"
        _check_same_samples(ds, "oracle val loader")
        _oracle_val_cache["v"] = DataLoader(Subset(ds, _val_idx), batch_size=BATCH_SIZE, shuffle=False, **_val_kw)
    return _oracle_val_cache["v"]

def loaders_for(spec: "LossSpec", aug_mode: str = "base") -> Tuple[DataLoader, DataLoader]:
    """(train, val) loaders for a loss: OXTS loaders only for oracle-pose losses; LiDAR loaded only when the
    loss really uses it.  Validation always has LiDAR (evaluation only)."""
    train = make_train_loader(aug_mode, load_gt=spec.uses_gt_depth, oracle=spec.needs_gt_pose)
    val   = get_oracle_val_loader() if spec.needs_gt_pose else _val_loader
    return train, val

test_df = index_kitti_test(TEST_ROOT)
if MAX_TEST > 0:
    test_df = test_df.head(MAX_TEST)

log.info("Dataset split summary:")
log.info("  TRAIN  : %6d samples  (aug=base, offsets=%s, LiDAR loaded only for supervised/hybrid losses)", len(_tr_idx), SOURCE_OFFSETS)
log.info("  VAL    : %6d samples  (aug=none)", len(_val_idx))
log.info("  TEST   : %6d frames (from TEST_ROOT, never seen during HPT)", len(test_df))
log.info("  optimizer updates / epoch = %d  (batches=%d, ACCUM_STEPS=%d)",
         updates_per_epoch_for(len(_tr_idx) // BATCH_SIZE), len(_tr_idx) // BATCH_SIZE, ACCUM_STEPS)

# ── Resumability ─────────────────────────────────────────────────────────────
def get_completed_ablations() -> set:
    """Ablations already in SUMMARY_CSV **for THIS experiment hash**.  Rows written under a different
    configuration / code version are ignored (and reported) — they are never treated as completed work."""
    if not os.path.exists(SUMMARY_CSV):
        return set()
    try:
        df = pd.read_csv(SUMMARY_CSV)
    except pd.errors.EmptyDataError:
        return set()
    if "config" not in df.columns:
        return set()
    if "experiment_hash" not in df.columns:
        log.warning("SUMMARY_CSV has no experiment_hash column (written by an older notebook) — ignoring ALL %d rows. "
                    "Use a fresh OUTPUT_BASE.", len(df))
        return set()
    stale = df[df["experiment_hash"] != EXPERIMENT_HASH]
    if len(stale):
        log.warning("SUMMARY_CSV contains %d row(s) from a DIFFERENT experiment configuration (%s) — they will NOT "
                    "be skipped and new rows would be appended next to them. Prefer a fresh OUTPUT_BASE.",
                    len(stale), sorted(stale["experiment_hash"].astype(str).unique()))
    return set(df[df["experiment_hash"] == EXPERIMENT_HASH]["config"].unique())

completed_ablations = get_completed_ablations()
if completed_ablations:
    log.info("[RESUME] %d ablation(s) already in SUMMARY_CSV — will skip: %s",
             len(completed_ablations), sorted(completed_ablations))


22:39:40  INFO      [KITTI-Seq-val] 1609 samples (offsets=(-1, 1), aug=none)
22:39:40  INFO      [OXTSDataset] 4 drives with GT pose | 1609 samples kept | 0 dropped (no GT pose)
22:39:40  INFO      Canonical sample set: 1,609 samples with complete OXTS pose + calibration
22:39:41  INFO      [KITTI-Seq-val] 1609 samples (offsets=(-1, 1), aug=none)


Indexing test split:   0%|          | 0/652 [00:00<?, ?it/s]

22:39:41  INFO      Test index: 652 frames, 28 drives
22:39:41  INFO      Dataset split summary:
22:39:41  INFO        TRAIN  :   1449 samples  (aug=base, offsets=(-1, 1), LiDAR loaded only for supervised/hybrid losses)
22:39:41  INFO        VAL    :    160 samples  (aug=none)
22:39:41  INFO        TEST   :    652 frames (from TEST_ROOT, never seen during HPT)
22:39:41  INFO        optimizer updates / epoch = 91  (batches=362, ACCUM_STEPS=4)


## 13b · Pre-flight sanity suite (Experiments 0, A – D)

All experiments A–D and the smoke test run with **no augmentation** (identical inputs for B and C). The gate also FAILS if `RUN_ORACLE_POSE_ABLATION=True` and the oracle infrastructure is missing or its pose convention check fails.

Cheap (≈ minutes) proof that each pipeline actually trains the depth network **before** the expensive HPT groups run.

| Exp | What | Pass criteria |
|---|---|---|
| **0a** | OXTS pose vs identity vs inverse on LiDAR pixels | `err(OXTS) < err(identity)` and `< err(inverse)` |
| **0b-1** | One real batch through every ordinary loss, `gt=None` (unless declared supervised/hybrid), backward | finite, non-zero `dL/d(depth)` |
| **0b-2** | The 5 stationary-object losses on **real SAM3 tracks** (scans up to 8 batches; falls back to synthetic masks, flagged in the `input` column) | finite, non-zero `dL/d(depth)` |
| **A** | `lidar_anchor_only` (supervised) | updates applied, skipped ≤ 25 %, depth-grad > 0, params changed, **val RMSE decreases** |
| **B** | Oracle pose + photometric (`gt_photo_reproj`), no GT depth | same health criteria |
| **C** | PoseNet pose + photometric (`min_reproj_photo`) | same **+ non-zero PoseNet gradient**; reports `pose_scale_ratio` (‖t_pred‖/‖t_gt‖, 1 = metric) — *B vs C isolates the effect of pose estimation (same images, no augmentation)* |
| **D** | `temporal_warp_consistency` (pred_next + pose + warped depth) | same |
| **E** | the real ablation | released only if all above pass |

Criteria are about **pipeline health**, not about a few dozen steps improving RMSE. The table (also saved to `results/sanity_*.csv`) reports `|g_depth|`, `|g_pose|`, skipped updates and relative parameter change — the quantities v5 never looked at.

> **v6.2.1 addition — Group 1c real-data coverage (Bug 3/4):** when `RUN_STATIONARY_ABLATION=True`, a new step between 0b and A–D scans `SANITY_STATIONARY_SCAN_BATCHES` real batches through the actual SAM3 loading / tracking / filtering pipeline (`stationary_pair_coverage_report`) and requires at least `SANITY_STATIONARY_MIN_PAIRS` real stationary pairs. This is deliberately separate from 0b's synthetic-mask fallback: synthetic masks only prove the loss *formulas* differentiate correctly, never that the real dataset yields usable stationary objects. Failing this check fails the whole gate, exactly like the oracle-pose check.


In [19]:
# =============================================================================
# PRE-FLIGHT SANITY SUITE  (Experiments 0, A, B, C, D)
# =============================================================================
# Purpose: prove — cheaply — that every pipeline the ablation relies on is ACTUALLY
# training the depth network, BEFORE spending GPU-days.  In v5 nothing checked this and
# 11 of 14 "self-supervised" losses turned out to be no-ops.
#
#   Exp 0a  GT-pose convention check     : OXTS pose must beat identity / inverse pose on LiDAR pixels
#   Exp 0b  Gradient smoke test          : EVERY loss (incl. the 5 tracking-based stationary losses, on real SAM3
#                                          tracks when available, else synthetic masks) -> finite, non-zero dL/d(depth)
#   Exp A   supervised sanity            : SILog on LiDAR must lower val RMSE (the network CAN move)
#   Exp B   oracle pose + photometric    : GT OXTS pose, NO GT depth in the loss
#   Exp C   predicted pose + photometric : same loss, PoseNet pose  (B vs C isolates the effect of pose)
#   Exp D   temporal geometry            : pred_next + pose + warped-depth consistency
#   Exp E   = the real ablation (Groups 1..6) — released only if the above pass.
# Pass criteria are about PIPELINE HEALTH (updates applied, finite non-zero depth gradient — and non-zero
# PoseNet gradient for predicted-pose losses — parameters changed), not about RMSE improving in a few dozen steps.
# =============================================================================

SANITY_PASSED: Optional[bool] = None

@torch.no_grad()
def check_gt_pose_convention(loader: DataLoader, device: torch.device, n_batches: int = 4) -> Dict[str, float]:
    """
    LiDAR-based test of the OXTS pose: at pixels with LiDAR depth, warping I_{t+1} into I_t with the
    GT depth and the OXTS pose must reproduce I_t better than (a) identity and (b) the inverse pose.
    This catches wrong direction / wrong axes (the v5 OXTS pose was expressed in IMU axes).
    """
    errs = {"identity": [], "gt_pose": [], "inverse": []}
    tvals = []
    ni = SOURCE_OFFSETS.index(+1)
    for bi, batch in enumerate(loader):
        if bi >= n_batches:
            break
        assert bool(batch["has_gt_pose"].all())
        rgb = batch["rgb_raw"].to(device); nxt = batch["next_rgb_raw"].to(device)
        depth = batch["depth_gt"].to(device); valid = batch["valid_mask"].to(device)
        K = batch["K"].to(device); Tn = batch["gt_pose_next"].to(device)
        tvals.append(Tn[:, :3, 3].abs().mean(0).cpu().numpy())
        for name, T_ in (("identity", torch.eye(4, device=device).expand_as(Tn)),
                         ("gt_pose", Tn), ("inverse", invert_se3(Tn))):
            w, v = warp_image(nxt, depth, T_, K)
            m = valid * v
            errs[name].append(float(((w - rgb).abs().mean(1, keepdim=True) * m).sum() / (m.sum() + EPS)))
    r = {k: float(np.mean(v)) for k, v in errs.items()}
    tabs = np.mean(tvals, axis=0)
    r["mean_abs_t_xyz"] = tabs.tolist()
    r["ok"] = bool(r["gt_pose"] < r["identity"] and r["gt_pose"] < r["inverse"])
    return r


def _synthetic_stationary_pairs(B: int, H: int, W: int) -> List[List[Tuple["MaskDetection", "MaskDetection"]]]:
    """Two fake stationary objects per sample (fallback when no real SAM3 tracks are available)."""
    out = []
    for _ in range(B):
        pairs = []
        for (y0, y1, x0, x1) in ((int(.45 * H), int(.65 * H), int(.20 * W), int(.35 * W)),
                                 (int(.50 * H), int(.70 * H), int(.60 * W), int(.80 * W))):
            m = np.zeros((H, W), bool); m[y0:y1, x0:x1] = True
            det = MaskDetection(mask=m, bbox=(x0, y0, x1, y1), centroid=((x0 + x1) / 2, (y0 + y1) / 2), area=float(m.sum()))
            pairs.append((det, det))
        out.append(pairs)
    return out


def _find_real_stationary_batch(loader: DataLoader, max_scan: int = 8):
    """Scan a few batches for real SAM3 stationary track pairs. -> (batch, pairs) or (batch0, None)."""
    first = None
    for bi, batch in enumerate(loader):
        first = first or batch
        if not ENABLE_TRACKING:
            break
        pairs = get_batch_stationary_pairs(batch)
        if any(len(p) for p in pairs):
            return batch, pairs
        if bi + 1 >= max_scan:
            break
    return first, None


def gradient_smoke_test(device: torch.device) -> pd.DataFrame:
    """One real batch, pretrained DA3 depth as a leaf tensor, PoseNet from the warm-up state.  For EVERY loss:
    forward with gt=None (unless it is a declared supervised/hybrid loss), backward, report the gradient that
    reaches the depth map.  Two sub-tests: 0b-1 ordinary losses, 0b-2 stationary-object (tracking) losses."""
    rows = []
    ssl_loader = make_train_loader("none", load_gt=False)      # aug "none": sanity comparisons (B vs C especially) use identical inputs
    gt_loader  = make_train_loader("none", load_gt=True)
    try:
        orc_loader = make_train_loader("none", load_gt=False, oracle=True)
    except Exception as e:
        log.warning("oracle loader unavailable for smoke test: %s", e); orc_loader = None
    warm = get_warm_pose_state(ssl_loader, device) if POSE_WARMUP_EPOCHS > 0 else None
    model = load_da3_model_fast(device); _apply_arch(model, "decoder_only")
    pn = PoseNet().to(device)
    if warm is not None:
        pn.load_state_dict(warm)
    pn.eval()
    b_ssl, b_gt = next(iter(ssl_loader)), next(iter(gt_loader))
    b_orc = next(iter(orc_loader)) if orc_loader is not None else None
    b_trk, real_pairs = _find_real_stationary_batch(ssl_loader)

    for name, crit0 in LOSS_REGISTRY.items():
        spec = LOSS_SPECS[name]
        row = {"loss": name, "category": spec.category, "test": "0b-2 tracking" if spec.needs_tracking else "0b-1"}
        if spec.needs_gt_pose and b_orc is None:
            row.update(status="skipped (needs OXTS loaders)"); rows.append(row); continue
        try:
            crit = copy.deepcopy(crit0).to(device)
            batch = (b_orc if spec.needs_gt_pose else b_gt if spec.uses_gt_depth
                     else b_trk if spec.needs_tracking else b_ssl)
            pairs, src = None, ""
            if spec.needs_tracking:
                Bn, Hh, Ww = batch["rgb"].shape[0], *batch["rgb"].shape[-2:]
                if real_pairs is not None and batch is b_trk:
                    pairs, src = real_pairs, "real SAM3 tracks"
                else:
                    pairs, src = _synthetic_stationary_pairs(Bn, Hh, Ww), "synthetic masks (no real tracks found)"
            dyn = (get_batch_dynamic_masks(batch, *batch["rgb"].shape[-2:])
                   if (spec.needs_dyn_mask and USE_DYNAMIC_MASK) else None)
            with torch.no_grad():
                pred, gt, mask, ctx, l_gt, l_mask = prepare_step(batch, model, pn, spec, device, train_depth=False,
                                                                 stationary_pairs=pairs, dyn_mask=dyn)
            leaf = pred.detach().clone().requires_grad_(True)
            loss, terms = _unified_forward(crit, leaf, l_gt, l_mask, **ctx)
            loss.backward()
            g = leaf.grad
            row.update(loss_value=float(loss), grad_norm=float(g.norm()), grad_finite=bool(torch.isfinite(g).all()),
                       gt_depth_visible=(l_gt is not None),
                       valid_frac=float(terms.get("valid_frac", float("nan"))),
                       auto_mask_frac=float(terms.get("auto_mask_frac", float("nan"))),
                       n_objects=float(terms.get("n_objects", float("nan"))),
                       dyn_mask_frac=(float(dyn.mean()) if dyn is not None else float("nan")), input=src)
            ok = row["grad_finite"] and row["grad_norm"] > 0
            if spec.needs_tracking and name == "stationary_uniform_motion" and not ok and \
                    row.get("n_objects", 0) == 0:
                ok = row["grad_finite"]      # needs >=2 objects in a sample; zero loss is legitimate then
            row["status"] = "ok" if ok else "FAIL (zero / non-finite gradient)"
        except Exception as e:
            row["status"] = f"FAIL ({type(e).__name__}: {e})"
        rows.append(row)
    del model, pn; torch.cuda.empty_cache()
    return pd.DataFrame(rows)


def pose_scale_check(pose_net: nn.Module, oracle_val_loader: DataLoader, device: torch.device, n_batches: int = 4) -> float:
    """Mean |t_pred| / mean |t_gt| on oracle-val batches (1.0 = metric scale).  Reported, not a pass criterion:
    photometric self-supervision does not by itself guarantee metric scale."""
    pose_net.eval(); ratios = []
    ni = SOURCE_OFFSETS.index(+1)
    with torch.no_grad():
        for bi, batch in enumerate(oracle_val_loader):
            if bi >= n_batches: break
            rgb, nxt = _to(batch["rgb"], device), _to(batch["next_rgb"], device)
            t_pred = pose_to_se3(pose_net(rgb, nxt))[:, :3, 3].norm(dim=1)
            t_gt = _to(batch["gt_pose_next"], device)[:, :3, 3].norm(dim=1)
            ratios.append(float((t_pred.mean() / t_gt.mean().clamp(min=1e-6))))
    return float(np.mean(ratios)) if ratios else float("nan")


def run_sanity_experiment(exp_id: str, loss_name: str, train_loader: DataLoader,
                          val_loader: DataLoader, device: torch.device,
                          scale_loader: Optional[DataLoader] = None) -> Dict:
    spec = LOSS_SPECS[loss_name]
    warm = get_warm_pose_state(train_loader, device, tag_extra="") if (spec.needs_pred_pose and POSE_WARMUP_EPOCHS > 0) else None
    model = load_da3_model_fast(device); _apply_arch(model, "decoder_only")
    pose_net = None
    if spec.needs_pred_pose:
        pose_net = PoseNet().to(device)
        if warm is not None:
            pose_net.load_state_dict(warm)
    depth_params = [p for p in model.parameters() if p.requires_grad]
    pose_params  = list(pose_net.parameters()) if pose_net is not None else []
    groups = [{"params": depth_params, "lr": SANITY_LR}]
    if pose_params:
        groups.append({"params": pose_params, "lr": POSE_LR})
    opt = build_optimizer(groups, "adamw", SANITY_LR)
    scaler = GradScaler("cuda" if torch.cuda.is_available() else "cpu", enabled=USE_GRAD_SCALER)
    crit = copy.deepcopy(LOSS_REGISTRY[loss_name]).to(device)

    v0 = validate(model, pose_net, val_loader, crit, device, 0, spec, max_batches=SANITY_VAL_BATCHES)
    fp0 = _param_fingerprint(depth_params)
    _, tl = train_one_epoch(model, pose_net, train_loader, crit, opt, scaler, None, device, 1, 0, spec,
                            depth_params=depth_params, pose_params=pose_params,
                            max_batches=SANITY_TRAIN_BATCHES)
    fp1 = _param_fingerprint(depth_params)
    v1 = validate(model, pose_net, val_loader, crit, device, 1, spec, max_batches=SANITY_VAL_BATCHES)

    dg = tl["diag_depth_grad_norm"]
    r = dict(exp=exp_id, loss=loss_name, category=spec.category,
             val_rmse_before=v0["rmse"], val_rmse_after=v1["rmse"], d_rmse=v1["rmse"] - v0["rmse"],
             train_loss=tl.get("total", float("nan")), depth_grad=dg, pose_grad=tl["diag_pose_grad_norm"],
             updates=tl["diag_updates_applied"], skipped=tl["diag_updates_skipped"],
             skip_frac=tl["diag_skip_frac"], bad_loss_batches=tl["diag_bad_loss_batches"],
             param_rel_change=abs(fp1 - fp0) / max(fp0, 1e-12),
             valid_frac=tl.get("valid_frac", float("nan")), auto_mask_frac=tl.get("auto_mask_frac", float("nan")))
    if pose_net is not None and scale_loader is not None:
        r["pose_scale_ratio"] = pose_scale_check(pose_net, scale_loader, device)   # 1.0 = metric; informational
    checks = {
        "updates_applied":   r["updates"] > 0,
        "skip_frac_ok":      r["skip_frac"] <= SANITY_MAX_SKIP_FRAC,
        "depth_grad_nonzero": bool(np.isfinite(dg) and dg > 1e-9),
        "params_changed":    r["param_rel_change"] > 1e-10,
    }
    if spec.needs_pred_pose:                      # a dead PoseNet can hide behind an indirect depth update
        pg = r["pose_grad"]
        checks["pose_grad_nonzero"] = bool(np.isfinite(pg) and pg > 1e-9)
    if exp_id == "A":
        checks["rmse_improves"] = r["val_rmse_after"] < r["val_rmse_before"]
    r["checks"] = checks
    r["PASS"] = all(checks.values())
    del model, pose_net, opt, scaler, crit
    torch.cuda.empty_cache(); gc.collect()
    return r


def run_sanity_suite(device: torch.device) -> bool:
    global SANITY_PASSED
    log.info("=" * 70); log.info("PRE-FLIGHT SANITY SUITE"); log.info("=" * 70)
    oracle_ok, oracle_val = True, None
    try:
        oracle_val = get_oracle_val_loader()
    except Exception as e:
        oracle_ok = False
        log.warning("Oracle (OXTS) loaders unavailable: %s — Exp 0a / B will be skipped.", e)

    ok_all = True
    if RUN_ORACLE_POSE_ABLATION and not oracle_ok:
        ok_all = False
        log.error("RUN_ORACLE_POSE_ABLATION=True but the oracle (OXTS) loaders are unavailable -> sanity gate FAILS "
                  "(Group 1b would crash later). Fix the OXTS data or set RUN_ORACLE_POSE_ABLATION=False.")
    # ── 0a: GT pose convention ────────────────────────────────────────────
    ok_oracle_pose = False
    if oracle_ok:
        r = check_gt_pose_convention(oracle_val, device)
        log.info("[0a] LiDAR photometric L1 | identity=%.4f  OXTS-pose=%.4f  inverse=%.4f  |t|xyz(m)=%s  -> %s",
                 r["identity"], r["gt_pose"], r["inverse"], np.round(r["mean_abs_t_xyz"], 2),
                 "PASS" if r["ok"] else "FAIL")
        if not r["ok"]:
            log.error("[0a] OXTS pose does not reproduce the current frame better than identity/inverse "
                      "-> oracle-pose experiments are NOT trustworthy. Check calibration files / axes.")
        ok_oracle_pose = r["ok"]
        if RUN_ORACLE_POSE_ABLATION and not r["ok"]:
            ok_all = False                       # oracle group requested but its pose convention is wrong

    # ── 0b: gradient smoke test (0b-1 ordinary losses, 0b-2 stationary-object losses) ─────────
    smoke = gradient_smoke_test(device)
    print(smoke.to_string(index=False))
    smoke.to_csv(os.path.join(RESULTS_DIR, "sanity_gradient_smoke.csv"), index=False)
    bad = smoke[smoke["status"].str.startswith("FAIL")]
    if RUN_ORACLE_POSE_ABLATION:                  # an oracle loss that could not be tested is a failure, not a pass
        bad = pd.concat([bad, smoke[smoke["status"].str.startswith("skipped (needs OXTS")]])
    if len(bad):
        ok_all = False
        log.error("[0b] %d loss(es) failed the gradient smoke test: %s", len(bad), bad["loss"].tolist())

    # ── 1c: Group 1c REAL-DATA stationary-pair coverage (v6.2.1, Bug 3/4) ──────────────────────────
    # NOT covered by the smoke test above: 0b-2 can fall back to synthetic masks (that only
    # proves the loss formulas differentiate correctly, not that this dataset yields usable
    # objects). This scans real batches through the actual SAM3 loading / tracking / filtering
    # pipeline (no second tracker implementation) and requires at least SANITY_STATIONARY_MIN_PAIRS
    # real stationary pairs before Group 1c is allowed to launch.
    if RUN_STATIONARY_ABLATION:
        _cov_loader = make_train_loader("none", load_gt=False)
        stationary_coverage_report = stationary_pair_coverage_report(
            _cov_loader, max_batches=SANITY_STATIONARY_SCAN_BATCHES)
        print_stationary_coverage_report(stationary_coverage_report, label="production DYNAMIC_FILTER_MODE")
        globals()["STATIONARY_COVERAGE"] = stationary_coverage_report   # exposed for inspection after this cell runs
        if stationary_coverage_report["total_stationary_pairs"] < SANITY_STATIONARY_MIN_PAIRS:
            ok_all = False
            log.error("[1c] RUN_STATIONARY_ABLATION=True but found only %d real stationary pair(s) in %d scanned "
                      "batch(es) (need >= %d). Group 1c cannot be trusted on this data. Synthetic masks used "
                      "elsewhere in this suite are NOT evidence Group 1c works. Investigate with "
                      "compare_filter_modes(_cov_loader) to see whether class filtering, flow filtering, or "
                      "SAM3_MASK_DIR itself is the cause, then re-run this cell.",
                      stationary_coverage_report["total_stationary_pairs"], SANITY_STATIONARY_SCAN_BATCHES,
                      SANITY_STATIONARY_MIN_PAIRS)

    # ── A–D: short training runs ──────────────────────────────────────────
    exps = [("A", "lidar_anchor_only", False), ("B", "gt_photo_reproj", True),
            ("C", "min_reproj_photo", False), ("D", "temporal_warp_consistency", False)]
    rows = []
    for exp_id, lname, needs_oracle in exps:
        if needs_oracle and not (oracle_ok and ok_oracle_pose):
            log.warning("[%s] skipped (oracle pose unavailable / failed check 0a)", exp_id)
            if RUN_ORACLE_POSE_ABLATION:
                ok_all = False
            continue
        spec = LOSS_SPECS[lname]
        trl, val = loaders_for(spec, aug_mode="none")      # no augmentation: B and C see byte-identical inputs
        log.info("[%s] %s ...", exp_id, lname)
        r = run_sanity_experiment(exp_id, lname, trl, val, device,
                                  scale_loader=(oracle_val if (oracle_ok and spec.needs_pred_pose) else None))
        rows.append(r)
        log.info("[%s] %s | RMSE %.4f -> %.4f | |g_depth|=%.2e |g_pose|=%.2e | updates=%d skipped=%d%s | %s",
                 exp_id, lname, r["val_rmse_before"], r["val_rmse_after"], r["depth_grad"], r["pose_grad"],
                 r["updates"], r["skipped"],
                 (f" | pose-scale ratio {r['pose_scale_ratio']:.2f}" if "pose_scale_ratio" in r else ""),
                 "PASS" if r["PASS"] else f"FAIL {r['checks']}")
        ok_all &= r["PASS"]
    tab = pd.DataFrame(rows)
    if len(tab):
        show = [c for c in tab.columns if c != "checks"]
        print(tab[show].to_string(index=False))
        tab[show].to_csv(os.path.join(RESULTS_DIR, "sanity_experiments.csv"), index=False)

    SANITY_PASSED = bool(ok_all)
    log.info("SANITY SUITE: %s", "ALL CHECKS PASSED — ablation groups released" if SANITY_PASSED
             else "FAILED — fix the pipeline before running any ablation group")
    return SANITY_PASSED


if RUN_SANITY_EXPERIMENTS:
    run_sanity_suite(device)
else:
    log.info("RUN_SANITY_EXPERIMENTS=False — sanity gate disabled.")


22:39:41  INFO      ======================================================================
22:39:41  INFO      PRE-FLIGHT SANITY SUITE
22:39:41  INFO      ======================================================================
22:39:41  INFO      [KITTI-Seq-val] 1609 samples (offsets=(-1, 1), aug=none)
22:39:41  INFO      [OXTSDataset] 4 drives with GT pose | 1609 samples kept | 0 dropped (no GT pose)
22:39:45  INFO      [0a] LiDAR photometric L1 | identity=0.0805  OXTS-pose=0.0300  inverse=0.1028  |t|xyz(m)=[0.01 0.   0.81]  -> PASS
22:39:45  INFO      [KITTI-Seq-train] 1609 samples (offsets=(-1, 1), aug=none)
22:39:45  INFO      [KITTI-Seq-train] 1609 samples (offsets=(-1, 1), aug=none)
22:39:46  INFO      [KITTI-Seq-train] 1609 samples (offsets=(-1, 1), aug=none)
22:39:46  INFO      [OXTSDataset] 4 drives with GT pose | 1609 samples kept | 0 dropped (no GT pose)
22:39:46  INFO      [pose warm-up] training PoseNet for 1 epoch(s), depth frozen (lr=1.0e-04)
22:39:49  INFO          batch

                        loss           category          test  loss_value    grad_norm  grad_finite  gt_depth_visible  valid_frac  auto_mask_frac  n_objects  dyn_mask_frac            input status
                    photo_l1                ssl          0b-1    0.089522 1.257708e-06         True             False    0.986594             NaN        NaN            NaN                      ok
                  photo_ssim                ssl          0b-1    0.282024 4.146861e-06         True             False    0.986594             NaN        NaN            NaN                      ok
               photo_ssim_l1                ssl          0b-1    0.206364 3.373952e-06         True             False    0.987008        1.000000        NaN            NaN                      ok
            min_reproj_photo                ssl          0b-1    0.125327 4.850365e-06         True             False    0.987008        0.522369        NaN            NaN                      ok
           multi_sca

22:43:15  INFO      [A] lidar_anchor_only ...
22:43:15  INFO        [arch=decoder_only] 29.8M params trainable
22:43:15  INFO        [arch=decoder_only] trainable 29.8M / 334.2M params (8.9 %)
22:43:15  INFO            backbone                 trainable    0.00M /  304.37M
22:43:15  INFO            head                     trainable   29.80M /   29.80M


[Group 1c REAL-DATA coverage [production DYNAMIC_FILTER_MODE]]  filter_mode='both'  batches_scanned=8  samples_scanned=32
    batches_with_detections=8  batches_with_matched_tracks=8  batches_with_>=1_stationary=8  batches_with_>=2_stationary=5
    total_stationary_pairs=23  mean/sample=0.719  max/sample=4  samples_with_zero=17  coverage=46.9%
    class histogram before filtering (top 10): {'stationary_object/tree': 192, 'stationary_object/street_lamp': 119, 'stationary_object/building': 67, 'vehicle/personal_vehicle_taillights': 55, 'road/road_surface': 52, 'vehicle_wheels/vehicle_sideways_both_wheels': 41, 'stationary_object/sidewalk': 37, 'vehicle_wheels/vehicle_single_rear_wheel': 28, 'stationary_object/traffic_sign': 24, 'vehicle/personal_vehicle_headlights': 21}


22:43:26  INFO          batch 0/120
22:43:55  INFO          batch 50/120
22:44:24  INFO          batch 100/120
22:44:42  INFO      [A] lidar_anchor_only | RMSE 3.5520 -> 2.3673 | |g_depth|=7.79e-01 |g_pose|=nan | updates=30 skipped=0 | PASS
22:44:42  INFO      [B] gt_photo_reproj ...
22:44:42  INFO        [arch=decoder_only] 29.8M params trainable
22:44:42  INFO        [arch=decoder_only] trainable 29.8M / 334.2M params (8.9 %)
22:44:42  INFO            backbone                 trainable    0.00M /  304.37M
22:44:42  INFO            head                     trainable   29.80M /   29.80M
22:44:52  INFO          batch 0/120
22:45:21  INFO          batch 50/120
22:45:51  INFO          batch 100/120
22:46:11  INFO      [B] gt_photo_reproj | RMSE 3.5520 -> 2.7512 | |g_depth|=7.21e-01 |g_pose|=nan | updates=30 skipped=0 | PASS
22:46:11  INFO      [C] min_reproj_photo ...
22:46:11  INFO        [arch=decoder_only] 29.8M params trainable
22:46:11  INFO        [arch=decoder_only] trainable 29.8M

exp                      loss           category  val_rmse_before  val_rmse_after    d_rmse  train_loss  depth_grad  pose_grad  updates  skipped  skip_frac  bad_loss_batches  param_rel_change  valid_frac  auto_mask_frac  PASS  pose_scale_ratio
  A         lidar_anchor_only supervised_control          3.55198        2.367259 -1.184722    0.084704    0.778728        NaN       30        0   0.000000                 0          0.000003         NaN             NaN  True               NaN
  B           gt_photo_reproj        oracle_pose          3.55198        2.751204 -0.800777    0.080021    0.720848        NaN       30        0   0.000000                 0          0.000020    0.999899        0.718296  True               NaN
  C          min_reproj_photo                ssl          3.55198       10.434252  6.882272    0.124112    0.015121   0.122384       30        0   0.000000                 0          0.000060    0.980502        0.499212  True          0.018395
  D temporal_warp_consis

## 14 · Run the Ablation Groups (1a, 1c, 1b, 2 – 6)


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Gate: never launch an ablation group on a pipeline that has not proven it trains ──────────
if RUN_SANITY_EXPERIMENTS and REQUIRE_SANITY_PASS and SANITY_PASSED is not True:
    raise RuntimeError(
        "Pre-flight sanity suite did not pass (SANITY_PASSED=%r). Fix the reported problem, re-run the "
        "sanity cell, or set REQUIRE_SANITY_PASS=False if you really want to proceed." % (SANITY_PASSED,))

# Loss used as the fixed baseline for the architecture / augmentation / optimizer groups (2-4).
# v6.1: PURE self-supervised by default, so Groups 2-4 really study a self-supervised recipe.
#   Use "combined_recommended" (adds a 0.5 x sparse-LiDAR SILog anchor) ONLY if your question for Groups 2-4 is
#   about the HYBRID objective — and then say so in the thesis.  After Group 1 you may prefer its best SSL loss.
BASE_LOSS_FOR_GROUPS = "combined_recommended_ssl"
if LOSS_SPECS[BASE_LOSS_FOR_GROUPS].uses_gt_depth:
    log.warning("BASE_LOSS_FOR_GROUPS=%r uses sparse LiDAR: Groups 2-4 will study a HYBRID objective, not pure SSL.", BASE_LOSS_FOR_GROUPS)

all_results: List[Dict] = []

def _run(label, grid, loss_name, arch="decoder_only", aug="base",
         optim="adamw", sched="cosine_warmup", criterion=None, spec=None):
    """Loaders are derived from the loss spec: OXTS loaders only for oracle-pose losses, LiDAR loaded only for
    supervised / hybrid losses, validation always with LiDAR (evaluation only)."""
    if label in completed_ablations:
        log.info("  [SKIP] %s already complete", label); return
    sp = spec_for(loss_name, spec)
    train_loader, val_loader = loaders_for(sp, aug_mode=aug)
    res = run_ablation_with_hpt(
        ablation_label=label, hpt_grid=grid, loss_name=loss_name, arch_mode=arch, aug_mode=aug,
        optim_mode=optim, sched_mode=sched, device=device,
        train_loader=train_loader, val_loader=val_loader, test_df=test_df,
        custom_criterion=criterion, spec=sp)
    all_results.append(res)


# ══════════════════════════════════════════════════════════════════════════════
# GROUP 1a - LOSS FUNCTION ABLATION  (predicted pose; SSL + regulariser + supervised/hybrid controls)
#   * every SSL loss is trained WITHOUT access to GT depth (gt=None is enforced)
#   * lidar_anchor_only / combined_recommended are labelled controls (they DO use LiDAR)
# ══════════════════════════════════════════════════════════════════════════════
if RUN_LOSS_ABLATION:
    log.info("\n" + "-" * 70)
    log.info("GROUP 1a: LOSS FUNCTION ABLATION (predicted pose)")
    log.info("-" * 70)
    for loss_name in SSL_LOSS_NAMES + CONTROL_LOSS_NAMES:
        _run(f"loss_{loss_name}", LOSS_HPT_GRID, loss_name)

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 1c - STATIONARY-OBJECT LOSSES (SAM3 tracks + rigid-scene heuristic)   [config prefix `stat_`]
#   Kept apart from 1a: these need instance masks + tracking and encode a different prior
#   (objects labelled static must move like the background), so they are not "plain" photometric/geometric SSL.
# ══════════════════════════════════════════════════════════════════════════════
if RUN_STATIONARY_ABLATION:
    log.info("\n" + "-" * 70)
    log.info("GROUP 1c: STATIONARY-OBJECT LOSSES (SAM3 tracking)")
    log.info("-" * 70)
    for loss_name in STATIONARY_LOSS_NAMES:
        _run(f"stat_{loss_name}", LOSS_HPT_GRID, loss_name)

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 1b - ORACLE-POSE DIAGNOSTIC  (GT OXTS pose + learned depth, NO GT depth in the loss)
#   NOT self-supervised.  Answers: "how much does a perfect pose buy the photometric / geometric loss?"
#   Uses EXACTLY the same samples / split as Group 1a (canonical sample set).
# ══════════════════════════════════════════════════════════════════════════════
if RUN_ORACLE_POSE_ABLATION:
    log.info("\n" + "-" * 70)
    log.info("GROUP 1b: ORACLE-POSE DIAGNOSTIC (OXTS pose)")
    log.info("-" * 70)
    for loss_name in ORACLE_LOSS_NAMES:
        _run(f"oracle_{loss_name}", LOSS_HPT_GRID, loss_name)

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 2 - ARCHITECTURE   |   GROUP 3 - AUGMENTATION   |   GROUP 4 - OPTIMIZER / SCHEDULER
# ══════════════════════════════════════════════════════════════════════════════
if RUN_ARCH_ABLATION:
    log.info("\n" + "-" * 70 + "\nGROUP 2: ARCHITECTURE ABLATION\n" + "-" * 70)
    for arch_mode in ["decoder_only", "last2_blocks", "full"]:
        _run(f"arch_{arch_mode}", ARCH_HPT_GRID, BASE_LOSS_FOR_GROUPS, arch=arch_mode)

if RUN_AUG_ABLATION:
    log.info("\n" + "-" * 70 + "\nGROUP 3: AUGMENTATION ABLATION (paired-safe, K / pose mirrored on flip)\n" + "-" * 70)
    for aug_mode in ["none", "flip", "base", "full"]:
        if f"aug_{aug_mode}" in completed_ablations:
            log.info("  [SKIP] aug_%s already complete", aug_mode); continue
        _run(f"aug_{aug_mode}", AUG_HPT_GRID, BASE_LOSS_FOR_GROUPS, aug=aug_mode)

if RUN_OPTIM_ABLATION:
    log.info("\n" + "-" * 70 + "\nGROUP 4: OPTIMIZER / SCHEDULER ABLATION\n" + "-" * 70)
    for optim_mode, sched_mode in [("adamw", "cosine_warmup"), ("adam", "cosine_warmup"),
                                   ("sgd", "cosine_warmup"), ("adamw", "onecycle"), ("adamw", "step")]:
        _run(f"optim_{optim_mode}_{sched_mode}", OPTIM_HPT_GRID, BASE_LOSS_FOR_GROUPS,
             optim=optim_mode, sched=sched_mode)


# ══════════════════════════════════════════════════════════════════════════════
# GROUP 5 - LOSS COMBINATION (cumulative recipes A -> G)
#   v5 defects fixed: recipes C == D and F == G were IDENTICAL configurations, and the
#   routing was forced through `loss_name="combined_recommended"`.  Routing now comes from an
#   explicit LossSpec derived from the recipe's terms.
#
#   A  photo (SSIM+L1, mean over sources)          (Godard 2017)
#   B  A + edge-aware smoothness
#   C  B + auto-mask                               (Monodepth2 idea 1)
#   D  C + per-pixel MIN over sources              (Monodepth2 idea 2  == Monodepth2 core, needs 3 frames)
#   E  D + warped-depth temporal consistency       (SSL only)
#   F  E + sparse-LiDAR SILog anchor               (HYBRID  == combined_recommended)
#   G  E + SC-SfM geometric consistency            (SSL only)
# ══════════════════════════════════════════════════════════════════════════════
class _Combo(nn.Module):
    """Configurable additive combination of strict loss terms."""
    def __init__(self, terms: Dict[str, float], photo_mode: str = "mean", lam_silog: float = 0.85):
        super().__init__()
        assert photo_mode in ("mean", "automask", "min")
        self.terms, self.lam_silog = dict(terms), lam_silog
        self.photo_mode = photo_mode
        self.photo    = MinReprojPhotoLoss(use_min=(photo_mode == "min"),
                                           use_automask=(photo_mode in ("automask", "min")))
        self.smooth   = EdgeAwareSmoothLoss()
        self.temporal = TemporalWarpConsistencyLoss(photo_weight=0.0)
        self.geom     = GeometricConsistencyLoss()

    def extra_repr(self) -> str:
        return f"terms={self.terms}, photo_mode={self.photo_mode}, lam_silog={self.lam_silog}"

    def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, poses=None,
                pred_next=None, pose_next=None, K=None, has_K=None, dyn_mask=None, **kw):
        total, out = torch.zeros((), device=pred.device), {}
        if "photo" in self.terms:
            l, _ = self.photo(pred, rgb=rgb, rgb_srcs=rgb_srcs, poses=poses, K=K, has_K=has_K)
            total = total + self.terms["photo"] * l; out["photo"] = l
        if "smooth" in self.terms:
            l, _ = self.smooth(pred, rgb=rgb)
            total = total + self.terms["smooth"] * l; out["smooth"] = l
        if "temporal" in self.terms:
            l, _ = self.temporal(pred, pred_next=pred_next, pose_next=pose_next, K=K, dyn_mask=dyn_mask)
            total = total + self.terms["temporal"] * l; out["temporal"] = l
        if "geom" in self.terms:
            l, _ = self.geom(pred, pred_next=pred_next, pose_next=pose_next, K=K, dyn_mask=dyn_mask)
            total = total + self.terms["geom"] * l; out["geom"] = l
        if "anchor" in self.terms:
            l = _silog(pred, gt, mask, self.lam_silog)          # raises if GT was not supplied
            total = total + self.terms["anchor"] * l; out["anchor"] = l
        out["total"] = total
        return total, out

def _combo_spec(terms: Dict[str, float]) -> "LossSpec":
    anchor = terms.get("anchor", 0) > 0
    dyn = any(terms.get(k, 0) > 0 for k in ("temporal", "geom"))
    return LossSpec(needs_pred_pose=True, needs_pred_next=dyn, needs_dyn_mask=dyn,
                    uses_gt_depth=anchor, category="hybrid" if anchor else "ssl")

if RUN_COMBINATION_ABLATION:
    log.info("\n" + "-" * 70 + "\nGROUP 5: LOSS COMBINATION ABLATION (cumulative recipes)\n" + "-" * 70)
    combo_specs = [
        ("A_photo_only",             {"photo": 1.0},                                              "mean"),
        ("B_photo_smooth",           {"photo": 1.0, "smooth": 1e-3},                              "mean"),
        ("C_B_plus_automask",        {"photo": 1.0, "smooth": 1e-3},                              "automask"),
        ("D_monodepth2_core",        {"photo": 1.0, "smooth": 1e-3},                              "min"),
        ("E_D_plus_temporal",        {"photo": 1.0, "smooth": 1e-3, "temporal": 0.1},             "min"),
        ("F_E_plus_anchor_hybrid",   {"photo": 1.0, "smooth": 1e-3, "temporal": 0.1, "anchor": 0.5}, "min"),
        ("G_E_plus_geom_consistency", {"photo": 1.0, "smooth": 1e-3, "temporal": 0.1, "geom": 0.1}, "min"),
    ]
    for combo_id, terms, pmode in combo_specs:
        log.info("Combo %s -> terms %s, photo_mode=%s", combo_id, terms, pmode)
        # the recipe id is the logical loss name -> `loss_fn` in the summary distinguishes the seven recipes
        _run(f"combo_{combo_id}", COMBO_HPT_GRID, f"recipe_{combo_id}",
             criterion=_Combo(terms, pmode), spec=_combo_spec(terms))

    # ── Group 5b: oracle-pose combinations (GT OXTS pose; NOT self-supervised) ─────────────
    class _GTCombo(nn.Module):
        def __init__(self, terms: Dict[str, float], lam_silog: float = 0.85):
            super().__init__()
            self.terms, self.lam_silog = dict(terms), lam_silog
            self.gt_photo, self.gt_flow = GTPosePhotoReprojLoss(), GTPoseRigidFlowLoss()
            self.gt_temporal = GTPoseTemporalWarpLoss(photo_weight=0.0)
            self.smooth = EdgeAwareSmoothLoss()

        def extra_repr(self) -> str:
            return f"terms={self.terms}, lam_silog={self.lam_silog}"

        def forward(self, pred, gt=None, mask=None, *, rgb=None, rgb_srcs=None, rgb_next=None,
                    gt_poses=None, gt_pose_next=None, pred_next=None, K=None, has_K=None, dyn_mask=None, **kw):
            total, out = torch.zeros((), device=pred.device), {}
            if "gt_photo" in self.terms:
                l, _ = self.gt_photo(pred, rgb=rgb, rgb_srcs=rgb_srcs, gt_poses=gt_poses, K=K, has_K=has_K)
                total = total + self.terms["gt_photo"] * l; out["gt_photo"] = l
            if "gt_flow" in self.terms:
                l, _ = self.gt_flow(pred, rgb=rgb, rgb_next=rgb_next, gt_pose_next=gt_pose_next, K=K, has_K=has_K)
                total = total + self.terms["gt_flow"] * l; out["gt_flow"] = l
            if "gt_temporal" in self.terms:
                l, _ = self.gt_temporal(pred, rgb=rgb, rgb_next=rgb_next, pred_next=pred_next,
                                        gt_pose_next=gt_pose_next, K=K, has_K=has_K, dyn_mask=dyn_mask)
                total = total + self.terms["gt_temporal"] * l; out["gt_temporal"] = l
            if "smooth" in self.terms:
                l, _ = self.smooth(pred, rgb=rgb)
                total = total + self.terms["smooth"] * l; out["smooth"] = l
            if "anchor" in self.terms:
                l = _silog(pred, gt, mask, self.lam_silog)
                total = total + self.terms["anchor"] * l; out["anchor"] = l
            out["total"] = total
            return total, out

    gt_combo_specs = [
        ("H_gt_photo_only",          {"gt_photo": 1.0}),
        ("I_gt_photo_smooth",        {"gt_photo": 1.0, "smooth": 1e-3}),
        ("J_gt_photo_full",          {"gt_photo": 1.0, "smooth": 1e-3, "gt_temporal": 0.1, "anchor": 0.5}),
        ("K_gt_rigid_flow_only",     {"gt_flow": 1.0}),
        ("L_gt_flow_smooth_anchor",  {"gt_flow": 1.0, "smooth": 1e-3, "anchor": 0.5}),
    ]
    if RUN_ORACLE_POSE_ABLATION:
        log.info("\n" + "-" * 70 + "\nGROUP 5b: ORACLE-POSE COMBINATIONS (OXTS)\n" + "-" * 70)
        for combo_id, terms in gt_combo_specs:
            gtemp = terms.get("gt_temporal", 0) > 0
            sp = LossSpec(needs_gt_pose=True, needs_pred_next=gtemp, needs_dyn_mask=gtemp,
                          uses_gt_depth=terms.get("anchor", 0) > 0, category="oracle_pose")
            _run(f"combo_gt_{combo_id}", COMBO_HPT_GRID, f"recipe_gt_{combo_id}",
                 criterion=_GTCombo(terms), spec=sp)


# ══════════════════════════════════════════════════════════════════════════════
# GROUP 6 - LOSS WEIGHT SWEEP (around combined_recommended)
# ══════════════════════════════════════════════════════════════════════════════
if RUN_WEIGHT_ABLATION:
    log.info("\n" + "-" * 70 + "\nGROUP 6: LOSS WEIGHT HYPER-PARAMETER SWEEP\n" + "-" * 70)
    weight_specs = [
        # (label, w_photo, w_smooth, w_temporal, w_anchor)
        ("photo_high",      2.0,  1e-3, 0.10, 0.50),
        ("photo_low",       0.5,  1e-3, 0.10, 0.50),
        ("smooth_high",     1.0,  1e-2, 0.10, 0.50),
        ("smooth_low",      1.0,  1e-4, 0.10, 0.50),
        ("temporal_high",   1.0,  1e-3, 0.50, 0.50),
        ("temporal_off",    1.0,  1e-3, 0.00, 0.50),
        ("anchor_high",     1.0,  1e-3, 0.10, 1.00),
        ("anchor_off",      1.0,  1e-3, 0.10, 0.00),   # purely SSL
    ]
    for label, wp, ws, wt, wa in weight_specs:
        log.info("Weights %s -> photo=%.2f smooth=%.0e temporal=%.2f anchor=%.2f", label, wp, ws, wt, wa)
        crit = CombinedRecommendedLoss(w_photo=wp, w_smooth=ws, w_temporal=wt, w_anchor=wa)
        sp = LossSpec(needs_pred_pose=True, needs_pred_next=wt > 0, needs_dyn_mask=wt > 0,
                      uses_gt_depth=wa > 0, category="hybrid" if wa > 0 else "ssl")
        _run(f"weight_{label}", WEIGHT_HPT_GRID, f"weights_{label}", criterion=crit, spec=sp)


# ══════════════════════════════════════════════════════════════════════════════
# Summary
# ══════════════════════════════════════════════════════════════════════════════
log.info("\n*** All HPT-ablation sweeps complete. ***")
summary_df = pd.read_csv(SUMMARY_CSV)
print(f"\nTotal winning configurations evaluated on Eigen test split: {len(summary_df)}")
display_cols = [c for c in ["config", "loss_category", "unscaled_rmse", "median_rmse",
                            "unscaled_abs_rel", "unscaled_delta1", "winner_hpt_tag", "winner_val_rmse",
                            "baseline_val_rmse", "winner_epoch", "n_hpt_trials"] if c in summary_df.columns]
print(summary_df[display_cols].to_string(index=False))
if "baseline_val_rmse" in summary_df.columns:
    stuck = summary_df[(summary_df["winner_val_rmse"] >= summary_df["baseline_val_rmse"] - 1e-3)]
    if len(stuck):
        print("\n[health] configurations that did NOT beat the pretrained baseline on validation "
              "(treat as 'no effect', not as a ranking):")
        print(stuck[["config", "loss_category", "winner_val_rmse", "baseline_val_rmse"]].to_string(index=False))


22:50:43  INFO      
----------------------------------------------------------------------
22:50:43  INFO      GROUP 1c: STATIONARY-OBJECT LOSSES (SAM3 tracking)
22:50:43  INFO      ----------------------------------------------------------------------
22:50:44  INFO      [KITTI-Seq-train] 1609 samples (offsets=(-1, 1), aug=base)
22:50:44  INFO      
22:50:44  INFO      ======================================================================
22:50:44  INFO      HPT SWEEP  ablation=stat_stationary_depth_motion
22:50:44  INFO        loss=stationary_depth_motion [stationary_ssl]  arch=decoder_only  aug=base  optim=adamw
22:50:44  INFO        routing: pred_pose=True gt_pose=False pred_next=True tracking=True uses_GT_depth=False
22:50:44  INFO        2 trial(s) in grid x up to 5 epochs each (patience=4)
22:50:44  INFO      ======================================================================
22:50:44  INFO        [HPT trial] ablation=stat_stationary_depth_motion  tag=lr=1e-05  depth_lr=1.00

Eval: stat_stationary_depth_motion:   0%|          | 0/652 [00:00<?, ?it/s]

01:19:11  INFO        CSVs written: summary, history, hpt_sweep
01:19:13  INFO        -- ablation=stat_stationary_depth_motion complete --
01:19:13  INFO      
01:19:13  INFO      ======================================================================
01:19:13  INFO      HPT SWEEP  ablation=stat_stationary_uniform_motion
01:19:13  INFO        loss=stationary_uniform_motion [stationary_ssl]  arch=decoder_only  aug=base  optim=adamw
01:19:13  INFO        routing: pred_pose=True gt_pose=False pred_next=True tracking=True uses_GT_depth=False
01:19:13  INFO        2 trial(s) in grid x up to 5 epochs each (patience=4)
01:19:13  INFO      ======================================================================
01:19:13  INFO        [HPT trial] ablation=stat_stationary_uniform_motion  tag=lr=1e-05  depth_lr=1.00e-05  pose_lr=1.00e-04  spec=LossSpec(needs_pred_pose=True, needs_gt_pose=False, needs_pred_next=True, needs_tracking=True, uses_gt_depth=False, needs_dyn_mask=False, category='stationary_

Eval: stat_stationary_uniform_motion:   0%|          | 0/652 [00:00<?, ?it/s]

03:38:36  INFO        CSVs written: summary, history, hpt_sweep
03:38:38  INFO        -- ablation=stat_stationary_uniform_motion complete --
03:38:38  INFO      
03:38:38  INFO      ======================================================================
03:38:38  INFO      HPT SWEEP  ablation=stat_full_3d_ego_motion
03:38:38  INFO        loss=full_3d_ego_motion [stationary_ssl]  arch=decoder_only  aug=base  optim=adamw
03:38:38  INFO        routing: pred_pose=True gt_pose=False pred_next=True tracking=True uses_GT_depth=False
03:38:38  INFO        2 trial(s) in grid x up to 5 epochs each (patience=4)
03:38:38  INFO      ======================================================================
03:38:38  INFO        [HPT trial] ablation=stat_full_3d_ego_motion  tag=lr=1e-05  depth_lr=1.00e-05  pose_lr=1.00e-04  spec=LossSpec(needs_pred_pose=True, needs_gt_pose=False, needs_pred_next=True, needs_tracking=True, uses_gt_depth=False, needs_dyn_mask=False, category='stationary_ssl')
03:38:38  INF

Eval: stat_full_3d_ego_motion:   0%|          | 0/652 [00:00<?, ?it/s]

... (2 outputs left out)

08:27:30  INFO        CSVs written: summary, history, hpt_sweep
08:27:32  INFO        -- ablation=stat_mask_chamfer complete --
08:27:32  INFO      
08:27:32  INFO      ======================================================================
08:27:32  INFO      HPT SWEEP  ablation=stat_combined_stationary
08:27:32  INFO        loss=combined_stationary [stationary_ssl]  arch=decoder_only  aug=base  optim=adamw
08:27:32  INFO        routing: pred_pose=True gt_pose=False pred_next=True tracking=True uses_GT_depth=False
08:27:32  INFO        2 trial(s) in grid x up to 5 epochs each (patience=4)
08:27:32  INFO      ======================================================================
08:27:32  INFO        [HPT trial] ablation=stat_combined_stationary  tag=lr=1e-05  depth_lr=1.00e-05  pose_lr=1.00e-04  spec=LossSpec(needs_pred_pose=True, needs_gt_pose=False, needs_pred_next=True, needs_tracking=True, uses_gt_depth=False, needs_dyn_mask=True, category='stationary_ssl')
08:27:32  INFO        [a

Eval: stat_combined_stationary:   0%|          | 0/652 [00:00<?, ?it/s]

10:56:41  INFO        CSVs written: summary, history, hpt_sweep
10:56:43  INFO        -- ablation=stat_combined_stationary complete --
10:56:43  INFO      
*** All HPT-ablation sweeps complete. ***



Total winning configurations evaluated on Eigen test split: 5
                        config  loss_category  unscaled_rmse  median_rmse  unscaled_abs_rel  unscaled_delta1 winner_hpt_tag  winner_val_rmse  baseline_val_rmse  winner_epoch  n_hpt_trials
  stat_stationary_depth_motion stationary_ssl       3.633418     2.661896          0.134793         0.870701       lr=1e-06         4.410983           3.513683             1             2
stat_stationary_uniform_motion stationary_ssl       3.581484     2.663537          0.130946         0.881832       lr=1e-06         4.344858           3.513683             1             2
       stat_full_3d_ego_motion stationary_ssl       3.778045     2.664091          0.141937         0.850012       lr=1e-06         4.575965           3.513683             1             2
             stat_mask_chamfer stationary_ssl       4.202773     2.719043          0.155841         0.794625       lr=1e-06         5.073579           3.513683             1            

## 15 · Load Results


In [21]:
# ── Load the summary CSV ─────────────────────────────────────────────────────
summary_df = pd.read_csv(SUMMARY_CSV)

SCALE_VARIANTS = ["unscaled", "median", "mean", "affine"]
METRIC_COLS    = ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log",
                  "log10", "delta1", "delta2", "delta3", "mean_error"]

# ── Pretty per-group readout ────────────────────────────────────────────────
key_cols = ["config"] + [f"{sv}_rmse" for sv in SCALE_VARIANTS] + [
    "unscaled_abs_rel", "unscaled_delta1", "unscaled_delta2", "unscaled_rmse_log",
    "unscaled_mean_error"]

print("\n" + "=" * 90)
print("ABLATION SUMMARY — Pixel-Weighted True RMSE (m) and Key Metrics")
print("=" * 90)
disp = summary_df[[c for c in key_cols if c in summary_df.columns]].copy()
float_cols = [c for c in disp.columns if c != "config"]
for c in float_cols:
    disp[c] = disp[c].map(lambda x: f"{x:.4f}" if pd.notna(x) else "NaN")
print(disp.to_string(index=False))

# ── Group-level summaries ────────────────────────────────────────────────────
print("\n" + "-" * 60)
print("GROUPED BY ABLATION DIMENSION:")
for prefix, name in [("loss_",   "LOSS (SSL + controls)"),
                     ("stat_",   "STATIONARY-OBJECT LOSSES"),
                     ("oracle_", "ORACLE-POSE DIAGNOSTIC (not SSL)"),
                     ("arch_",   "ARCHITECTURE"),
                     ("aug_",    "AUGMENTATION"),
                     ("optim_",  "OPTIMIZER/SCHED"),
                     ("combo_",  "LOSS COMBINATION"),
                     ("weight_", "LOSS WEIGHT")]:
    sub = summary_df[summary_df["config"].str.startswith(prefix)]
    if sub.empty:
        continue
    print(f"\n  {name} ablation ({len(sub)} configs):")
    for _, r in sub.iterrows():
        uRMSE = r.get("unscaled_rmse", float("nan"))
        mRMSE = r.get("median_rmse",   float("nan"))
        abrel = r.get("unscaled_abs_rel", float("nan"))
        d1    = r.get("unscaled_delta1", float("nan"))
        bias  = r.get("unscaled_mean_error", float("nan"))
        print(f"    {r['config']:<40}  "
              f"RMSE(uns)={uRMSE:.4f}  RMSE(med)={mRMSE:.4f}  "
              f"AbsRel={abrel:.4f}  d<1.25={d1:.4f}  Bias={bias:+.4f}")



ABLATION SUMMARY — Pixel-Weighted True RMSE (m) and Key Metrics
                        config unscaled_rmse median_rmse mean_rmse affine_rmse unscaled_abs_rel unscaled_delta1 unscaled_delta2 unscaled_rmse_log unscaled_mean_error
  stat_stationary_depth_motion        3.6334      2.6619    2.5357      2.4745           0.1348          0.8707          0.9893            0.1666              1.9805
stat_stationary_uniform_motion        3.5815      2.6635    2.5370      2.4761           0.1309          0.8818          0.9895            0.1626              1.9155
       stat_full_3d_ego_motion        3.7780      2.6641    2.5357      2.4759           0.1419          0.8500          0.9886            0.1742              2.1351
             stat_mask_chamfer        4.2028      2.7190    2.5529      2.4822           0.1558          0.7946          0.9864            0.1898              2.4904
      stat_combined_stationary        4.0129      2.6813    2.5332      2.4694           0.1488          

## 16 · Visualisation (15 plots)


In [22]:
# ── Colour palette ───────────────────────────────────────────────────────────
GROUP_COLORS = {
    "loss_":   "#2ecc71",
    "stat_":   "#16a085",
    "oracle_": "#f39c12",
    "arch_":   "#3498db",
    "aug_":    "#e67e22",
    "optim_":  "#9b59b6",
    "combo_":  "#e74c3c",
    "weight_": "#1abc9c",
}
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
}

def _grp_color(label: str) -> str:
    for prefix, col in GROUP_COLORS.items():
        if label.startswith(prefix):
            return col
    return "#7f8c8d"

def _save(fig: plt.Figure, name: str, dpi: int = 140):
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    plt.close(fig)
    log.info("  v %s", name)


# ════════════════════════════════════════════════════════════════════════════
# Plot 00 — Grand Summary Table
# ════════════════════════════════════════════════════════════════════════════
def plot_summary_table(df: pd.DataFrame):
    cols_show = (["config"] +
                 [f"{sv}_{m}" for sv in SCALE_VARIANTS
                              for m in ["rmse", "abs_rel", "delta1", "mean_error"]])
    cols_show = [c for c in cols_show if c in df.columns]
    disp = df[cols_show].copy()
    n_rows = len(disp) + 1
    n_cols = len(cols_show)
    fig_h  = max(6, 0.45 * n_rows)
    fig, ax = plt.subplots(figsize=(min(32, 2.5 * n_cols), fig_h))
    ax.axis("off")
    headers = [c.replace("_", " ") for c in cols_show]
    cell_data = [[str(v) if isinstance(v, str) else f"{v:.4f}" if pd.notna(v) else "NaN"
                  for v in row] for _, row in disp.iterrows()]
    tbl = ax.table(cellText=cell_data, colLabels=headers,
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(7.5); tbl.scale(1.0, 1.6)
    for j in range(n_cols):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    for i, (_, row) in enumerate(disp.iterrows(), start=1):
        bg  = "#f0f4f8" if i % 2 == 1 else "white"
        col = _grp_color(str(row["config"]))
        for j in range(n_cols):
            tbl[i, j].set_facecolor(bg)
        tbl[i, 0].set_facecolor(col)
        tbl[i, 0].set_text_props(color="white", fontweight="bold")
    ax.set_title("SSL Ablation - Full Metric Matrix (pixel-weighted true RMSE, m)",
                 fontsize=12, fontweight="bold", pad=8)
    _save(fig, "00_summary_table.png", dpi=150)

plot_summary_table(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 01 — RMSE Comparison: all 4 scale variants, all configs
# ════════════════════════════════════════════════════════════════════════════
def plot_rmse_all_scales(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.2
    fig, ax = plt.subplots(figsize=(max(14, len(configs) * 0.9), 6))
    for i, (sv, col) in enumerate(SCALE_COLORS.items()):
        vals = [df.loc[df["config"] == c, f"{sv}_rmse"].values[0]
                if f"{sv}_rmse" in df.columns else float("nan")
                for c in configs]
        ax.bar(x + (i - 1.5) * w, vals, width=w, label=sv, color=col, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Pixel-Weighted True RMSE (m)"); ax.set_xlabel("Configuration")
    ax.set_title("RMSE Across All Scale Variants — SSL Ablation Configurations")
    ax.legend(title="Scale Variant"); ax.grid(axis="y", alpha=0.4)
    for prefix in ["arch_", "aug_", "optim_", "combo_", "weight_"]:
        idxs = [i for i, c in enumerate(configs) if c.startswith(prefix)]
        if idxs:
            ax.axvline(x=min(idxs) - 0.5, color="grey", linestyle="--", alpha=0.5)
    _save(fig, "01_rmse_all_scales.png")
plot_rmse_all_scales(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 02 — AbsRel / SqRel / RMSE_log / log10 (unscaled)
# ════════════════════════════════════════════════════════════════════════════
def plot_error_metrics(df: pd.DataFrame):
    metrics = ["unscaled_abs_rel", "unscaled_sq_rel", "unscaled_rmse_log", "unscaled_log10"]
    labels  = ["AbsRel", "SqRel", "RMSE_log", "log10"]
    configs = df["config"].tolist()
    x = np.arange(len(configs))
    fig, axes = plt.subplots(2, 2, figsize=(max(14, len(configs)), 10))
    axes = axes.flatten()
    for ax, metric, lbl in zip(axes, metrics, labels):
        vals = [df.loc[df["config"] == c, metric].values[0]
                if metric in df.columns else float("nan") for c in configs]
        colors = [_grp_color(c) for c in configs]
        bars = ax.bar(x, vals, color=colors, alpha=0.85,
                       edgecolor="white", linewidth=0.8)
        ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45,
                                              ha="right", fontsize=6)
        ax.set_ylabel(lbl + " (unscaled)"); ax.set_title(lbl)
        ax.grid(axis="y", alpha=0.4)
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=5)
    fig.suptitle("Error Metrics (Unscaled) — SSL Ablation",
                 fontsize=13, fontweight="bold")
    _save(fig, "02_error_metrics_unscaled.png")
plot_error_metrics(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 03 — Delta accuracy
# ════════════════════════════════════════════════════════════════════════════
def plot_delta_accuracy(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.25
    fig, ax = plt.subplots(figsize=(max(14, len(configs) * 0.9), 6))
    delta_info = [
        ("unscaled_delta1", "d<1.25",   "#e74c3c"),
        ("unscaled_delta2", "d<1.25^2", "#27ae60"),
        ("unscaled_delta3", "d<1.25^3", "#2980b9"),
    ]
    for i, (col, lbl, c) in enumerate(delta_info):
        vals = [df.loc[df["config"] == cf, col].values[0]
                if col in df.columns else float("nan") for cf in configs]
        ax.bar(x + (i-1)*w, vals, width=w, label=lbl, color=c, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45,
                                          ha="right", fontsize=7)
    ax.set_ylabel("Accuracy (higher = better)"); ax.set_xlabel("Configuration")
    ax.set_title("Threshold Accuracy (Unscaled) — SSL Ablation")
    ax.legend(); ax.grid(axis="y", alpha=0.4)
    ax.set_ylim([0, 1.05])
    _save(fig, "03_delta_accuracy.png")
plot_delta_accuracy(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 04 — Signed bias
# ════════════════════════════════════════════════════════════════════════════
def plot_bias(df: pd.DataFrame):
    configs = df["config"].tolist()
    x = np.arange(len(configs)); w = 0.2
    fig, ax = plt.subplots(figsize=(max(14, len(configs)), 6))
    for i, (sv, col) in enumerate(SCALE_COLORS.items()):
        mc = f"{sv}_mean_error"
        vals = [df.loc[df["config"] == c, mc].values[0]
                if mc in df.columns else float("nan") for c in configs]
        ax.bar(x + (i-1.5)*w, vals, width=w, label=sv, color=col, alpha=0.8)
    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_xticks(x); ax.set_xticklabels(configs, rotation=45,
                                          ha="right", fontsize=7)
    ax.set_ylabel("Mean Signed Error (LiDAR - Pred), m")
    ax.set_title("Depth Bias Across Scale Variants — SSL Ablation")
    ax.legend(title="Scale Variant"); ax.grid(axis="y", alpha=0.4)
    _save(fig, "04_bias_signed.png")
plot_bias(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 05 — Radar: Loss function ablation
# ════════════════════════════════════════════════════════════════════════════
def plot_radar(df: pd.DataFrame, prefix: str, fname: str, title_suffix: str):
    sub = df[df["config"].str.startswith(prefix)].copy()
    if sub.empty:
        return
    radar_metrics = ["unscaled_rmse", "unscaled_abs_rel", "unscaled_rmse_log",
                     "unscaled_delta1", "median_rmse", "affine_rmse"]
    radar_labels  = ["RMSE(uns)", "AbsRel(uns)", "RMSE_log(uns)",
                     "1-d<1.25(uns)", "RMSE(med)", "RMSE(aff)"]
    if "unscaled_delta1" in sub.columns:
        sub["_d1_inv"] = 1.0 - sub["unscaled_delta1"]
        radar_metrics[3] = "_d1_inv"
    metrics_avail = [m for m in radar_metrics if m in sub.columns]
    labels_avail  = [l for m, l in zip(radar_metrics, radar_labels)
                     if m in sub.columns]
    normed = sub[metrics_avail].copy()
    mn = normed.min(); mx = normed.max()
    normed = (normed - mn) / (mx - mn + 1e-9)
    N = len(metrics_avail)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(9, 9),
                           subplot_kw=dict(projection="polar"))
    colors = plt.cm.tab10(np.linspace(0, 1, len(sub)))
    for (_, row), color in zip(sub.iterrows(), colors):
        values = [normed.loc[row.name, m] for m in metrics_avail]
        values += values[:1]
        ax.plot(angles, values, color=color, linewidth=1.6,
                label=row["config"][:30])
        ax.fill(angles, values, color=color, alpha=0.10)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels_avail, fontsize=9)
    ax.set_title(f"{title_suffix} — Radar (normalised, lower area = better)",
                 fontsize=11, fontweight="bold", pad=25)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=7)
    ax.set_ylim(0, 1)
    _save(fig, fname)

plot_radar(summary_df, "loss_",   "05_radar_loss.png",   "Loss Function Ablation")
plot_radar(summary_df, "arch_",   "06_radar_arch.png",   "Architecture Ablation")
plot_radar(summary_df, "combo_",  "06b_radar_combo.png", "Loss Combination Ablation")
plot_radar(summary_df, "weight_", "06c_radar_weight.png","Loss Weight Sweep")


# ════════════════════════════════════════════════════════════════════════════
# Plot 07 — Heatmap: RMSE x scale variant
# ════════════════════════════════════════════════════════════════════════════
def plot_heatmap_rmse(df: pd.DataFrame):
    rmse_cols = [f"{sv}_rmse" for sv in SCALE_VARIANTS]
    rmse_cols = [c for c in rmse_cols if c in df.columns]
    heat_df = df.set_index("config")[rmse_cols].copy()
    heat_df.columns = [c.replace("_rmse", "") for c in rmse_cols]
    fig, ax = plt.subplots(figsize=(max(8, len(SCALE_VARIANTS) * 2),
                                    max(6, len(heat_df) * 0.45)))
    sns.heatmap(heat_df.astype(float), annot=True, fmt=".3f",
                cmap="RdYlGn_r", linewidths=0.5, linecolor="grey",
                ax=ax, cbar_kws={"label": "True RMSE (m)"})
    ax.set_title("RMSE Heatmap — Configuration x Scale Variant",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Scale Variant"); ax.set_ylabel("Configuration")
    _save(fig, "07_heatmap_rmse.png")
plot_heatmap_rmse(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 08 — Heatmap: all metrics (unscaled, normalised)
# ════════════════════════════════════════════════════════════════════════════
def plot_heatmap_all_metrics(df: pd.DataFrame):
    unscaled_cols = [f"unscaled_{m}" for m in
                     ["rmse", "mae", "abs_rel", "sq_rel", "rmse_log", "log10",
                      "delta1", "delta2", "delta3"]]
    unscaled_cols = [c for c in unscaled_cols if c in df.columns]
    heat_df = df.set_index("config")[unscaled_cols].copy()
    heat_df.columns = [c.replace("unscaled_", "") for c in unscaled_cols]
    normed = (heat_df - heat_df.min()) / (heat_df.max() - heat_df.min() + 1e-9)
    # Invert delta cols (higher = better → lower = better in normalised view)
    for c in ["delta1", "delta2", "delta3"]:
        if c in normed.columns:
            normed[c] = 1.0 - normed[c]
    fig, ax = plt.subplots(figsize=(14, max(6, len(df) * 0.45)))
    sns.heatmap(normed, annot=heat_df.map(lambda x: f"{x:.3f}"),
                fmt="", cmap="RdYlGn_r", linewidths=0.5,
                ax=ax, cbar_kws={"label": "Normalised (0=best, 1=worst)"})
    ax.set_title("Metric Heatmap (Unscaled) — All SSL Configurations",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Metric"); ax.set_ylabel("Configuration")
    _save(fig, "08_heatmap_all_metrics.png")
plot_heatmap_all_metrics(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 09 — Learning curves (val RMSE per epoch)
# ════════════════════════════════════════════════════════════════════════════
def plot_learning_curves():
    hist_files = sorted(glob.glob(os.path.join(RESULTS_DIR, "history_*.csv")))
    if not hist_files:
        log.warning("No history CSVs found — skipping learning curves.")
        return
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for hist_file in hist_files:
        try:
            h = pd.read_csv(hist_file)
            label = h["config"].iloc[0]
            color = _grp_color(label)
            if "val_rmse" in h.columns:
                axes[0].plot(h["epoch"], h["val_rmse"], label=label,
                             color=color, linewidth=1.2, alpha=0.7)
            if "val_val_loss" in h.columns or "train_loss" in h.columns:
                lc = "val_val_loss" if "val_val_loss" in h.columns else "train_loss"
                axes[1].plot(h["epoch"], h[lc], label=label,
                             color=color, linewidth=1.2, alpha=0.7)
        except Exception as e:
            log.warning("Could not load %s: %s", hist_file, e)
    for ax, title, ylabel in zip(axes,
            ["Validation RMSE (m) by Epoch", "Loss by Epoch"],
            ["Val RMSE (m)", "Loss"]):
        ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
        ax.legend(fontsize=5, ncol=3); ax.grid(alpha=0.4)
    fig.suptitle("Learning Curves — All SSL Ablation Configurations",
                 fontsize=13, fontweight="bold")
    _save(fig, "09_learning_curves.png")
plot_learning_curves()


# ════════════════════════════════════════════════════════════════════════════
# Plot 10 — Scale sensitivity scatter
# ════════════════════════════════════════════════════════════════════════════
def plot_scale_sensitivity(df: pd.DataFrame):
    if "unscaled_rmse" not in df.columns or "median_rmse" not in df.columns:
        return
    fig, ax = plt.subplots(figsize=(10, 8))
    for _, row in df.iterrows():
        color = _grp_color(str(row["config"]))
        ax.scatter(row["unscaled_rmse"], row["median_rmse"],
                   color=color, s=80, zorder=3,
                   edgecolors="white", linewidth=0.8)
        ax.annotate(
            str(row["config"]).replace("loss_", "").replace("stat_", "").replace("oracle_", "").replace("arch_", "")
                              .replace("aug_", "").replace("optim_", "")
                              .replace("combo_", "").replace("weight_", ""),
            (row["unscaled_rmse"], row["median_rmse"]),
            fontsize=6, xytext=(5, 3), textcoords="offset points")
    vmin = min(df["unscaled_rmse"].min(), df["median_rmse"].min())
    vmax = max(df["unscaled_rmse"].max(), df["median_rmse"].max())
    ax.plot([vmin, vmax], [vmin, vmax], "k--", linewidth=1, alpha=0.5,
            label="y=x (no improvement)")
    ax.set_xlabel("RMSE (unscaled)"); ax.set_ylabel("RMSE (median-aligned)")
    ax.set_title("Scale Sensitivity: Unscaled vs Median-Aligned RMSE")
    ax.grid(alpha=0.4)
    patches = [mpatches.Patch(color=c, label=p.rstrip("_"))
               for p, c in GROUP_COLORS.items()]
    ax.legend(handles=patches, title="Ablation Group", loc="upper left", fontsize=8)
    _save(fig, "10_scale_sensitivity.png")
plot_scale_sensitivity(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 11 — RMSE ranking bar (top 15)
# ════════════════════════════════════════════════════════════════════════════
def plot_ranking_top(df: pd.DataFrame, k: int = 15):
    if "unscaled_rmse" not in df.columns:
        return
    sub = df.sort_values("unscaled_rmse").head(k).copy()
    sub = sub.iloc[::-1]
    colors = [_grp_color(c) for c in sub["config"]]
    fig, ax = plt.subplots(figsize=(10, max(6, 0.4 * len(sub))))
    bars = ax.barh(sub["config"], sub["unscaled_rmse"], color=colors,
                    edgecolor="white", linewidth=0.8)
    for bar, v in zip(bars, sub["unscaled_rmse"]):
        ax.text(v + 0.02, bar.get_y() + bar.get_height() / 2,
                f"{v:.4f}", va="center", fontsize=8)
    ax.set_xlabel("Pixel-Weighted RMSE (unscaled, m)")
    ax.set_title(f"Top-{k} SSL Configurations by Unscaled RMSE")
    ax.grid(axis="x", alpha=0.4)
    _save(fig, "11_top_ranking.png")
plot_ranking_top(summary_df, k=15)


# ════════════════════════════════════════════════════════════════════════════
# Plot 12 — Group-level box plot (RMSE by group)
# ════════════════════════════════════════════════════════════════════════════
def plot_group_box(df: pd.DataFrame):
    if "unscaled_rmse" not in df.columns:
        return
    df = df.copy()
    df["group"] = df["config"].apply(
        lambda c: next((p.rstrip("_") for p in GROUP_COLORS if c.startswith(p)),
                       "other"))
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=df, x="group", y="unscaled_rmse", ax=ax,
                palette={p.rstrip("_"): c for p, c in GROUP_COLORS.items()})
    sns.stripplot(data=df, x="group", y="unscaled_rmse", ax=ax,
                  color="black", size=4, alpha=0.7)
    ax.set_title("RMSE Distribution by Ablation Group")
    ax.set_ylabel("RMSE (unscaled, m)"); ax.set_xlabel("Group")
    ax.grid(axis="y", alpha=0.4)
    _save(fig, "12_group_box.png")
plot_group_box(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 13 — Test RMSE vs winning val RMSE scatter
# ════════════════════════════════════════════════════════════════════════════
def plot_val_vs_test_rmse(df: pd.DataFrame):
    if "winner_val_rmse" not in df.columns or "unscaled_rmse" not in df.columns:
        return
    sub = df.dropna(subset=["winner_val_rmse", "unscaled_rmse"]).copy()
    if sub.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 8))
    for _, row in sub.iterrows():
        color = _grp_color(str(row["config"]))
        ax.scatter(row["winner_val_rmse"], row["unscaled_rmse"],
                   color=color, s=80, edgecolors="white", linewidth=0.8,
                   zorder=3)
    ax.set_xlabel("Best Val RMSE (m)")
    ax.set_ylabel("Test RMSE (unscaled, m)")
    ax.set_title("Generalisation: Validation -> Test RMSE")
    # y = x reference
    vmin = min(sub["winner_val_rmse"].min(), sub["unscaled_rmse"].min())
    vmax = max(sub["winner_val_rmse"].max(), sub["unscaled_rmse"].max())
    ax.plot([vmin, vmax], [vmin, vmax], "k--", alpha=0.4)
    ax.grid(alpha=0.4)
    patches = [mpatches.Patch(color=c, label=p.rstrip("_"))
               for p, c in GROUP_COLORS.items()]
    ax.legend(handles=patches, title="Ablation Group", fontsize=8)
    _save(fig, "13_val_vs_test.png")
plot_val_vs_test_rmse(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 14 — Combo cumulative gain chart
# ════════════════════════════════════════════════════════════════════════════
def plot_combo_cumulative(df: pd.DataFrame):
    sub = df[df["config"].str.startswith("combo_")].copy()
    if sub.empty:
        return
    sub = sub.sort_values("config")
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(range(len(sub)), sub["unscaled_rmse"], marker="o",
            color="#e74c3c", linewidth=2.0, markersize=8)
    for i, (_, r) in enumerate(sub.iterrows()):
        ax.text(i, r["unscaled_rmse"] + 0.02, f"{r['unscaled_rmse']:.4f}",
                ha="center", fontsize=8)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub["config"], rotation=20, ha="right", fontsize=8)
    ax.set_ylabel("Pixel-weighted RMSE (m)")
    ax.set_title("Loss Combination Cumulative Gain — does each added term help?")
    ax.grid(alpha=0.4)
    _save(fig, "14_combo_cumulative.png")
plot_combo_cumulative(summary_df)


# ════════════════════════════════════════════════════════════════════════════
# Plot 15 — Weight sweep heatmap (semantic)
# ════════════════════════════════════════════════════════════════════════════
def plot_weight_sweep(df: pd.DataFrame):
    sub = df[df["config"].str.startswith("weight_")].copy()
    if sub.empty:
        return
    sub = sub.sort_values("unscaled_rmse")
    fig, ax = plt.subplots(figsize=(11, 5))
    colors = [_grp_color(c) for c in sub["config"]]
    bars = ax.bar(range(len(sub)), sub["unscaled_rmse"], color=colors,
                   edgecolor="white", linewidth=0.8)
    for bar, v in zip(bars, sub["unscaled_rmse"]):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02,
                f"{v:.4f}", ha="center", fontsize=8)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub["config"], rotation=20, ha="right", fontsize=8)
    ax.set_ylabel("RMSE (unscaled, m)")
    ax.set_title("Loss Weight Sweep — Sensitivity Around Recommended Defaults")
    ax.grid(axis="y", alpha=0.4)
    _save(fig, "15_weight_sweep.png")
plot_weight_sweep(summary_df)

print(f"\nAll plots saved to {PLOT_DIR}")


10:56:44  INFO        v 00_summary_table.png
10:56:45  INFO        v 01_rmse_all_scales.png
10:56:45  INFO        v 02_error_metrics_unscaled.png
10:56:45  INFO        v 03_delta_accuracy.png
10:56:46  INFO        v 04_bias_signed.png
10:56:46  INFO        v 07_heatmap_rmse.png
10:56:46  INFO        v 08_heatmap_all_metrics.png
10:56:47  INFO        v 09_learning_curves.png
10:56:47  INFO        v 10_scale_sensitivity.png
10:56:47  INFO        v 11_top_ranking.png
10:56:47  INFO        v 12_group_box.png
10:56:48  INFO        v 13_val_vs_test.png



All plots saved to /home/arka/Downloads/MMDE/outputs/self_supervised_FT_output/ablation_study_v6_2_1_stationary_fix/plots


## 17 · Per-Configuration Trial Analysis


In [23]:
# =============================================================================
# Per-configuration HPT trial-level analysis
# =============================================================================
# For every ablation label we plot:
#   (a) Epoch x Val-RMSE curves for each of the 3 LR candidates
#   (b) A bar comparison of best-val RMSE vs final test RMSE (unscaled)
#   (c) A trajectory-quality label per LR (steady-descent / bouncy / early-stop-ok)
#
# This replicates the supervised study's analysis layer so the two studies can
# be compared apples-to-apples.
# =============================================================================

per_config_analysis_rows = []

def _classify_trajectory(vals: List[float]) -> str:
    if len(vals) < 2:
        return "too-short"
    # v6: a flat curve is NOT "steady descent" — it means the network did not move
    if (max(vals) - min(vals)) < 1e-3:
        return "flat/no-effect"
    # Steady descent: monotone-decreasing within ~5% tolerance
    diffs = np.diff(vals)
    if np.all(diffs <= 0.001):
        return "steady-descent"
    # Counts of clear up-moves (>2% of starting value)
    threshold = 0.02 * abs(vals[0]) if vals[0] != 0 else 0.05
    up_moves = (diffs > threshold).sum()
    # Final-vs-min gap (overfit indicator)
    overfit_gap = vals[-1] - min(vals)
    if up_moves >= 2 and overfit_gap > 0.05 * abs(min(vals)):
        return "bouncy/overfit"
    if up_moves >= 1 and overfit_gap > 0.10 * abs(min(vals)):
        return "early-stop-ok"
    if diffs[-1] > 0:
        return "early-stop-ok"
    return "improving"


hist_files = sorted(glob.glob(os.path.join(RESULTS_DIR, "history_*.csv")))
log.info("Found %d history CSVs", len(hist_files))

for hist_file in hist_files:
    h = pd.read_csv(hist_file)
    if h.empty or "hpt_tag" not in h.columns or "val_rmse" not in h.columns:
        continue
    config_label = h["config"].iloc[0]
    trial_groups = h.groupby("hpt_tag")
    n_trials     = len(trial_groups)

    # (a) Per-trial epoch curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    trial_summary = []
    palette = plt.cm.viridis(np.linspace(0.1, 0.85, n_trials))
    for (tag, sub), color in zip(trial_groups, palette):
        sub = sub.sort_values("epoch")
        axes[0].plot(sub["epoch"], sub["val_rmse"], marker="o",
                     color=color, linewidth=1.6, label=tag)
        traj_label = _classify_trajectory(sub["val_rmse"].tolist())
        trial_summary.append({
            "config":      config_label,
            "hpt_tag":     tag,
            "best_epoch":  int(sub.loc[sub["val_rmse"].idxmin(), "epoch"]),
            "best_val":    float(sub["val_rmse"].min()),
            "final_val":   float(sub["val_rmse"].iloc[-1]),
            "trajectory":  traj_label,
            "n_epochs":    int(len(sub)),
            "lr": float(sub["lr"].iloc[0]) if "lr" in sub.columns else float("nan"),
        })
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Val RMSE (m)")
    axes[0].set_title(f"{config_label} — Val RMSE by epoch")
    axes[0].grid(alpha=0.4); axes[0].legend(fontsize=7, ncol=2)

    # (b) Test RMSE bar (from summary_df)
    row_in_summary = summary_df[summary_df["config"] == config_label]
    if not row_in_summary.empty:
        test_rmse = float(row_in_summary["unscaled_rmse"].iloc[0])
        winner_val = float(row_in_summary["winner_val_rmse"].iloc[0]
                            if "winner_val_rmse" in row_in_summary.columns
                            else float("nan"))
        axes[1].bar(["Best Val RMSE", "Test RMSE (unscaled)"],
                    [winner_val, test_rmse],
                    color=["#3498db", "#e74c3c"], width=0.55,
                    edgecolor="white", linewidth=1.5)
        axes[1].set_ylabel("RMSE (m)")
        axes[1].set_title(f"{config_label} — winner: Val vs Test RMSE")
        for i, v in enumerate([winner_val, test_rmse]):
            axes[1].text(i, v + 0.05, f"{v:.4f}",
                          ha="center", fontsize=10, fontweight="bold")
        axes[1].grid(axis="y", alpha=0.4)
    fig.tight_layout()
    plot_path = os.path.join(ANALYSIS_DIR, f"trial_analysis_{config_label}.png")
    fig.savefig(plot_path, dpi=130, bbox_inches="tight")
    plt.close(fig)
    per_config_analysis_rows.extend(trial_summary)
    log.info("  v %s", os.path.basename(plot_path))

# Save per-trial trajectory table
if per_config_analysis_rows:
    trial_table_df = pd.DataFrame(per_config_analysis_rows)
    trial_table_df.to_csv(os.path.join(ANALYSIS_DIR, "trial_trajectory_table.csv"),
                          index=False)
    print("\nTrajectory-quality table written to",
          os.path.join(ANALYSIS_DIR, "trial_trajectory_table.csv"))
    print("\nTrajectory label distribution:")
    print(trial_table_df["trajectory"].value_counts().to_string())
else:
    print("No history files found — skipping per-config analysis.")


10:56:48  INFO      Found 5 history CSVs
10:56:48  INFO        v trial_analysis_stat_combined_stationary.png
10:56:48  INFO        v trial_analysis_stat_full_3d_ego_motion.png
10:56:49  INFO        v trial_analysis_stat_mask_chamfer.png
10:56:49  INFO        v trial_analysis_stat_stationary_depth_motion.png
10:56:49  INFO        v trial_analysis_stat_stationary_uniform_motion.png



Trajectory-quality table written to /home/arka/Downloads/MMDE/outputs/self_supervised_FT_output/ablation_study_v6_2_1_stationary_fix/analysis/trial_trajectory_table.csv

Trajectory label distribution:
trajectory
bouncy/overfit    10


## 18 · Group Winners + Overall Ranking


In [24]:
# =============================================================================
# Winner declaration per group + final summary
# =============================================================================

def _print_group_winners(df: pd.DataFrame, prefix: str, group_name: str):
    sub = df[df["config"].str.startswith(prefix)].copy()
    if sub.empty:
        return None
    sub = sub.sort_values("unscaled_rmse")
    winner = sub.iloc[0]

    print(f"\n{'='*78}")
    print(f"GROUP: {group_name}  ({len(sub)} configurations)")
    print(f"{'='*78}")
    print(f"  WINNER       : {winner['config']}")
    print(f"  RMSE (uns)   : {winner['unscaled_rmse']:.4f} m")
    print(f"  RMSE (med)   : {winner.get('median_rmse', float('nan')):.4f} m")
    print(f"  RMSE (aff)   : {winner.get('affine_rmse', float('nan')):.4f} m")
    print(f"  AbsRel       : {winner.get('unscaled_abs_rel', float('nan')):.4f}")
    print(f"  delta<1.25   : {winner.get('unscaled_delta1', float('nan')):.4f}")
    print(f"  Bias         : {winner.get('unscaled_mean_error', float('nan')):+.4f} m")
    print(f"  Winning HPT  : {winner.get('winner_hpt_tag', 'n/a')}")
    print(f"  Winner epoch : {winner.get('winner_epoch', 'n/a')}")
    print(f"  Trials run   : {winner.get('n_hpt_trials', 'n/a')}")
    print(f"  Test frames  : {winner.get('n_frames', 'n/a')}")

    if len(sub) > 1:
        runner_up = sub.iloc[1]
        gap = runner_up["unscaled_rmse"] - winner["unscaled_rmse"]
        print(f"\n  Runner-up    : {runner_up['config']} "
              f"(gap: +{gap:.4f} m)")
    if len(sub) > 2:
        third = sub.iloc[2]
        gap3 = third["unscaled_rmse"] - winner["unscaled_rmse"]
        print(f"  Third place  : {third['config']} (gap: +{gap3:.4f} m)")

    # Full ranking
    print(f"\n  Full ranking by unscaled RMSE:")
    for i, (_, row) in enumerate(sub.iterrows(), 1):
        marker = " <- WIN" if i == 1 else ""
        gap_w = row["unscaled_rmse"] - winner["unscaled_rmse"]
        print(f"    {i:2d}. {row['config']:<40} {row['unscaled_rmse']:.4f}m  "
              f"(+{gap_w:.4f}){marker}")
    return winner


# v6: oracle-pose runs and the supervised LiDAR control are REFERENCE rows, never "winners" of an SSL study
_ref_cats = ["oracle_pose", "supervised_control"]
if "loss_category" in summary_df.columns:
    _reference_rows = summary_df[summary_df["loss_category"].isin(_ref_cats)]
    _winner_pool = summary_df[~summary_df["loss_category"].isin(_ref_cats)].copy()
    print("\nREFERENCE ROWS (excluded from winner selection):")
    print(_reference_rows[["config", "loss_category", "unscaled_rmse", "median_rmse"]].to_string(index=False))
else:
    _winner_pool = summary_df

group_specs = [
    ("loss_",   "Self-Supervised Loss Function"),
    ("stat_",   "Stationary-Object Loss"),
    ("arch_",   "Architecture (which params to train)"),
    ("aug_",    "Augmentation"),
    ("optim_",  "Optimizer / Scheduler"),
    ("combo_",  "Loss Combination (cumulative)"),
    ("weight_", "Loss Weight Sensitivity"),
]
group_winners = {}
for prefix, name in group_specs:
    w = _print_group_winners(_winner_pool, prefix, name)
    if w is not None:
        group_winners[prefix] = w


# ─── OVERALL WINNER ──────────────────────────────────────────────────────────
print("\n\n" + "*" * 78)
print(" " * 25 + "*** OVERALL WINNER ***")
print("*" * 78)
overall = _winner_pool.sort_values("unscaled_rmse").iloc[0]
print(f"\n  Configuration: {overall['config']}")
print(f"  Test RMSE (unscaled, true pixel-weighted): {overall['unscaled_rmse']:.4f} m")
print(f"  Test RMSE (median-aligned)               : "
      f"{overall.get('median_rmse', float('nan')):.4f} m")
print(f"  Test RMSE (affine-aligned)               : "
      f"{overall.get('affine_rmse', float('nan')):.4f} m")
print(f"  AbsRel: {overall.get('unscaled_abs_rel', float('nan')):.4f}  "
      f"delta<1.25: {overall.get('unscaled_delta1', float('nan')):.4f}  "
      f"Bias: {overall.get('unscaled_mean_error', float('nan')):+.4f} m")
print(f"\n  Winning loss      : {overall.get('loss_fn', 'n/a')}")
print(f"  Winning arch      : {overall.get('arch', 'n/a')}")
print(f"  Winning aug       : {overall.get('aug', 'n/a')}")
print(f"  Winning optim/sched: {overall.get('optim', 'n/a')} / "
      f"{overall.get('sched', 'n/a')}")
print(f"  Winning HPT tag   : {overall.get('winner_hpt_tag', 'n/a')}")
print(f"  Winning epoch     : {overall.get('winner_epoch', 'n/a')}")
print()



REFERENCE ROWS (excluded from winner selection):
Empty DataFrame
Columns: [config, loss_category, unscaled_rmse, median_rmse]
Index: []

GROUP: Stationary-Object Loss  (5 configurations)
  WINNER       : stat_stationary_uniform_motion
  RMSE (uns)   : 3.5815 m
  RMSE (med)   : 2.6635 m
  RMSE (aff)   : 2.4761 m
  AbsRel       : 0.1309
  delta<1.25   : 0.8818
  Bias         : +1.9155 m
  Winning HPT  : lr=1e-06
  Winner epoch : 1
  Trials run   : 2
  Test frames  : 652

  Runner-up    : stat_stationary_depth_motion (gap: +0.0519 m)
  Third place  : stat_full_3d_ego_motion (gap: +0.1966 m)

  Full ranking by unscaled RMSE:
     1. stat_stationary_uniform_motion           3.5815m  (+0.0000) <- WIN
     2. stat_stationary_depth_motion             3.6334m  (+0.0519)
     3. stat_full_3d_ego_motion                  3.7780m  (+0.1966)
     4. stat_combined_stationary                 4.0129m  (+0.4314)
     5. stat_mask_chamfer                        4.2028m  (+0.6213)


*********************

## Group 1 Analysis — Self-Supervised Loss Function  *(rewritten for v6)*

> **The v5 Group 1 write-up is void.** Its ranking (SSL ≈ 2.938 m vs `gt_*` ≈ 2.4 m) compared *inactive* SSL runs (no update was ever applied — see banner, findings 1, 3, 5) with runs that were silently supervised by LiDAR. It supported no conclusion about self-supervised objectives.

### How to read the v6 results

| Block | Configs (`config` prefix) | Meaning |
|---|---|---|
| **1a — SSL** | `loss_<name>` with `loss_category ∈ {ssl, regulariser}` | Genuine self-supervision: PoseNet pose, **no GT depth in the loss** |
| **1a — controls** | `loss_lidar_anchor_only` (supervised), `loss_combined_recommended` (hybrid) | Reference rows, excluded from the "SSL winner" |
| **1c — stationary-object** | `stat_<name>` | SAM3-tracking losses (rigid-scene prior on objects labelled static); reported separately from 1a |
| **1b — oracle pose** | `oracle_<name>` | GT OXTS pose + learned depth, no GT depth. *Not self-supervised.* Upper bound on what a perfect pose gives each objective |

### Mandatory checks before interpreting any number

1. **Beat the baseline?** Compare `winner_val_rmse` with `baseline_val_rmse` (pretrained DA3, epoch 0). A config within ~1e-3 of the baseline had **no effect** — do not rank it. The run-summary cell lists these explicitly.
2. **Health columns** in `history_<config>.csv`: `train_diag_depth_grad_norm`, `train_diag_pose_grad_norm`, `train_diag_skip_frac`, `train_diag_updates_applied`. Zero depth-gradient or a large skip fraction = broken pipeline, not a bad loss.
3. **B vs C** (oracle vs predicted pose, same photometric loss, same canonical sample set and split; augmentation is independently drawn but identically distributed — the sanity B/C runs use none): the gap is the cost of pose estimation. Also inspect `pose_scale_ratio` in `sanity_experiments.csv` — predicted-pose self-supervision does not guarantee metric scale.
4. Differences below ≈ 0.05 m RMSE are within noise for this budget (10 epochs, single seed).

### Questions this group can now answer

* Does any pure-SSL objective improve a metric foundation model at all, and by how much relative to the supervised control?
* Does 3-frame `min_reproj_photo` beat single-term photometric losses?
* Does `temporal_warp_consistency` beat `naive_temporal_delta`? (`naive_temporal_delta` is a *negative control*.)
* Is `edge_aware_smooth` alone degenerate (constant depth), as expected?

*Hypotheses from the literature (Monodepth2 ordering `min_reproj ≥ photo_ssim_l1 ≥ photo_l1`; anchor restores scale) are **hypotheses only** until the sanity suite passes and the numbers above are checked.*


## Group 2 Analysis — Architecture (which parameters do we update?)

### What this group tests

Holding the loss fixed at `combined_recommended`, we vary **which subset of DA3METRIC-LARGE's parameters is trainable** during fine-tuning:

| Mode | Trainable parameters | Roughly what this means |
|------|----------------------|--------------------------|
| `decoder_only` | Decoder + depth head only; encoder frozen | "Adapt the readout, keep the visual prior intact." Cheapest. |
| `last2_blocks` | Decoder + last ~30% of decoder blocks | "Touch only the final fusion stages." Middle-ground. |
| `full` | All parameters trainable | "Adapt everything." Highest capacity, highest risk of forgetting. |

### What to look for

1. **Does adapting more parameters help or hurt?** With a foundation model trained on huge supervised + SSL data, **more trainable params often hurts** in small-data regimes — the model overwrites useful priors. Watch for `decoder_only` ≥ `full` here; that would be the typical foundation-model fine-tuning pattern.

2. **`full` may need a smaller learning rate** to be competitive. The HPT loop tests 3 LRs per architecture, so a high LR catastrophically training `full` will simply lose to `decoder_only`. If `full` wins at the *smallest* LR but loses at the larger ones, that's a sign it could do better with longer training + an even smaller LR.

3. **`last2_blocks` is the most interesting variant** — it tests whether fine-grained adaptation of just the *fusion* layers (not the visual feature extractor) is the sweet spot. Several recent papers argue that adapting only the final cross-attention blocks gives the best generalisation.

### Expected outcomes

- For DA3METRIC-LARGE on KITTI Eigen-test with a 5-epoch budget, **`decoder_only` is expected to win or tie**. The model already has a strong scene prior; we just need to nudge the metric-scale readout.
- **`full` is expected to overfit visibly** in the validation curves (val RMSE first drops then climbs back up). Look at the learning-curve plots.
- Bias direction may flip between architectures: `decoder_only` typically inherits the foundation-model's slight positive bias; `full` can develop new biases as the encoder drifts.

### Interaction with the SSL signal

There is a subtle interaction worth flagging: when the encoder is *frozen* (decoder_only), the photometric residual gradient flows only into the decoder + pose-net. When the encoder is unfrozen (full), the same gradient also rewrites visual features — and *that* is where most foundation-model damage happens in fine-tuning. Hence we expect SSL+full to be *harder to tune* than SSL+decoder_only.


## Group 3 Analysis — Paired-Safe Augmentation

### What this group tests

Self-supervised losses operate on **pairs of consecutive frames** and require that pixel correspondence between the two frames is preserved. This rules out several augmentations that are trivial in the fully-supervised setting:

| Augmentation | Supervised setting | SSL setting |
|--------------|---------------------|--------------|
| Random crop | ✓ Fine | ✗ Breaks principal-point assumption used by warping |
| Random scale | ✓ Fine | ✗ Changes K — pose net would have to learn new intrinsics |
| Independent jitter per frame | ✓ Fine | ✗ Photometric residual becomes meaningless |
| **Paired flip + paired jitter** | ✓ Fine | ✓ Safe — apply identically to both frames |

Our four augmentation modes are therefore:

- **`none`** — raw frames, no augmentation. Lower-bound for what no augmentation buys.
- **`flip`** — paired horizontal flip only (both frames flipped with same coin).
- **`base`** — flip + brightness + contrast jitter, paired.
- **`full`** — for SSL, `full` deliberately equals `base`. We do not add crops / scales here for the reasons above.

### What to look for

1. **Does augmentation help SSL at all?** SSL training is already exposed to a huge amount of visual variation through the photometric residual itself — small lighting changes between frame *t* and frame *t+1* serve as natural data augmentation. So the marginal value of paired jitter may be *smaller* in SSL than in supervised fine-tuning.

2. **Watch the validation curves.** If `none` plateaus rapidly and `base` keeps improving, that's the standard augmentation-helps story. If both curves overlap, augmentation isn't buying much in this regime.

3. **`flip` should be safe and slightly help** — KITTI's scene distribution has left/right asymmetries that horizontal flip removes.

### Expected outcomes

- We expect `base` ≈ `full` (same recipe), both ≥ `flip` > `none`, but the spread should be small (probably < 0.1 m of RMSE) on the foundation-model setting.
- If `none` ties with `base`, it strongly suggests augmentation is largely *redundant with the SSL signal* — useful information for any practitioner building an SSL pipeline.

### Why no random crop?

Random cropping moves the principal point of the camera. The warping operator `pix' = K [R|t] [K^{-1} p * d ; 1]` uses the calibrated intrinsics `K` from the calibration file. Cropping invalidates K because the cropped image has a different principal point. We could re-derive K per crop, but the more honest path is to not crop and stay metric.


## Group 4 Analysis — Optimizer + Scheduler

### What this group tests

We hold the loss (`combined_recommended`), architecture (`decoder_only`), and augmentation (`base`) fixed and vary the **optimizer + LR schedule** across 5 combinations:

| Combination | Optimizer | LR schedule | Notes |
|-------------|-----------|-------------|-------|
| `adamw_cosine_warmup` | AdamW | linear warmup → cosine | Default. AdamW separates weight-decay from gradient updates. |
| `adam_cosine_warmup` | Adam | linear warmup → cosine | Same shape, but weight decay is folded into the gradient → typically slightly worse for transformers. |
| `sgd_cosine_warmup` | SGD+momentum=0.9, Nesterov | linear warmup → cosine | Classical SGD. Requires more careful tuning but generalises well. |
| `adamw_onecycle` | AdamW | OneCycle (super-convergence) | Smith 2018. Fast warmup, slow anneal. |
| `adamw_step` | AdamW | Step LR (halve at midpoint) | Simplest non-constant schedule. Baseline. |

### What to look for

1. **AdamW vs Adam** — AdamW is the modern default for transformer/foundation-model fine-tuning. The expected delta is small but consistent in favor of AdamW.

2. **SGD generalisation** — SGD often **generalises better** than Adam-family optimizers but requires more tuning to converge in the same epoch budget. Watch whether `sgd_cosine_warmup` beats AdamW on TEST despite a worse VAL — that's the classical Adam-vs-SGD finding.

3. **Schedule shape matters more than optimizer choice** at small epoch budgets. `cosine_warmup` and `onecycle` both ramp up then ramp down; `step` is binary. The difference between cosine and onecycle is typically small.

4. **Pose-net + depth-net co-optimization** — A subtle point specific to SSL: both networks are in the same optimizer group, so the LR schedule applies to both. The pose net is typically smaller and easier to train than the depth net, so it benefits less from sophisticated schedules. We accept this co-update as standard practice (Monodepth2 does the same).

### Expected outcomes

- **AdamW + cosine_warmup** is the safe default and is expected to win or tie.
- **AdamW + onecycle** may produce the *best* result if the budget is exactly matched (5 epochs); OneCycle is sensitive to total step count.
- **SGD** is expected to *underperform in this small-budget regime* — SGD needs 2-3x more epochs to catch up to AdamW with foundation models.

### Why no LAMB / LARS / Sophia / Lion?

These optimizers are designed for very large batch sizes or very long pretraining. At our fine-tune scale (effective batch = 8, ~5 epochs) they would just look like noisy variants of AdamW. We omit them for ablation parsimony.


## Group 5 Analysis — Loss Combination (cumulative recipes A → G)

### What this group tests

This is the **central ablation** that directly answers the question raised by the supporting document: *"What is each term in the recommended pipeline actually buying us?"*. We build the recipe additively, starting from pure photometric and adding one term at a time:

| ID | Recipe | What's added | Why |
|----|--------|--------------|-----|
| **A** | photo (SSIM+L1, mean over t±1) | nothing | Godard 2017 baseline |
| **B** | A + smooth | edge-aware smoothness | texture-copy suppression |
| **C** | B + auto-mask | Monodepth2 auto-mask | ignore pixels where warping is worse than identity |
| **D** | C + per-pixel **min** over sources | Monodepth2 core (needs 3 frames) | occlusion handling |
| **E** | D + temporal_warp | warped-depth consistency | SSL only |
| **F** | E + LiDAR anchor | sparse SILog | **hybrid** (= `combined_recommended`) |
| **G** | E + SC-SfM geometric consistency | second geometric term | SSL only |

*(v5 defect: C ≡ D and F ≡ G were identical configurations; v6 makes all seven distinct. Recipes F and G are the only ones with / without LiDAR at the top of the ladder.)*

### What to look for — *the key question*

For each successive recipe, ask:

> Does adding this term **decrease the unscaled test RMSE**, or does it not move the needle?

A monotonically decreasing curve A → G means every term is contributing positively. A flat or rising step means that term is *not useful* in this setting (or even *harmful*).

### Hypotheses

1. **A → B (add smoothness)** should drop RMSE by 0.5–2.0 m. Pure photometric losses produce noisy depth maps; the smoothness prior cleans them up substantially.

2. **B → C (add auto-masking)** should drop RMSE further — auto-masking is the single biggest Monodepth2 contribution.

3. **C → D** isolates the *per-pixel min over t−1/t+1* (needs the 3-frame setting) — the second Monodepth2 idea. v5 had C ≡ D, so no such comparison existed.

4. **D → E (add temporal_warp)** is the most uncertain. If the depth predictions are already reasonably consistent between frames, the temporal term adds little. If they drift, it adds more. **This is the litmus test for whether SSL alone is enough or whether temporal supervision is essential.**

5. **E → F (add anchor)** is expected to drop RMSE noticeably *especially in the unscaled metric*. Pure SSL losses give *scale-ambiguous* depth — they don't know whether your scene is 2 m or 200 m. The anchor injects metric scale from sparse LiDAR. **If F substantially beats E on unscaled but ties on median-aligned RMSE, that means anchor is doing exactly its job: fixing scale.**

6. **F vs G** compares a LiDAR anchor (hybrid) against an extra SSL geometric term (SC-SfM) on top of E — anchor vs. no anchor at the top of the ladder.

### The supporting document's prediction

The accompanying document predicts: photometric (top tier) + smoothness + proper temporal consistency + sparse LiDAR anchor will dominate. This group tests that prediction directly. If A → G monotonically improves, the document is corroborated.

### Caveat: weights are held at defaults

Each cumulative recipe uses the **same default weight** for each term (`w_photo=1.0, w_smooth=1e-3, w_temporal=0.1, w_anchor=0.5`). Sub-optimal weights for one term could artificially mask its contribution. The next group (weight sweep) addresses this.


## Group 6 Analysis — Loss Weight Sensitivity

### What this group tests

Group 5 established *whether each term helps*. Group 6 asks *how sensitive the result is to the relative weight of each term*. We hold the recipe fixed at `combined_recommended` and sweep one weight at a time around its default:

| Setting | w_photo | w_smooth | w_temporal | w_anchor | Hypothesis |
|---------|---------|----------|------------|----------|------------|
| `photo_high`    | **2.0** | 1e-3 | 0.1  | 0.5 | More weight on photometric → more local accuracy, possible scale drift. |
| `photo_low`     | **0.5** | 1e-3 | 0.1  | 0.5 | Less photometric → smoother but coarser depth. |
| `smooth_high`   | 1.0  | **1e-2** | 0.1  | 0.5 | 10× stronger smoothness → over-smoothing, loss of edges. |
| `smooth_low`    | 1.0  | **1e-4** | 0.1  | 0.5 | 10× weaker → texture-copy artefacts return. |
| `temporal_high` | 1.0  | 1e-3 | **0.5** | 0.5 | 5× stronger temporal → cross-frame consistency emphasised. |
| `temporal_off`  | 1.0  | 1e-3 | **0.0** | 0.5 | No temporal — should regress to monodepth2+anchor. |
| `anchor_high`   | 1.0  | 1e-3 | 0.1  | **1.0** | 2× stronger anchor → less SSL, more supervised. |
| `anchor_off`    | 1.0  | 1e-3 | 0.1  | **0.0** | Pure SSL — should regress to Group 5 recipe E. |

### What to look for

1. **Flatness implies robustness.** If RMSE varies by < 5 % across all 8 settings, the recipe is **insensitive** to weight choice → easy to deploy. That is the desired result.

2. **Sharp peaks imply fragility.** If `smooth_high` is much worse than `smooth_low`, the smoothness term is brittle — you'd have to tune it per dataset.

3. **`temporal_off` is the most diagnostic setting.** If it ties with the full recipe, the temporal term is essentially decorative. If it loses by a clear margin, the temporal term is doing real work.

4. **`anchor_off` is the second most diagnostic.** It tells us how much of the recipe's performance comes from the sparse LiDAR anchor *vs.* the SSL terms. A large drop without the anchor suggests SSL alone cannot recover metric scale at this data scale — which is the document's main argument.

5. **`photo_high` vs `photo_low`** — photometric term dominates the gradient signal. Increasing it usually trades off scale stability for local accuracy. Watch the unscaled RMSE: high photometric weight + anchor present should be stable.

### Why this matters in practice

If a single weight setting is *clearly* better than the defaults, the recipe should be retuned. If they're all comparable, ship the defaults. Either outcome is useful information; both are recorded here.


## Final Findings — *to be written after the v6 run*

The v5 "Final Findings" text described expected outcomes as if they had been observed; none were, because the v5 self-supervised runs never updated the network (see the banner at the top). It has been removed.

Fill this section only after: (1) the pre-flight sanity suite passed, (2) each reported configuration beat `baseline_val_rmse`, (3) the health columns show non-zero depth gradients and negligible skipped updates.

**Template**

* **Headline (SSL only, excluding controls/oracle):** best config, unscaled / median-aligned test RMSE, gain over the pretrained baseline.
* **Reference rows:** `lidar_anchor_only` (supervised), best hybrid, best oracle-pose config.
* **Pose cost:** gap between oracle-pose and predicted-pose photometric runs (B vs C).
* **Loss hierarchy:** photometric vs geometric vs combined; naive-temporal negative control.
* **Architecture / augmentation / optimizer / combination / weights:** only if run with a base loss that demonstrably trains.
* **Limitations:** 10-epoch budget, one seed, one dataset, ResNet-18 PoseNet, rigid-scene assumption, `feature_metric` and `rigid_flow_consistency` are simplified variants, DA3 metric conversion uses avg(fx, fy).

All raw numbers: `SUMMARY_CSV`, `history_<config>.csv`, `hpt_sweep_<config>.csv`, `sanity_*.csv`, `trial_trajectory_table.csv`, `plots/`, `analysis/`.


## Change Log (running — updated each build stage)

---

**Stage 8 — v6.2.1 (surgical Group 1c stationary-object fix)**

The user ran Group 1c and observed `train_loss = val_loss = 0` (and `val_rmse ≈ baseline`) for all four
standalone stationary-object losses (`stationary_depth_motion`, `stationary_uniform_motion`,
`full_3d_ego_motion`, `mask_chamfer`), while `combined_stationary` trained normally (its photometric /
smoothness / temporal terms are independent of the stationary-object subterm). This stage is a **surgical**
patch: only the cells listed below were touched; Group 1a, Group 1b, Groups 2–6, and every other v6/v6.1/v6.2
fix are unchanged.

| Cell(s) | Change | Reason |
|---|---|---|
| Part 4 (`is_class_dynamic`) | Normalize SAM3's compound `"super_cat/sub_cat"` labels (underscores/hyphens → spaces, check sub_cat then super_cat then the whole string) before the `STATIC_CLASS_NAMES` / `DYNAMIC_CLASS_NAMES` membership check | **Bug 2**: an exact match against the raw compound string almost never succeeded, so nearly every real detection fell through to the conservative "unknown → dynamic" default regardless of its true class |
| Part 4 self-test | Added compound-label regression cases (`"vehicle/car"`, `"traffic/traffic_sign"`, `"road/road_surface"`, ...) | Prevent Bug 2 from being reintroduced |
| Part 7 (`track_and_filter_pair`) | Use `det_curr.mask` (not `d_next.mask`) when building the pseudo-tracks passed to the flow-stationarity check; unique per-pair `track_id` via `enumerate` instead of a shared `-1`; added an additive, default-`False` `return_diag` flag | **Bug 1 (primary fix)**: `compute_dense_flow(rgb_curr_np, rgb_next_np)` is indexed in frame-t pixel coordinates, so evaluating it with the frame-(t+1) mask samples the wrong image location and can wrongly reject genuinely stationary objects |
| Part 7 (`get_batch_stationary_pairs`) | Added an additive, default-`False` `return_diag` flag (identical behaviour for every existing caller) | Needed by the new coverage diagnostic without a second tracker implementation |
| Part 7 (new) | `stationary_pair_coverage_report()`, `print_stationary_coverage_report()`, `compare_filter_modes()`, `_self_test_coordinate_fix()` | **Bug 3/4**: expose REAL-DATA stationary-pair coverage (detections → matched tracks → stationary pairs, class histogram) separately from the generic smoke test's synthetic-mask fallback; regression-test the Bug 1 fix (fails under the old `d_next.mask` behaviour, passes under the fix — verified both ways) |
| Sanity suite | New step between 0b and A–D: runs `stationary_pair_coverage_report` on real data when `RUN_STATIONARY_ABLATION=True` and fails the whole sanity gate if it finds fewer than `SANITY_STATIONARY_MIN_PAIRS` real stationary pairs | **Bug 3/4**: a synthetic-mask pass must never be read as evidence Group 1c works on real data |
| Runner (`rec` dict) | History-CSV column filter widened to also capture `n_objects` / `n_samples_used` when a loss returns them | **Bug 5**: distinguish "loss is genuinely near zero" from "no stationary objects were supplied" (only the stationary losses populate these keys; every other trial's row simply gets `NaN` in these columns — no schema change for them) |
| Config | `SANITY_STATIONARY_SCAN_BATCHES=8`, `SANITY_STATIONARY_MIN_PAIRS=1` (named like the other `SANITY_*` knobs → automatically excluded from `EXPERIMENT_HASH`, `_HASH_EXCLUDE` itself untouched); `RUN_LOSS_ABLATION` and `RUN_ORACLE_POSE_ABLATION` set to `False` (reversible) for a Group-1c-only rerun; `OUTPUT_BASE` → a dedicated `ablation_study_v6_2_1_stationary_fix` directory; `EXPERIMENT_CODE_VERSION` → `"v6.2.1-stationary-fix"`, which changes `EXPERIMENT_HASH` through the existing (untouched) hashing mechanism | Keep v5/v6/v6.1/v6.2 results auditable; guarantee no Group-1c (or any) checkpoint computed under the buggy code is ever resumed |

**Not changed:** `SOURCE_OFFSETS`, PoseNet architecture/LR/warm-up, OXTS conversion, camera calibration, the
horizontal-flip pose transform, fp32 warp geometry, `MinReprojPhotoLoss`, temporal-warp geometry, the dynamic-mask
geometry (`get_batch_dynamic_masks`), LiDAR loading, the canonical sample set, the train/val split, the DA3 model,
optimizer/scheduler implementations, early stopping, test-set isolation, scale-alignment evaluation, HPT
learning-rate candidates, checkpoint serialization format (beyond the version/hash naturally produced by the
existing mechanism), Group 1a/1b loss definitions, Groups 2–6, CSV evaluation formulas, or plotting logic. The
five stationary loss *classes* themselves (Part 5) and their `LossSpec` routing (`_TR` / `_TRD`) are also unchanged
— the bug was entirely upstream, in what `track_and_filter_pair` handed them.

**Verification performed (cheap/synthetic only — no ablation group was launched):** all 24 notebook code cells
parse; a standalone reproduction of the exact fixed vs. buggy `track_and_filter_pair` logic confirms the fix keeps
a synthetic stationary pair that the old code rejected (1 vs 0); `is_class_dynamic` was checked against 15
plausible SAM3 label formats including the compound forms named in the task; the notebook's own
`_self_test_coordinate_fix()` and extended class-name self-test run as part of cell execution. **Real-data
stationary-pair coverage on the user's actual SAM3 output was NOT run as part of producing this patch** — that
diagnostic must be executed by running the sanity-suite cell in the user's own environment; only its presence,
wiring, and gating logic were verified with synthetic/mocked data here.

**Stage 1 — Parts 3 & 4 (object tracking + dynamic-object filter)**

| Cell(s) | Change | Reason | Expected impact |
|---|---|---|---|
| Config (Part 1) | Added `ENABLE_TRACKING`, `ENABLE_DYNAMIC_FILTER`, `TRACKER_TYPE`, `SAM3_MASK_DIR`, `DYNAMIC_FILTER_MODE`, `FLOW_RESIDUAL_PX`, `TRACK_*`, `BYTETRACK_*` | New switches for Parts 3/4/7 | None when `ENABLE_TRACKING=False` (default trials unaffected) |
| Imports | Added `dataclasses`, `scipy.optimize.linear_sum_assignment` | Needed by `Track`/`MaskDetection` and Trackers 2–3 | Negligible |
| New cells (2c) | Added `MaskDetection`/`Track`, `load_sam3_masks_for_frame`, `MaskIoUTracker`, `HungarianMaskTracker`, `ByteTrackMaskTracker`, `build_tracker` | Part 3 requirement | Pure addition, no existing code touched |
| New cells (2d) | Added `is_class_dynamic`, optical-flow filter, `filter_dynamic_objects` | Part 4 requirement | Pure addition |

**Stage 2 — Parts 5 & 6 (stationary-object losses + training-loop integration), plus one Part 1/2 bugfix found while building Stage 2**

| Cell(s) | Change | Reason | Expected impact |
|---|---|---|---|
| **`KITTISequentialOXTSDataset._build_pairs`** (Part 2b cell) | **Bugfix:** relative pose was computed as `inv(T_curr) @ T_next`; corrected to `inv(T_next) @ T_curr`. | The old formula was the *inverse* of the `pose_curr_to_next` convention `Project3D`/`warp_image` actually consume (verified by explicit derivation — see inline comment). Predicted-pose (PoseNet) losses were unaffected since PoseNet has no fixed external convention and self-corrects during training; the three `gt_pose_*` losses were silently using the wrong-direction transform. | **Retraining any prior run of `gt_photo_reproj`, `gt_flow_depth_consistency`, or `gt_temporal_warp_consistency` is recommended** — their photometric/geometric warps were systematically direction-inverted. New Part 5 losses depend on this same convention and were built/verified against the corrected version. |
| New cells (4a) | Added `track_and_filter_pair`, `get_batch_stationary_pairs` | Bridges Part 3/4 (numpy, per-sample) to Part 5 (torch, batched) | Pure addition |
| New cells (4b) | Added `StationaryObjectDepthMotionLoss` (Loss 1), `StationaryUniformMotionLoss` (Loss 2), `Full3DEgoMotionLoss` (Loss 3), `MaskPointCloudChamferLoss` (Loss 4), `CombinedStationaryConsistencyLoss` (Loss 5), `chamfer_distance` | Part 5 requirement | Pure addition. Registered as 5 new `LOSS_REGISTRY` entries; existing 16 entries untouched |
| Loss registry / routing | Appended 5 keys to `LOSS_REGISTRY`; added `NEED_TRACKING_LOSSES`; extended `NEED_POSE_LOSSES` / `NEED_NEXT_DEPTH_LOSSES` with the same 5 keys; added a 3rd, final `_unified_forward` definition with a new `stationary_pairs=None` kwarg | New losses need pose + `pred_next` + tracked objects to run | Additive only — Python's "last binding wins" means this new `_unified_forward` transparently supersedes the notebook's existing one; all prior isinstance branches copied verbatim, `stationary_pairs` defaults to `None` everywhere it isn't relevant |
| `train_one_epoch`, `validate` (Part 6 cell) | Added `needs_tracking` parameter; compute `stationary_pairs = get_batch_stationary_pairs(batch)` once per step (CPU-only, before the autocast block) when needed; threaded into both `_unified_forward` calls | Part 6: single computation point, reused across every stationary loss term that step, zero cost for trials not using them | New losses functional; existing 16 loss variants see **zero behavior change** (`needs_tracking` defaults `False`, `stationary_pairs` stays `None`, computation is skipped entirely) |
| `_train_one_hpt_trial` (Part 8 cell) | Added `_loss_needs_tracking`; derives and threads `needs_tracking` through both `train_one_epoch`/`validate` calls, mirroring the existing `needs_pose`/`needs_pred_next` pattern | Keeps the ablation runner's per-loss gating consistent for the 5 new registry entries | Additive, same pattern as pre-existing code |
| Part 1 config | Added `DEFAULT_W_STATIONARY`, `DEFAULT_W_UNIFORMITY`, `DEFAULT_W_3D_MOTION`, `DEFAULT_W_CHAMFER`, `CHAMFER_MAX_POINTS` | Part 5/7 configurable weights | None |

**Not yet done (subsequent stages):** Part 9 critical-evaluation pass tying the ablation *results* (once trained) back to a recommended default recipe; Part 10 final section reorg; Part 11 final top-to-bottom consistency pass across the whole notebook (this log will be updated again at that point rather than duplicated).


**Stage 3 — Parts 10 & 11 (notebook cleanup + final consistency pass)**

| Change | Reason | Expected impact |
|---|---|---|
| Renumbered every section header sequentially (0 → 18): `2b/2c/2d` → `3/4/5`, `4a`/`4`/`4b` (which were out of numeric order) → `7/8/9`, and every section from Metrics onward shifted accordingly | Part 10: clean, unambiguous section numbering now that Parts 3-5's new sections are interleaved with the original notebook | Cosmetic only — no code semantics changed. One stale in-text cross-reference ("Section 4" → "Section 8") in the Part 5 markdown was also corrected. |
| Automated full-notebook consistency check (AST-based): every `Name` used anywhere in the notebook is either a builtin or is defined/imported somewhere in the notebook; cross-checked for duplicate global definitions | Part 11 requirement | **Zero unresolved names found.** The only global name defined more than once is `_unified_forward` (3 versions across the original notebook + Stage 2), which is an intentional, already-existing "last binding wins" pattern in this codebase (same pattern the original notebook already used for `LOSS_REGISTRY`/`_unified_forward` before I touched it) — not a new bug. |
| Re-ran `nbformat.validate()` + a full per-cell `compile()` syntax pass, plus the Stage 1/2 self-tests (synthetic tracker sequence, synthetic geometry with known ground truth, and direct execution of all 5 new loss classes with real gradient checks) | Part 11 requirement: verify the notebook is actually executable, not just plausible-looking | All pass. (Full top-to-bottom *execution* against real KITTI data + the actual DA3 checkpoint on your V100 server is the one verification step I cannot perform from here — everything short of that has been checked.) |


**Stage 4 — Part 9 (critical evaluation & principled improvements)**

| Change | Reason | Expected impact |
|---|---|---|
| New Section 9a markdown: 5 findings from re-examining Stages 1-2 against alternatives | Part 9 requirement | Documents *why* each Stage-4 code change below was made, and honestly flags one change considered and **not** made (sequence-based re-architecture for real multi-frame tracking) as future work rather than attempted under time pressure |
| `_erode_mask` + patched `_StationaryObjectLossBase._resize_mask` | Finding 3: SAM3 mask boundaries leak background pixels into small objects' median-depth estimate | All 4 stationary-object losses + the new geometric filter get cleaner depth samples; verified on a synthetic mask (100px → 36px after 2px erosion, boundary strip removed) |
| Patched `_iter_batch_objects` to yield an area-based weight; `StationaryObjectDepthMotionLoss.forward` / `Full3DEgoMotionLoss.forward` replaced with area-weighted-mean versions | Finding 2: unweighted averaging let a 9px noisy sliver contribute equally to a 1024px reliable object | Verified: a planted bad tiny object pulled an unweighted mean to 6.0, weighted mean landed at 1.03 — reliable objects now dominate as intended |
| `StationaryUniformMotionLoss.forward` replaced with MAD-based version (`UNIFORMITY_METRIC="mad"` default, `"variance"` kept for ablation) | Finding 4: variance is quadratic — one bad object can swamp the whole sample's signal | Verified: identical planted-outlier scenario gave variance=330.75 vs. MAD-loss=10.5 — over 30x less sensitive to the single outlier |
| New `geometric_consistency_filter` (no_grad), wired into `train_one_epoch`/`validate` right before the loss call, gated by `ENABLE_GEOMETRIC_FILTER` (default `False`) + `GEOMETRIC_FILTER_WARMUP_EPOCHS` | Finding 5: an extra outlier-rejection layer beyond Part 4's class+flow filter, using the model's own geometric residual — with an explicit, honest chicken-and-egg caveat in the markdown about why it's off by default | Verified: correctly identified and dropped the 1 planted geometric outlier from a 4-object synthetic sample, kept all 3 consistent ones. Zero effect on any trial with `ENABLE_GEOMETRIC_FILTER=False` (the default) |
| `TRACKER_TYPE` reviewed against Finding 1 (2-frame-only tracker usage) | Tracker choice is a configuration choice; no optimality claim is retained because no held-out tracking-quality evaluation was performed | Earlier 'confirmed decision' wording withdrawn in v6.1 |

**Bug caught during Stage 4's own testing (worth noting for the log's honesty, not just the shipped result):** the first draft of the mask-erosion patch used `_StationaryObjectLossBase._resize_mask.__func__`, which raised `AttributeError` — Python 3.10+ returns a plain function (not a bound/wrapped method) when a `staticmethod` is accessed via the class, so `.__func__` doesn't exist. Caught by the synthetic test harness before this reached the shipped notebook; fixed to a direct reference.

---

**Stage 5 — v6 corrections (post-mortem of the v5 Group 1 failure)**

| Cell(s) | Change | Reason |
|---|---|---|
| Config | New `ablation_study_v6` output dir; `SOURCE_OFFSETS`, `POSE_LR`, `POSE_OUT_SCALE`, `POSE_WARMUP_*`, `AMP_DTYPE_MODE`, sanity-suite and oracle-group switches; removed the unused `POSE_NET_FREEZE_EPOCHS` | v5 results must not be resumed; new mechanisms need knobs |
| Dataset | Multi-frame (t−1, t, t+1) samples; flip / jitter drawn once per sample; `K.cx` mirrored on flip; `flipped` flag; GT depth only for t and t+1 | 3-frame Monodepth2 training; geometry-correct augmentation |
| OXTS | Camera-frame relative pose by conjugation with `T_ic`; drives without calibration dropped; no identity/None substitute; pose mirrored on flip; `require_gt_pose`; own train/val split | v5 pose was in IMU axes; silent fallbacks removed |
| Model / warp | fp32 backprojection / projection / warping (autocast disabled); `z>0` validity; PoseNet: non-zero init, out-scale 0.1, offline-safe weights; `invert_se3`; focal = avg(fx, fy) | fp16 overflow (`K·X ≈ 9.6e4 > 65504`), zero depth-gradient at identity pose |
| Batch tracking | SAM3 detections mirrored for flipped samples | mask/image mismatch |
| Losses | Rewritten: masked photo-L1; `MinReprojPhotoLoss(use_min, use_automask)` with identity tie-break noise; image-pyramid multi-resolution loss; bounded-L1 temporal residual with separate visibility mask; feature loss at 1/4 resolution with ImageNet normalisation; `flow_depth_consistency` → `rigid_flow_consistency`; new `combined_recommended_ssl`; strict `GTPose*` wrappers; all fallbacks → `SSLInputError` | findings 1, 2, 10 |
| Stationary losses | Missing inputs raise; combined loss on the new photometric API | no silent zero-loss |
| Specs (new) | `LOSS_SPECS` + `_unified_forward` (autocast off, fp32) | single source of truth for routing |
| Training | Per-group LR (depth / pose), per-group grad-clip, `updates_per_epoch` scheduler sizing, final-window flush, skipped-update accounting, look-ahead tracking iterator, `gt=None` for SSL losses | findings 3, 4, 5, 8 |
| Runner | PoseNet warm-up (cached), epoch-0 baseline validation, fail-fast on zero updates, health columns in history/CSV | make "network never moved" impossible to miss |
| Sanity suite (new) | Exp 0a/0b/A/B/C/D + gate | verify pipelines before spending GPU-days |
| Groups | Group 1a (SSL + controls) / 1b (oracle) split; distinct Group 5 recipes; specs per recipe; `BASE_LOSS_FOR_GROUPS` | findings 2, 9 |
| Analysis cells | `oracle_` prefix; winners exclude oracle & supervised-control rows; trajectory classifier flags flat runs | avoid ranking controls as winners |

**Verification done while building v6 (CPU, synthetic data):** warp recovers a known 0.5 m motion (photometric L1 0.001 vs 0.054 identity vs 0.102 inverse pose); depth-gradient at identity pose = 2.7e-10 vs 3.6e-5 at 0.5 m; every one of the 23 losses runs with `gt=None`, gives finite non-zero depth gradient, and raises when an input is missing; OXTS relative pose yields `t = (0, 0, −1.002)` for 1 m forward driving; scheduler trace confirms warm-up/cosine are sized by optimizer updates.

---

**Stage 6 — v6.1 (second review)**

| Cell(s) | Change |
|---|---|
| Config | `SAVE_EVERY_N_EPOCHS=0`; `POSE_WARMUP_CODE_VERSION`; `CANONICAL_SAMPLE_SET`, `USE_DYNAMIC_MASK`, `DYN_MASK_DILATE_PX`, `RUN_STATIONARY_ABLATION`, `STRICT_ARCH_CHECK`, `EXPECTED_TRAINABLE_FRACTION`; scale caveat; tracker comment |
| Dataset / OXTS | `load_gt`, `allowed_paths`; `probe_oxts_sample_ids`, `assert_same_samples` (replaces `make_oxts_loaders`) |
| Model | `describe_trainable()` + assertion after every `_apply_arch` |
| Batch tracking | `get_batch_dynamic_masks` (SAM3 dynamic-class prior, flip-aware, dilated) |
| Losses | `dyn_mask` in geometric / temporal / combined / GT-temporal; identity noise before min; separate x/y `K` scaling in `FeatureMetricLoss` and `MultiScalePhotoLoss` |
| Stationary losses | Part 9 patches merged into the classes; patch cell reduced to the optional geometric filter; neutral tracker wording |
| Specs | `needs_dyn_mask`; category `stationary_ssl`; `STATIONARY_LOSS_NAMES` |
| Training | per-window gradient scaling; look-ahead iterator for tracking + dynamic masks; optional GT |
| Runner | baseline computed once per validation set (cached); strong pose-warm-up cache key; `temp_ep*` cleanup |
| Loaders | canonical sample set, shared split, lazy `make_train_loader` / `loaders_for(spec)` |
| Sanity | 0b-2 tracking losses; `pose_grad_nonzero`; `pose_scale_ratio`; fixed clobbered `loss` column |
| Groups | `BASE_LOSS_FOR_GROUPS="combined_recommended_ssl"`; Group 1c (`stat_`); loaders derived from specs; dyn-mask flags on recipes/weights |

**Verification (CPU, synthetic):** all 23 losses (incl. merged stationary classes) run with `gt=None` and give finite non-zero gradients; dynamic mask mirrors on flip; `load_gt=False` removes all depth keys; SSL and oracle datasets yield identical sample ids; HPT trial with cached baseline and warm-up runs; smoke test + sanity experiments A/C/D pass on fakes.

---

**Stage 7 — v6.2 (third review)**

| Cell(s) | Change |
|---|---|
| Config | output dir `ablation_study_v6_2`; `EXPERIMENT_CODE_VERSION`; `EXPERIMENT_CONFIG` / `EXPERIMENT_HASH` (all training-relevant constants + data roots) at the END of the cell; hash-namespaced `TEMP_CKPT_DIR`; `experiment_config_<hash>.json` written to `results/` |
| Runner | `_trial_hash` (loss repr incl. weights, arch, aug, optim, sched, spec, HPT cfg, samples); hash in checkpoint names + verified on resume; `experiment_hash` column in the summary; warm-up cache key = hash of a canonical dict |
| Loaders | `get_completed_ablations()` counts only rows of the current experiment hash and warns about foreign rows |
| Losses / stationary | `extra_repr` on all parametrised losses (so weights are part of the trial hash) |
| Batch tracking | dynamic mask from the CURRENT frame only |
| Training | rescale accumulated gradients by `window_size / valid_in_window` when non-finite batches were skipped |
| Specs | `_TR` (no dynamic mask) for four stationary losses, `_TRD` for `combined_stationary` |
| Sanity | `aug_mode="none"` for smoke test and A–D; gate fails when oracle infrastructure is required but unavailable / fails 0a / B skipped |
| Groups | recipe / weight-set ids as logical loss names (`recipe_*`, `recipe_gt_*`, `weights_*`) |
| Change log (v5 row) | stale "Hungarian is the theoretically correct choice" row rewritten |

**Verification (CPU, synthetic):** update with one skipped batch equals the exact mean gradient of the three finite batches (max abs error 1.9e-7); identical trial configuration resumes, a different loss weight creates a separate checkpoint; experiment hash changes when a late-defined config constant changes; sanity gate returns False when oracle loaders are required but unavailable; full test battery re-run (23 losses, train/validate, HPT trial, smoke test, dataset / OXTS / dynamic mask).
